# [1.4.1] Indirect Object Identification (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/21_[1.4.1]_Indirect_Object_Identification)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part41_indirect_object_identification/1.4.1_Indirect_Object_Identification_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part41_indirect_object_identification/1.4.1_Indirect_Object_Identification_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널에 알려주시고, 본 장의 내용에 관한 질문은 전용 채널에 문의해 주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어서 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 가는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-14-1.png" width="350">

# 소개

이 노트북/문서는 [Interpretability in the Wild](https://arxiv.org/abs/2211.00593) 논문을 중심으로 구성되어 있으며, 저자들은 GPT-2 small의 **indirect object identification circuit**을 이해하는 것을 목표로 합니다. 이 circuit은 `"John and Mary went to the shops, John gave a bag to"`과 같은 문장을 올바른 token인 "`" Mary"`"로 완성하는 모델의 능력을 담당합니다.

내용은 성격이 서로 다른 여러 섹션으로 느슨하게 나뉘어 있습니다. 섹션 1, 2, 3은 Neel Nanda의 노트북 [Exploratory_Analysis_Demo](https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/Exploratory_Analysis_Demo.ipynb#scrollTo=WXktSe0CvBdh)에서 파생되었습니다. 이 연습 문제들의 성격은 실험적이고 자유로우며, transformerlens 라이브러리를 사용하여 실제 탐색적 분석(exploratory analysis)이 어떻게 이루어지는지 보여주는 데 중점을 둡니다. 엄격함보다는 이 circuit에 대한 암시적인 증거를 찾는 과정을 빠르게 진행하는 방식을 취합니다. 코드와 연습 문제는 단순하고 일반적이지만, 각 단계가 무엇을 하는지, 그리고 왜 하는지에 대한 상세한 설명(그리고 몇 가지 선택적인 세부 사항과 여담)이 함께 제공됩니다. 섹션 4에서는 모델의 동작을 분석하는 더 엄격하고 구조적인 방법인 **path patching** 개념을 소개합니다. 여기서는 논문의 결과 일부를 재현하며, 이를 통해 이전 섹션에서 얻은 통찰을 엄격하게 검증하게 됩니다. 다섯 섹션 중 기술적으로 가장 밀도가 높은 부분입니다. 마지막으로 섹션 5와 6은 구조가 훨씬 덜 잡혀 있으며, 개방형 연습 문제와 스스로 탐색해 보는 것에 더 큰 중점을 둡니다.

어떤 연습 문제를 수행할지는 이 과정을 통해 얻고자 하는 바에 따라 달라집니다. 예를 들어:

* activation patching을 이해하고 싶은 경우 - **1, 2, 3**
* 모델에 대해 탐색적 분석을 수행하는 방법을 익히고 싶은 경우 - **1, 2, 3**
* activation 및 path patching을 이해하고 싶은 경우 - **1, 2, 3, 4**
* IOI circuit을 완전히 이해하고 논문의 핵심 결과를 재현하고 싶은 경우 - **1, 2, 3, 4, 5**
* IOI circuit을 완전히 이해하고 논문의 핵심 결과를 재현하고 싶은 경우 (단, activation patching은 이미 이해하고 있는 경우) - **1, 2, 4, 5**
* IOI를 이해한 후, 모델에서 더 많은 circuit을 찾거나 이상 현상을 조사하는 등 더 깊이 파고들고 싶은 경우 - **1, 2, 3, 4, 5, 6**

*참고 - CUDA 메모리 에러가 자주 발생한다면, 주기적으로 `torch.cuda.empty_cache()`를 호출하여 [free up some memory](https://stackoverflow.com/questions/57858433/how-to-clear-gpu-memory-after-pytorch-model-training-without-restarting-kernel) 하십시오.*

각 연습 문제에는 5점 만점의 난이도와 중요도 등급, 예상 최대 소요 시간, 그리고 때로는 짧은 주석이 달려 있습니다. 등급과 예상 시간은 상대적으로 해석하시기 바랍니다 (예: 예상 시간보다 약 50% 더 많은 시간을 쓰고 있다면 그에 맞춰 조정하십시오). 충분히 중요하지 않다고 느껴지거나 더 핵심적인 내용으로 빠르게 넘어가고 싶다면, 연습 문제를 건너뛰거나 솔루션을 확인하셔도 좋습니다!

## 이 실습들의 목적 / 구조

표면적으로 이 실습들은 여러분이 indirect object identification circuit을 살펴볼 수 있도록 설계되었습니다. 하지만 동시에 여러분을 더 뛰어난 interpretability 연구자로 만들기 위해 설계되었습니다! 그 결과, 대부분의 실습은 다음의 조합으로 구성됩니다:

1. circuit의 새로운 feature/component를 보여주는 것, 그리고
2. 더 넓은 mech interp 맥락에서 도구를 사용하고 결과를 해석하는 방법을 가르치는 것입니다.

이 실습들을 진행하는 동안 염두에 두어야 할 핵심 아이디어는 **더 단순하고 탐색적인 도구부터 더 엄격하고 복잡한 도구까지의 스펙트럼**입니다. 단순한 쪽에는 attention patterns를 조사하는 것과 같은 방법이 있으며, 이는 attention head가 무엇을 하고 있는지에 대해 꽤 괜찮은(하지만 때로는 오해의 소지가 있는) 그림을 제공할 수 있습니다. 이러한 도구들은 가장 먼저 사용해야 할 도구들이며, circuit에 대한 구체적인 가설이 세워지기 전에도 많이 사용해야 합니다. 더 엄격한 쪽에는 path patching과 같은 방법이 있으며, 이는 꽤 엄격하고 많은 노력이 필요한 도구로, circuit에 대해 이미 상당히 구체적인 가설을 가지고 있을 때 사용하는 것이 가장 좋습니다. 실습을 진행하면서 우리는 이 스펙트럼을 따라 왼쪽에서 오른쪽으로 이동하게 됩니다.

## IOI 태스크

모델의 circuit을 역공학(reverse engineer)하려 할 때 첫 번째 단계는 우리가 *어떤* 능력을 역공학하고 싶은지 식별하는 것입니다. Indirect Object Identification(IOI)은 Redwood Research의 훌륭한 [Interpretability in the Wild](https://arxiv.org/abs/2211.00593) 논문에서 연구된 태스크입니다 (개요는 [Neel Nanda's interview with the authors](https://www.youtube.com/watch?v=gzwj0jWbvbo) 또는 [Kevin Wang's Twitter thread](https://threadreaderapp.com/thread/1587601532639494146.html)를 참조하십시오). 이 태스크는 "When Mary and John went to the store, John gave a drink to"와 같은 문장을 " John"이 아닌 " Mary"로 완성하는 것입니다.

논문에서 저자들은 이 능력을 수행하는 데 사용되는 7가지의 서로 다른 head 카테고리를 포함하여, 26개의 head로 구성된 circuit을 엄격하게 역공학했습니다. 그들이 발견한 circuit은 크게 세 부분으로 나뉩니다:

1. 문장에 어떤 이름들이 있는지 식별합니다.
2. 어떤 이름이 중복되었는지 식별합니다.
3. 중복되지 *않은* 이름을 예측합니다.

왜 이 태스크가 선택되었을까요? 저자들은 [video walkthrough of their paper](https://www.youtube.com/watch?v=gzwj0jWbvbo)에서 그 선택 이유에 대해 매우 훌륭한 설명을 제공하며, 이를 시청하시는 것을 권장합니다. 짧게 요약하자면, 몇 가지 이유는 다음과 같습니다:

* 이는 상당히 일반적인 문법 구조이므로, 모델이 (n-gram, 구두점, induction, 그리고 이보다 더 단순한 문법 구조와 같은 더 기본적인 것들을 모두 마친 후) 상당히 이른 시점에 이를 해결하기 위한 circuitry를 구축했을 것으로 기대할 수 있습니다.
* 측정이 쉽습니다: 모델은 항상 다른 어떤 token보다 IO 및 S token (즉, `" Mary"` 및 `" John"`)에 훨씬 더 높은 확률을 부여하며, 이는 특히 모델이 우리가 연구하는 circuit의 핵심 부분만 남기고 제거되기 시작할 때 더욱 두드러집니다. 따라서 이 두 token 사이의 logit 차이를 구하여, 모델이 태스크를 얼마나 잘 해결하는지에 대한 지표로 사용할 수 있습니다.
* 명확하고 잘 정의된 태스크이므로, 방대한 휴리스틱 뭉치를 암기하는 방식으로 해결될 가능성이 낮습니다 (예를 들어, "숫자 `n+1`이 `n` 다음에 올 것이라고 예측하라"와 같은 태스크와는 다릅니다. Neel이 비디오 가이드에서 언급했듯이, 후자는 처음 보이는 것보다 훨씬 더 성가시고 미묘합니다!).

용어 참고: `IO`은 간접 목적어(예시에서는 `" Mary"`)를, `S1`과 `S2`은 subject token의 두 인스턴스(즉, `" John"`)를, 그리고 `end`은 마지막 token인 `" to"`를 가리킵니다 (이 위치에서 예측을 수행하며, 이 지점 이후의 token은 고려하지 않기 때문입니다). 또한 때때로 subject token의 정체성(특정 첫 번째 또는 두 번째 인스턴스가 아닌)을 가리키기 위해 `S`를 사용하겠습니다.

## 추측 및 예측 기록하기

이 실습들을 진행하면서 추적해야 할 내용이 많습니다. 여러분은 transformerlens의 새로운 함수와 모듈, 모델에 인과적으로 개입하는 새로운 방법들을 접하게 될 것이며, 동시에 IOI task가 어떻게 수행되는지에 대한 이해를 쌓아가게 될 것입니다. 노트북은 탐색적인 성격(많은 시각화와 조사)으로 시작하여, IOI circuit에 대한 이해가 깊어짐에 따라 점차 더 기술적인 세부 사항, 정교한 분석, 그리고 논문의 결과 재현으로 이동합니다. 각 섹션의 주요 핵심 내용과 모델이 task를 어떻게 수행하는지에 대한 가설, 그리고 노트북이 갑자기 끝났을 때 이러한 가설들을 스스로 어떻게 테스트할 수 있을지에 대한 아이디어를 기록할 수 있도록, 실습을 진행하는 동안 옆에 문서나 노트 페이지를 준비하시는 것을 권장합니다.

진행 중 어느 시점에서든 매우 혼란스럽다면, circuit가 어떻게 작동하는지 설명하는 다이어그램이 포함된 아래 드롭다운 메뉴로 돌아오실 수 있습니다. 더 도움이 될 수 있는 직관적인 설명도 함께 제공됩니다. 하지만 이 내용들을 확인하기 전에 먼저 도움 없이 노트북을 끝까지 진행해 보시는 것을 추천합니다.

<details>
<summary>IOI circuit의 직관적인 설명</summary>

먼저, transformer가 작동하는 방식에 대한 비유로 시작해 보겠습니다 (이미 [my post](https://www.lesswrong.com/posts/euam65XjigaCJQkcN/an-analogy-for-understanding-transformers)을 읽으셨다면 이 부분은 건너뛰셔도 됩니다). 앞만 볼 수 있는 사람들이 한 줄로 서 있다고 상상해 보십시오. 각 사람은 가슴에 token이 하나씩 적혀 있으며, 이들의 목표는 자신의 앞에 있는 사람이 어떤 token을 가지고 있는지 알아내는 것입니다. 각 사람은 줄을 따라 뒤쪽으로만 질문을 전달할 수 있으며(앞으로는 불가능), 누구든 그 질문에 답하기 위해 질문을 한 사람에게 앞쪽으로 정보를 전달하는 방식을 선택할 수 있습니다. 이 경우, 문장은 `"When Mary and John went to the store, John gave a drink to Mary"` 입니다. 여러분은 `" to"` token을 가진 사람이며, 여러분의 목표는 자신의 앞에 있는 사람이 `" Mary"` token을 가지고 있다는 것을 알아내는 것입니다.

이 비유가 transformer와 어떻게 연결되는지 명확히 설명하겠습니다:
* 줄에 서 있는 각 사람은 residual stream의 벡터를 나타냅니다. 처음에는 자신의 token만 저장하지만, 질문을 던지고 답변을 받으면서(즉, 컴포넌트들이 residual stream에 기록함에 따라) 더 많은 정보를 축적합니다.
* attention head의 작동은 질문과 답변으로 표현됩니다:
    * 질문을 하는 사람은 destination token이며, 답변을 하는 사람들은 source token입니다.
    * 질문은 query 벡터입니다.
    * *누가 질문에 답할지를 결정하는* 정보는 key 벡터입니다.
    * *원래 질문자에게 다시 전달되는* 정보는 value 벡터입니다.

이제 이 비유에서 IOI circuit가 어떻게 작동하는지 설명하겠습니다. 각 불렛 포인트는 attention head의 한 클래스를 나타냅니다.

* 두 번째 `" John"` token을 가진 사람이 "다른 사람 중에 `" John"`이라는 이름을 가진 사람이 있나요?"라고 질문합니다. 그는 첫 번째 `" John"` token을 가진 사람으로부터 답변을 받으며, 그 사람은 자신의 위치 정보도 함께 알려줍니다. 이제 그는 `" John"`이 반복되었다는 것을 알게 되었고, 첫 번째 `" John"` token이 시퀀스의 4번째에 있다는 것을 알게 됩니다.
    * 이들이 바로 *Duplicate Token Heads* 입니다.
* 여러분은 "어떤 이름들이 반복되나요?"라고 질문하고, 두 번째 `" John"` token을 가진 사람으로부터 답변을 받습니다. 이제 여러분은 `" John"`이 반복되었다는 것과 첫 번째 `" John"` token이 어디에 있는지를 알게 됩니다.
    * 이들이 바로 *S-Inhibition Heads* 입니다.
* 여러분은 "`" John"`가 아니면서, 시퀀스의 4번째 위치에 있지 않은 이름을 가진 사람이 있나요?"라고 질문합니다. `" Mary"` token을 가진 사람으로부터 답변을 받게 되며, 그는 자신이 `" Mary"`이라는 이름을 가지고 있다고 알려줍니다. 여러분은 이것을 예측값으로 사용합니다.
    * 이들이 바로 *Name Mover Heads* 입니다.

이것은 circuit가 어떻게 작동하는지에 대한 훌륭한 1차적 이해입니다. 몇 가지 다른 특징들은 다음과 같습니다:

* 첫 번째 `" John"` 다음의 사람(`" went"`을 가진 사람)은 이전에 자신의 뒤에 있는 사람의 정체에 대해 질문했습니다. 따라서 그는 시퀀스의 4번째 사람이 `" John"` token을 가지고 있다는 것을 알며, 이는 그가 두 번째 `" John"` token을 가진 사람의 질문에도 답할 수 있음을 의미합니다. *(previous token heads / induction heads)*
    * 이것이 반드시 필요해 보이지 않을 수도 있지만, previous token heads / induction heads는 일반적으로 매우 유용한 기능이므로, 이 정보를 활용하는 것이 합리적입니다!
* 어떤 이유로 "`" John"`가 아니면서, 시퀀스의 4번째 위치에 있지 않은 이름을 가진 사람이 있나요?"라는 질문을 하는 것을 잊었다면, 이를 수행할 또 다른 기회가 있습니다.
    * 이들이 바로 *(Backup Name Mover Heads)* 입니다.
    * 이들의 존재는 부분적으로 transformer가 **dropout**과 함께 학습되기 때문일 수 있습니다. dropout은 모델이 무언가를 "잊게" 만들 수 있으므로, 해당 정보를 복구하기 위한 백업 방법이 있는 것이 중요합니다!
* 과잉 확신을 피하기 위해, 여러분은 "`" John"`가 아니면서, 시퀀스의 4번째 위치에 있지 않은 이름을 가진 사람이 있나요?"라는 질문을 한 번 더 던져서, 이 질문에서 얻은 응답을 ***반대로*** 예측(anti-predict)합니다. *(negative name mover heads)*
    * 네, 들리는 그대로 매우 이상한 작동 방식입니다! 저자들은 이 head들이 예측에 "헤징(hedge)"을 하여, 실수했을 때 높은 cross-entropy loss가 발생하는 것을 방지한다고 추측합니다.

</details>

<details>
<summary>다이어그램 1 (단순)</summary>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ioi-main-simple-a.png" width="1000">

</details>

<details>
<summary>다이어그램 2 (복잡)</summary>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ioi-main-full-d.png" width="1250">

</details>

## 내용 및 학습 목표

### 1️⃣ 모델 및 태스크 설정

이 섹션에서는 모델을 설정하고, IOI 태스크에서 모델의 성능을 분석하는 방법을 살펴봅니다. 또한 logit difference와 같은 도구를 사용하여 성능을 측정하는 방법을 배웁니다.

> ##### 학습 목표
>
> * IOI 태스크와 저자들이 왜 이 태스크를 연구하기로 선택했는지 이해합니다.
> * 이 태스크에 대한 모델의 성능을 입증하는 함수들을 구축합니다.

### 2️⃣ Logit Attribution

다음으로, 일부 컴포넌트 attribution, 즉 IOI 태스크에 대해 각 모델 컴포넌트의 중요도를 평가하는 단계로 넘어갑니다. 하지만 이러한 유형의 분석은 간접적인 효과가 아닌 컴포넌트의 직접적인 효과를 측정하는 것으로 제한됩니다. 간접적인 효과는 이후 섹션에서 측정하겠습니다.

> ##### 학습 목표
>
> * direct logit attribution을 수행하여 어떤 head들이 residual stream에 유의미하게 쓰고 있는지 파악합니다.
> * residual stream을 다양한 방식으로 분해하는 transformerlens의 여러 helper 함수 사용법을 배웁니다.

### 3️⃣ Activation Patching

이 섹션에서 사용할 두 가지 중요한 patching 도구 중 하나인 **activation patching**을 소개합니다. 이는 corrupted input으로 특정 컴포넌트를 patch했을 때, 이전에 정의한 태스크 메트릭의 변화를 측정함으로써 모델의 어떤 컴포넌트가 특정 태스크에 중요한지 발견하는 데 사용될 수 있습니다.

> ##### 학습 목표
>
> * activation patching의 개념과 사용 방법을 이해합니다.
>     * transformerlens의 activation patching helper 함수 중 일부를 처음부터(즉, hook을 사용하여) 구현합니다.
> * activation patching을 사용하여 중요한 정보가 저장되고 처리되는 residual stream의 layer 및 sequence position을 추적합니다.
> * 이 섹션을 마칠 때쯤에는 IOI circuit의 대략적인 스케치를 그릴 수 있어야 합니다.

### 4️⃣ Path Patching

다음으로, 모델 컴포넌트 사이의 특정 경로의 중요도를 조사하는 더 정교한 형태의 activation patching인 path patching으로 넘어갑니다. 이를 통해 circuit이 어떻게 작동하는지에 대한 더 정확한 그림을 얻을 수 있습니다.

> ##### 학습 목표
>
> * path patching의 개념과 activation patching과의 차이점을 이해합니다.
> * path patching을 처음부터(즉, hook을 사용하여) 구현합니다.
> * [IOI paper](https://arxiv.org/abs/2211.00593)의 여러 결과들을 재현합니다.

### 5️⃣ 전체 재현: Minimal Circuits 및 기타

마지막으로, IOI 논문의 다른 결과들을 재현하며 내용을 정리합니다. 여기에는 이전 분석에서 식별한 컴포넌트를 제외한 모델의 모든 컴포넌트를 제거하는 복잡한 형태의 ablation을 구현하고, 성능이 회복됨을 보여주는 과정이 포함됩니다. 이 섹션은 더 개방적이며 덜 구조화되어 있습니다.

> ##### 학습 목표
>
> * [IOI paper](https://arxiv.org/abs/2211.00593)의 다른 결과 대부분을 재현합니다.
> * 더 개방적이고 가이드가 적은 코딩을 연습합니다.

### ☆ 보너스 / 이상 현상 탐구

이 특정 circuit에 대해 제안하는 몇 가지 보너스 연습 문제와 캡스톤 프로젝트 및 논문 재현을 위한 아이디어로 마무리합니다.

> ##### 학습 목표
>
> * 모델의 다른 부분들(예: negative name mover heads 및 induction heads)을 탐구합니다.
> * 모델 circuit에 존재하는 미묘한 점들과, 초기 조사 후에 분명해 보이는 것보다 더 많은 부분이 circuit에 존재하는 경우가 많다는 사실을 이해합니다.
> * 논문에서 사용된 세 가지 정량적 기준인 **faithfulness**, **completeness**, **minimality**의 중요성을 이해합니다.

## 설정 코드

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0 einops jaxtyping git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import re
import sys
from functools import partial
from itertools import product
from pathlib import Path
from typing import Callable, Literal

import circuitsvis as cv
import einops
import numpy as np
import plotly.express as px
import torch as t
from IPython.display import HTML, display
from jaxtyping import Bool, Float, Int
from rich import print as rprint
from rich.table import Column, Table
from torch import Tensor
from tqdm.notebook import tqdm
from transformer_lens import ActivationCache, HookedTransformer, utils
from transformer_lens.components import MLP, Embed, LayerNorm, Unembed
from transformer_lens.hook_points import HookPoint

t.set_grad_enabled(False)
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part41_indirect_object_identification"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part41_indirect_object_identification.tests as tests
from plotly_utils import bar, imshow, line, scatter

MAIN = __name__ == "__main__"

# 1️⃣ 모델 및 태스크 설정

> ##### 학습 목표
>
> * IOI 태스크와 저자들이 왜 이 태스크를 연구하기로 선택했는지 이해합니다.
> * 이 태스크에 대한 모델의 성능을 입증하는 함수들을 구축합니다.

## 모델 로드하기

첫 번째 단계는 `HookedTransformer.from_pretrained`를 통해 12개의 layer와 80M parameter를 가진 transformer인 GPT-2 Small 모델을 로드하는 것입니다. 다양한 flag들은 모델의 출력은 유지하면서 내부 구조를 단순화하기 위한 설정들입니다.

In [ ]:
model = HookedTransformer.from_pretrained(
    "gpt2-small",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=True,
)

<details>
<summary><code>refactor_factored_attn_matrices</code>에 관한 참고 사항 (선택 사항)</summary>

이 인자는 모델의 실제 동작을 변경하지 않고, 모델 내의 행렬 $W_Q$, $W_K$, $W_V$ 및 $W_O$을 재정의함을 의미합니다.

예를 들어, $W_Q$와 $W_K$을 개별적으로 다루는 대신, 모델에서 실제로 사용해야 하는 유일한 행렬은 low-rank 행렬인 $W_Q W_K^T$이라는 점을 알고 있습니다 (여기서는 transformerlens의 코드 및 이 시리즈의 이전 연습 문제와 일치하는 우측 행렬 곱셈 관례를 사용하며, 이는 Anthropic의 Mathematical Frameworks 논문과는 일치하지 않음에 유의하십시오). 따라서 특이값 분해(singular value decomposition) $W_Q W_K^T = U S V^T$를 수행하면, $W_Q = U \sqrt{S}$과 $W_K = V \sqrt{S}$를 정의하여 대신 사용할 수 있음을 알 수 있습니다. 이는 $W_Q$와 $W_K$이 모두 일치하는 norm을 가진 orthogonal column을 가짐을 의미합니다. 여러분이 직접 이를 조사해 볼 수 있습니다 (예: 아래 코드를 사용). 이는 이제 key와 query 사이에 명백한 비대칭성이 없으므로, 논쟁의 여지가 있겠으나 더 해석 가능한 설정입니다.

이 factorization에서 bias가 처리되는 방식에 일부 까다로운 점이 있으며, 이 때문에 위의 설명이 절대적으로 적용되지는 않습니다 (자세한 내용은 문서를 참조하십시오).

```python
# Show column norms are the same (except first few, for fiddly bias reasons)
line([model.W_Q[0, 0].pow(2).sum(0), model.W_K[0, 0].pow(2).sum(0)])
# Show columns are orthogonal (except first few, again)
W_Q_dot_products = einops.einsum(
    model.W_Q[0, 0], model.W_Q[0, 0], "d_model d_head_1, d_model d_head_2 -> d_head_1 d_head_2"
)
imshow(W_Q_dot_products)
```

유사한 방식으로, $W_{OV} = W_V W_O = U S V^T$이므로 $W_V = U S$와 $W_O = V^T$을 정의할 수 있습니다. 이는 $W_O$이 단순한 rotation이며 norm을 변경하지 않으므로, $z$가 head의 결과와 동일한 norm을 갖게 되어 논쟁의 여지가 있겠으나 더 해석 가능한 설정입니다.
</details>

<details>
<summary><code>fold_ln</code>, <code>center_unembed</code> 및 <code>center_writing_weights</code>에 관한 참고 사항 (선택 사항)</summary>

설명은 링크 [here](https://github.com/neelnanda-io/TransformerLens/blob/main/further_comments.md#what-is-layernorm-folding-fold_ln)을 참조하십시오.
</details>

다음 단계는 모델이 *실제로* 이 작업을 수행할 수 있는지 확인하는 것입니다! 여기서는 `utils.test_prompt`를 사용하며, 모델이 John보다 Mary를 예측하는 데 훨씬 더 능숙하다는 것을 알 수 있습니다!

<details><summary>참고 사항</summary>

참고: 더 신중하게 접근한다면, 다양한 prompt 범위에서 모델을 실행하고 평균 성능을 측정해야 합니다. 네 번째 섹션에서(논문의 결과 중 일부를 재현하고 더 엄격한 접근 방식을 취할 때) 이와 같은 작업을 더 많이 수행할 예정입니다.

`prepend_bos`는 prompt의 시작 부분에 BOS (beginning of sequence)를 추가하는 flag입니다. GPT-2는 이것으로 학습되지 않았지만, 첫 번째 token이 특이하게 처리되기 때문에 이를 추가하면 모델의 동작이 더 안정적이 되는 경우가 많습니다.
</details>

In [ ]:
# Here is where we test on a single prompt
# Result: 70% probability on Mary, as we expect

example_prompt = "After John and Mary went to the store, John gave a bottle of milk to"
example_answer = " Mary"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

이제 모델을 실행할 참조 프롬프트를 찾고자 합니다. 우리의 최종 목표는 이러한 동작이 일반적으로 어떻게 이루어지는지 역공학하는 것이지만, mechanistic interpretability를 시작하는 가장 좋은 방법은 종종 구체적인 예시에 집중하여 세부적으로 이해한 *다음*, 분석 내용을 확장하여 일반화되는지 확인하는 것입니다. 섹션 3에서는 논문 저자들이 사용한 것과 유사한 데이터셋으로 작업하겠지만, 초기 조사를 수행하는 단계라면 아마 이것이 가장 먼저 선택할 방법은 아닐 것입니다.

우리는 이 태스크의 4가지 인스턴스에 대해 모델을 실행할 것이며, 각 프롬프트는 두 번씩 제공됩니다. 하나는 첫 번째 이름이 간접 목적어인 경우이고, 다른 하나는 두 번째 이름이 간접 목적어인 경우입니다. 작업을 더 쉽게 만들기 위해, 단일 token 이름과 동일한 token 위치에 해당하는 이름들이 포함된 프롬프트를 신중하게 선택하겠습니다.

<details><summary>tokenization에 관한 부연 설명</summary>

우리는 임의의 텍스트를 입력받을 수 있는 모델을 원하지만, 모델은 고정된 vocabulary를 가져야 합니다. 따라서 해결책은 **tokens**의 vocabulary를 정의하고, 임의의 텍스트를 결정론적으로 token으로 나누는 것입니다. Token은 기본적으로 서브워드(subwords)이며, 가장 빈번하게 나타나는 부분 문자열을 찾아 결정됩니다. 이는 token의 길이와 빈도가 매우 다양하다는 것을 의미합니다!

Token은 매우 골치 아픈 문제이며, 언어 모델을 역공학할 때 가장 짜증 나는 요소 중 하나입니다... 이름마다 token의 수가 다를 수 있고, 프롬프트마다 관련 token의 위치가 다를 수 있으며, 프롬프트마다 전체 token 수가 다를 수 있습니다. 언어 모델은 종종 초기 레이어에서 입력된 token을 더 합리적인 내부 형식으로 변환하는 데(그리고 이후 레이어에서 그 반대 작업을 수행하는 데) 상당한 양의 파라미터를 할당합니다. 탐색적 분석을 수행할 때는 가능한 한 tokenization에 대해 생각하는 것을 정말로 피하고 싶을 것입니다(물론 나중에 분석 내용을 구체화하고 엄격하게 만들 때는 중요합니다!). HookedTransformer는 token을 처리하기 위한 여러 헬퍼 메서드를 제공합니다: `to_tokens, to_string, to_str_tokens, to_single_token, get_token_position`

**연습 문제:** 모델이 서로 다른 문자열을 어떻게 tokenize하는지 탐색하기 위해 `model.to_str_tokens`를 사용하는 것을 추천합니다. 특히, 시작 부분에 공백을 추가하거나 제거하거나, 대소문자를 변경해 보십시오. 이러한 변화가 tokenization을 바꿉니다!</details>

In [ ]:
prompt_format = [
    "When John and Mary went to the shops,{} gave the bag to",
    "When Tom and James went to the park,{} gave the ball to",
    "When Dan and Sid went to the shops,{} gave an apple to",
    "After Martin and Amy went to the park,{} gave a drink to",
]
name_pairs = [
    (" Mary", " John"),
    (" Tom", " James"),
    (" Dan", " Sid"),
    (" Martin", " Amy"),
]

# Define 8 prompts, in 4 groups of 2 (with adjacent prompts having answers swapped)
prompts = [prompt.format(name) for (prompt, names) in zip(prompt_format, name_pairs) for name in names[::-1]]
# Define the answers for each prompt, in the form (correct, incorrect)
answers = [names[::i] for names in name_pairs for i in (1, -1)]
# Define the answer tokens (same shape as the answers)
answer_tokens = t.concat([model.to_tokens(names, prepend_bos=False).T for names in answers])

rprint(prompts)
rprint(answers)
rprint(answer_tokens)

table = Table("Prompt", "Correct", "Incorrect", title="Prompts & Answers:")

for prompt, answer in zip(prompts, answers):
    table.add_row(prompt, repr(answer[0]), repr(answer[1]))

rprint(table)

<details>
<summary>참고 - <code>rich</code> 라이브러리</summary>

위의 출력물들은 내용을 보기 좋은 형식으로 출력해 주는 재미있는 라이브러리인 `rich`로 생성되었습니다. 이 라이브러리는 `rich.table.Table`과 같은 함수들을 가지고 있으며, 사용법이 매우 간단하면서도 때때로 유용한 시각적으로 명확한 출력물을 만들어낼 수 있습니다.

또한 `style` 파라미터와 함께 `rich.table.Column` 인자를 사용하여 테이블의 열에 색상을 입힐 수도 있습니다:

```python
cols = [
    "Prompt",
    Column("Correct", style="rgb(0,200,0) bold"),
    Column("Incorrect", style="rgb(255,0,0) bold"),
]
table = Table(*cols, title="Prompts & Answers:")

for prompt, answer in zip(prompts, answers):
    table.add_row(prompt, repr(answer[0]), repr(answer[1]))

rprint(table)
```
</details>

이제 이 프롬프트들로 모델을 실행하고, 나중에 분석하기 위해 logit과 모든 내부 activation의 cache를 모두 얻기 위해 `run_with_cache`를 사용합니다.

In [ ]:
tokens = model.to_tokens(prompts, prepend_bos=True)
# Move the tokens to the GPU
tokens = tokens.to(device)
# Run the model and cache all activations
original_logits, cache = model.run_with_cache(tokens)

나중에 다양한 intervention을 수행했을 때 모델 성능이 어떻게 달라지는지 평가할 예정이므로, 모델 성능을 측정할 수 있는 metric이 있으면 유용합니다. 여기서 사용할 metric은 **logit difference**이며, 이는 간접 목적어의 이름과 주어의 이름 사이의 logit 차이입니다 (예: `logit(Mary) - logit(John)`).

### 연습 문제 - 성능 평가 함수 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> It's important to understand exactly what this function is computing, and why it matters.
> ```

이 함수는 모델의 logit 출력(shape `(batch, seq, d_vocab)`)과 정답 token 배열(shape `(batch, 2)`, 각 시퀀스에 대해 정답과 오답의 token id를 각각 포함)을 입력으로 받아, 위에서 설명한 logit 차이를 반환해야 합니다. 만약 `per_prompt`이 False라면 batch 차원에 대해 평균을 내어 반환하고, 그렇지 않다면 길이가 `batch`인 배열을 반환해야 합니다.

In [ ]:
def logits_to_ave_logit_diff(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Int[Tensor, "batch 2"] = answer_tokens,
    per_prompt: bool = False,
) -> Float[Tensor, "*batch"]:
    """
    Returns logit difference between the correct and incorrect answer.

    If per_prompt=True, return the array of differences rather than the average.
    """
    raise NotImplementedError()


tests.test_logits_to_ave_logit_diff(logits_to_ave_logit_diff)

original_per_prompt_diff = logits_to_ave_logit_diff(original_logits, answer_tokens, per_prompt=True)
print("Per prompt logit difference:", original_per_prompt_diff)
original_average_logit_diff = logits_to_ave_logit_diff(original_logits, answer_tokens)
print("Average logit difference:", original_average_logit_diff)

cols = [
    "Prompt",
    Column("Correct", style="rgb(0,200,0) bold"),
    Column("Incorrect", style="rgb(255,0,0) bold"),
    Column("Logit Difference", style="bold"),
]
table = Table(*cols, title="Logit differences")

for prompt, answer, logit_diff in zip(prompts, answers, original_per_prompt_diff):
    table.add_row(prompt, repr(answer[0]), repr(answer[1]), f"{logit_diff.item():.3f}")

rprint(table)

<details><summary>솔루션</summary>

```python
def logits_to_ave_logit_diff(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Int[Tensor, "batch 2"] = answer_tokens,
    per_prompt: bool = False,
) -> Float[Tensor, "*batch"]:
    """
    Returns logit difference between the correct and incorrect answer.

    If per_prompt=True, return the array of differences rather than the average.
    """
    # Only the final logits are relevant for the answer
    final_logits: Float[Tensor, "batch d_vocab"] = logits[:, -1, :]
    # Get the logits corresponding to the indirect object / subject tokens respectively
    answer_logits: Float[Tensor, "batch 2"] = final_logits.gather(dim=-1, index=answer_tokens)
    # Find logit difference
    correct_logits, incorrect_logits = answer_logits.unbind(dim=-1)
    answer_logit_diff = correct_logits - incorrect_logits
    return answer_logit_diff if per_prompt else answer_logit_diff.mean()
```
</details>

## 실제로 어떤 일이 일어나고 있는지 브레인스토밍하기

실험을 직접 실행하기 전에, 문제가 되는 동작이 transformer 내에서 어떻게 구현될 수 있을지 실제로 추론하며 시간을 보내는 것이 유용할 때가 많습니다. **이 과정은 선택 사항이며, transformer가 무엇인지 그리고 어떻게 작동하는지에 대해 이미 어느 정도 이해하고 계신 분들이 이 섹션을 통해 가장 많은 도움을 얻으실 수 있을 것입니다!**

반드시 이렇게 해야 하는 것은 아니며, 탐색 후에 가설을 세우는 것도 합리적입니다. 하지만 무엇을 발견하게 될지에 대한 어느 정도의 근거를 가지고 탐색하고 결과를 해석하는 것이 더 쉽다고 생각합니다. 이번 사례의 경우, 저는 이미 답을 알고 있기 때문에 어느 정도 편법을 쓰고 있지만, 추론하는 과정을 시뮬레이션하려고 노력하고 있습니다!

종종 여러분의 가설이 어떤 면에서 틀리거나 완전히 빗나갈 수 있다는 점에 유의하십시오. 우리는 여기서 과학을 하고 있으며, 목표는 모델이 *실제로* 어떻게 작동하는지 이해하고 진실된 믿음을 형성하는 것입니다! 여기서 주의 깊게 살펴봐야 할 양 극단의 두 가지 함정이 있습니다:
* 혼란: 가설이 전혀 없는 상태에서 많은 데이터를 얻었지만 그것으로 무엇을 해야 할지 몰라 갈팡질팡하는 경우입니다.
* 독단: 잘못된 가설에 과도하게 확신하여 현실이 그와 상반됨에도 불구하고 이를 놓지 않으려 하거나, 가설을 부정할 수 있는 실험을 실행하는 것을 꺼리는 경우입니다.

**연습 문제:** 이 동작이 transformer에서 어떻게 구현될 수 있을지 상상하며 생각하는 시간을 가져보십시오. 제 생각을 읽기 전에 스스로 먼저 생각해보시기 바랍니다!

<details> <summary>(*) <b>나의 추론</b></summary>

<h3>브레인스토밍:</h3>

그렇다면, 이 작업에서 어려운 점은 무엇일까요? 첫 번째 프롬프트의 구체적인 예시인 `"When John and Mary went to the shops, John gave the bag to" -> " Mary"`에 집중해 보겠습니다.

좋은 시작점은 아주 작은 모델, 예를 들어 <a href="https://transformer-circuits.pub/2021/framework/index.html">1L Attn-Only 모델</a>이 이 작업을 수행할 수 있을지 생각해보는 것입니다. 제 생각에 답은 '아니오'입니다! attention은 주변을 살피거나 정보를 복사하는 원시적인 연산에 매우 능숙합니다. 아주 작은 모델이라도 `to`에서 이름을 찾고 그 이름들이 다음에 올 것이라고 예측하는 것(예: skip trigram " John...to -> John")은 알아낼 수 있다고 믿습니다. 하지만 이전 이름들이 각각 <i>얼마나</i> 많이 있는지 판단하는 것은 훨씬 더 어렵습니다. John의 각 복사본에 attention을 주는 것은 단일 John token에 attention을 주는 것과 정확히 똑같이 보일 것이기 때문입니다. 따라서 " to" token에서 이를 알아내는 것은 꽤 어려울 것입니다!

이러한 대칭성을 깨뜨릴 수 있는 자연스러운 지점은 두 번째 `" John"` token입니다. <i>현재</i> token의 이전 복사본이 있는지 판단하는 것은 훨씬 더 쉬운 작업일 것입니다. 따라서 두 번째 `" John"` token에서 중복 token을 감지하는 head가 있고, 그 다음 다른 head가 그 정보를 두 번째 `" John` token에서 `" to"` token으로 이동시킬 것이라고 예상할 수 있습니다.

그 후 모델은 `" Mary"`를 예측하고 <i>아니라</i> `" John"`를 예측하는 법을 배워야 합니다. 이를 수행하는 두 가지 자연스러운 방법이 보입니다:
1. 앞에 나오는 모든 이름을 감지하여 이 정보를 " to"로 이동시킨 다음, 중복 token feature에 해당하는 모든 이름을 삭제합니다. 벡터를 정확하게 상쇄시키는 것은 어렵기 때문에 이는 non-linearity를 통해 수행하는 것이 더 쉬워 보이며, 따라서 MLP layer가 residual stream의 `" John"` 방향을 삭제한다고 상상할 수 있습니다.
2. 이전의 모든 이름에 attention을 주지만, 중복 token feature가 <i>억제하여</i> 특정 이름에 attention을 주지 못하게 하는 head를 갖는 것입니다. 이렇게 하면 Mary에게만 attention을 주게 됩니다. 그리고 이 head의 출력이 logit으로 매핑됩니다.

<details>
<summary>스포일러 - 이 두 가지 중 어느 것이 맞을까요</summary>

두 번째 방법이 맞습니다.
</details>

<h3>실험 아이디어</h3>

이 두 가지를 구분할 수 있는 테스트는 모델의 어떤 구성 요소가 logit에 직접적으로 더해지는지 살펴보는 것입니다. 만약 `" Mary"`에는 attention을 주고 `" John"`에는 둘 다 주지 않는 attention head들이 주를 이룬다면 가설 2일 가능성이 높고, 주로 MLP라면 가설 1일 가능성이 높습니다.

또한 `" John"`에서 `" John"`로 attention을 주고, 그 출력이 다른 head와의 V-Composition을 통해 `" to"` token으로 이동하는 head들을 찾음으로써 중복 token head들을 식별할 수 있을 것입니다. (스포일러: 실제로는 그보다 더 복잡합니다!)

위의 모든 추론은 매우 단순화된 것이며 실제 모델에서는 쉽게 깨질 수 있다는 점에 유의하십시오! 이 circuit을 사용할지 말지를 결정하는 모델의 상당한 부분이 존재할 것이며(예를 들어 <i>다음</i> 문장의 시작 부분에 무엇이 올지 판단할 때는 중복된 이름을 억제하고 싶지 않을 것입니다), 최종 출력 직전에 "후처리"를 수행하는 모델 끝부분의 일부가 있을 수도 있습니다. 하지만 이는 무엇이 일어나고 있는지 생각하기 위한 좋은 시작점입니다.

</details>

# 2️⃣ Logit Attribution

> ##### 학습 목표
>
> * 어떤 head가 residual stream에 유의미하게 기록하고 있는지 파악하기 위해 direct logit attribution을 수행합니다.
> * residual stream을 다양한 방식으로 분해하는 여러 transformerlens helper 함수들의 사용법을 익힙니다.

## Direct Logit Attribution

모델에서 가장 이해하기 쉬운 부분은 출력입니다. 이는 모델이 최적화하도록 훈련된 대상이므로, 항상 직접적으로 해석할 수 있습니다! circuit을 역공학하는 올바른 접근 방식은 종종 끝에서 시작하여 모델이 어떻게 정답을 생성하는지 이해하고, 그 다음 역방향으로 추적하는 것입니다 (balanced bracket classifier 과제를 수행하셨다면 이를 경험하셨을 것이며, 실제로 그렇게 하셨다면 이 섹션이 매우 익숙할 것이므로 가볍게 훑어보셔도 좋습니다). 이를 위해 사용되는 주요 기술을 **direct logit attribution**이라고 합니다.

**배경:** transformer의 중심 객체는 **residual stream**입니다. 이는 각 layer의 출력과 원래의 token 및 positional embedding의 합입니다. 중요한 점은, residual stream의 모든 선형 함수가 transformer의 각 layer가 기여한 값들의 합으로 완벽하게 분해될 수 있다는 것입니다. 더 나아가, 각 attention layer의 출력은 각 head의 출력 합으로 분해될 수 있으며 (자세한 내용은 [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) 참조), 각 MLP layer의 출력은 각 neuron의 출력 합(및 각 layer의 bias 항)으로 분해될 수 있습니다.

모델의 logit은 `logits=Unembed(LayerNorm(final_residual_stream))` 입니다. Unembed는 선형 맵이고 LayerNorm은 대략적으로 선형 맵이므로, 우리는 logit을 각 구성 요소의 기여도 합으로 분해하여 어떤 구성 요소가 정답 token의 logit에 가장 많이 기여하는지 살펴볼 수 있습니다! 이를 **direct logit attribution**이라고 합니다. 여기서는 logit 차이에 대한 direct attribution을 살펴보겠습니다!

### logit difference의 배경 및 동기

logit difference는 실제로 매우 훌륭하고 우아한 metric이며, 특히 Indirect Object Identification 설정에서 매우 유용한 측면입니다. 일반적으로 모델의 출력을 해석하는 두 가지 자연스러운 방법이 있습니다. 바로 output logit 또는 output log probability(또는 probability)입니다.

앞서 언급했듯이 logit은 훨씬 더 다루기 쉽고 이해하기 좋습니다. 하지만 모델은 cross-entropy loss(정답 token의 log probability의 평균)를 최적화하도록 학습됩니다. 이는 모델이 logit을 직접적으로 최적화하지 않음을 의미하며, 실제로 모델이 모든 logit에 임의의 상수를 더하더라도 log probability는 변하지 않습니다.

하지만 우리는 다음과 같은 식을 가집니다:

```
log_probs == logits.log_softmax(dim=-1) == logits - logsumexp(logits)
```

그리고 이들은 상수의 차이만 있기 때문에, 다음과 같은 결과가 나옵니다:

```
log_probs(" Mary") - log_probs(" John") = logits(" Mary") - logits(" John")
```

- 임의의 상수를 더하는 능력이 서로 상쇄됩니다!

<details>
<summary>기술적 세부 사항 (이 등가성이 명확하지 않은 경우)</summary>

$\vec{\textbf{x}}$를 logit, $\vec{\textbf{L}}$를 log prob, $\vec{\textbf{p}}$를 prob라고 합시다. 그러면 다음과 같은 관계가 성립합니다:

$$
p_i = \operatorname{softmax}(\vec{\textbf{x}})_i = \frac{e^{x_i}}{\sum_{i=1}^n e^{x_i}}
$$

그리고:

$$
L_i = \log p_i
$$

이들을 결합하면 다음과 같습니다:

$$
L_i = \log \frac{e^{x_i}}{\sum_{j=1}^n e^{x_j}} = x_i - \log \sum_{j=1}^n e^{x_j}
$$

우변의 합산 항(sum term)이 모든 $i$에 대해 동일하다는 점에 주목하십시오. 따라서 다음과 같은 결과를 얻습니다:

$$
L_i - L_j = x_i - x_j
$$

다시 말해, logit diff $x_i - x_j$는 log prob diff와 동일합니다. 이는 logit diff를 metric으로 선택하게 된 동기가 됩니다 (모델이 정답 token의 log prob는 크게, 나머지 log prob는 작게 만들도록 직접 학습되기 때문입니다).

</details>

나아가, 이 metric은 우리가 관심을 가지는 정확한 능력, 즉 *어떤* 이름이 Indirect Object인지 파악하는 능력을 분리하는 데 도움을 줍니다. 이 태스크에는 관사(the)나 대명사(her) 또는 이름을 반환할지 결정하는 것, 문장에서 다음에 사람이 와야 한다는 것을 인식하는 것 등 많은 다른 구성 요소가 있습니다. logit difference를 취함으로써 우리는 이러한 모든 요소를 제어할 수 있습니다.

우리의 metric은 각 prompt가 가능한 모든 indirect object에 대해 두 번씩 반복되므로 더욱 정교해집니다. 이는 모델이 Mary보다 John이 더 빈번한 token이라고 학습하는 것과 같은 무관한 동작을 제어합니다 (실제로 이런 일이 일어납니다! 최종 layernorm bias가 Mary logit에 비해 John logit을 1만큼 증가시킵니다). 이를 처리하는 또 다른 방법은 이름이 무작위로 선택된 충분히 큰 데이터셋을 사용하여 이러한 효과가 평균화되도록 하는 것이며, 이는 섹션 3에서 다룰 내용입니다.

<details> <summary>LayerNorm 무시하기</summary>

LayerNorm은 transformer가 사용하는 BatchNorm과 유사한 정규화 기법입니다 (BatchNorm보다 대규모 병렬 처리에 더 적합합니다). transformer layer가 residual stream에서 정보를 읽을 때마다, 각 위치의 벡터를 정규화하기 위해 LayerNorm을 적용하고 (평균을 0으로 설정하는 이동과 분산을 1로 설정하는 스케일링을 수행), 그 다음 학습된 가중치와 편향 벡터를 적용하여 정규화된 벡터를 스케일링하고 이동시킵니다. 이는 스케일링 단계를 제외하면 *거의* 선형 사상(linear map)입니다. 스케일링 단계에서는 벡터의 norm으로 나누는데, norm은 선형 함수가 아니기 때문입니다. (모델을 로드할 때 사용하는 `fold_ln` 플래그는 모든 선형 부분을 분리해 냅니다).

하지만 스케일 인자를 고정한다면, LayerNorm은 완전히 선형적이 됩니다. 그리고 residual stream의 스케일은 스트림의 *모든* 구성 요소의 함수인 전역적 속성인 반면, 실제로는 특정 구성 요소와 관련된 방향은 보통 몇 가지뿐이므로, 실제적으로 이는 허용 가능한 근사치입니다. 따라서 direct logit attribution을 수행할 때, `cache`의 `apply_ln` 플래그를 사용하여 각 상수에 전역 LayerNorm 스케일 인자를 적용합니다. LayerNorm에 대한 자세한 내용은 [my clean GPT-2 implementation](https://colab.research.google.com/github/neelnanda-io/TransformerLens/blob/clean-transformer-demo/Clean_Transformer_Demo.ipynb#scrollTo=Clean_Transformer_Implementation)을 참조하십시오.
</details>

### Logit diff directions

***output logit을 얻는 것은 residual stream의 특정 방향으로 projection하는 것과 동일하며, logit diff를 얻는 경우에도 마찬가지입니다.***

<details>
<summary>이 문장의 의미가 명확하지 않다면, 이 드롭다운을 읽어주시기 바랍니다.</summary>

단일 시퀀스와 그 시퀀스 내의 특정 위치에 대한 residual stream의 최종 값이 $x$라고 가정해 보겠습니다 (즉, $x$는 길이가 $d_{model}$인 벡터입니다). 이때 (layernorm은 무시합니다 - 왜 이것이 가능한지는 위의 내용을 참고하십시오), 우리는 shape이 $(d_{model}, d_{vocab})$인 unembedding matrix $W_U$를 곱하여 logits를 얻습니다:

$$
\text{output} = x^T W_U
$$

이제 우리가 원하는 것은 logit diff이며, 이는 $\text{output}_{IO} - \text{output}_{S}$ (indirect object와 subject의 logits 차이)입니다. 이를 다음과 같이 쓸 수 있습니다:

$$
\text{logit diff} = (x^T W_U)_{IO} - (x^T W_U)_{S} = x^T (u_{IO} - u_{S})
$$

여기서 $u_{IO}$과 $u_S$는 각각 indirect object와 subject token에 해당하는 **unembedding matrix** $W_U$의 **열(column)**들입니다.

요약하자면, 우리는 logit diff를 residual stream의 벡터와 상수 벡터(모델의 unembedding matrix의 함수) 사이의 dot product로 표현했습니다. 우리는 이 벡터 $u_{IO} - u_{S}$를 **logit difference direction**이라고 부릅니다 (이 벡터가 *"가장 큰 logit 차이가 발생하는 방향을 가리키기"* 때문입니다). 다른 방식으로 말하면, $x$가 고정된 크기를 가진 벡터일 때, 이 벡터가 $u_{IO} - u_{S}$ 벡터와 같은 방향을 가리킬 때 logit 차이가 최대화됩니다. 여기서는 "projection"이라는 용어를 "dot product"와 동의어로 사용합니다.

(만약 balanced / unbalanced bracket 문자열을 처리하는 transformer를 해석하는 실습을 완료하셨다면, 이것은 기본적으로 동일한 원리입니다. 유일한 차이점은 여기서는 단순한 분류 `{balanced, unbalanced}`보다 훨씬 더 큰 unembedding vocabulary를 가지고 있다는 점입니다. 하지만 우리는 IO와 S에 대한 모델의 예측을 비교하는 데에만 관심이 있고, 이 두 token의 logits는 보통 다른 대부분의 token보다 크기 때문에 이 방법은 여전히 타당합니다).
</details>

우리는 `model.tokens_to_residual_directions`를 사용하여 정답 token들을 해당 방향으로 매핑한 다음, 이를 각 batch에 대한 logit difference direction으로 변환합니다.

In [ ]:
answer_residual_directions = model.tokens_to_residual_directions(answer_tokens)  # [batch 2 d_model]
print("Answer residual directions shape:", answer_residual_directions.shape)

correct_residual_directions, incorrect_residual_directions = answer_residual_directions.unbind(dim=1)
logit_diff_directions = correct_residual_directions - incorrect_residual_directions  # [batch d_model]
print("Logit difference directions shape:", logit_diff_directions.shape)

이것이 제대로 작동하는지 확인하기 위해, 캐시된 prompt의 최종 residual stream(LayerNorm scaling 적용 후)에 이를 적용하여 동일한 답을 얻는지 확인할 수 있습니다.

<details> <summary>기술적 세부 사항</summary>

`logits = Unembed(LayerNorm(final_residual_stream))`, 따라서 엄밀하게는 단순히 variance 1 scaling뿐만 아니라, centering과 layernorm의 학습된 translation 및 scaling을 고려해야 합니다.

centering은 preprocessing 플래그 `center_writing_weights`로 처리되며, 이는 residual stream에 기록되는 모든 weight matrix가 평균 0을 갖도록 보장합니다.

학습된 scaling은 `W_U_fold = layer_norm.weights[:, None] * unembed.W_U`을 통해 unembedding weights `model.unembed.W_U`에 통합됩니다.

학습된 translation은 logit에 추가되는 bias인 `model.unembed.b_U`로 통합됩니다 (GPT-2는 기존의 `b_U`로 학습되지 않았음에 유의하십시오). 이는 대략적으로 unigram 통계를 나타냅니다. 하지만 각 prompt가 이름의 순서만 바뀐 채로 두 번씩 등장하므로, 이는 완벽하게 상쇄되어 무시할 수 있습니다.

layernorm scaling을 사용하는 대신 `cache["ln_final.hook_normalised"]`를 직접 연구할 수도 있음에 유의하십시오.

</details>

아래 코드는 다음과 같은 작업을 수행합니다:

* `cache` 객체(이미 위에서 정의했을 것입니다)로부터 최종 residual stream 값을 가져옵니다.
* 이 값들에 layernorm scaling을 적용합니다.
    * 이는 `cache.apply_to_ln_stack`에 의해 수행됩니다. 이 유용한 함수는 residual stream 값들의 스택(예: batch, 또는 컴포넌트로 분해된 residual stream)을 받아 이를 특정 layer의 입력으로 취급하고, 해당 layer의 layer norm scaling을 적용합니다.
    * 여기서 키워드 인자들은 입력값이 마지막 sequence position의 residual stream 값이며, 모델의 최종 layernorm을 적용하고자 함을 나타냅니다.
* 이를 unembedding 방향(위에서 `logit_diff_directions`로 이미 정의했습니다)으로 project 합니다.

In [ ]:
# Cache syntax: resid_post is the residual stream at the end of the layer, -1 gets the final layer.
# The general syntax is [activation_name, layer_index, sub_layer_type].
final_residual_stream: Float[Tensor, "batch seq d_model"] = cache["resid_post", -1]
print(f"Final residual stream shape: {final_residual_stream.shape}")
final_token_residual_stream: Float[Tensor, "batch d_model"] = final_residual_stream[:, -1, :]

# Apply LayerNorm scaling (to just the final sequence position)
# pos_slice is the subset of the positions we take - here the final token of each prompt
scaled_final_token_residual_stream = cache.apply_ln_to_stack(final_token_residual_stream, layer=-1, pos_slice=-1)

average_logit_diff = einops.einsum(
    scaled_final_token_residual_stream, logit_diff_directions, "batch d_model, batch d_model ->"
) / len(prompts)

print(f"Calculated average logit diff: {average_logit_diff:.10f}")
print(f"Original logit difference:     {original_average_logit_diff:.10f}")

t.testing.assert_close(average_logit_diff, original_average_logit_diff)

## Logit Lens

이제 residual stream을 분해할 수 있습니다! 먼저 [**logit lens**](https://www.alignmentforum.org/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)라고 불리는 기법을 적용합니다. 이 기법은 각 layer 이후의 residual stream을 살펴보고, 그로부터 logit difference를 계산합니다. 이는 이후의 모든 layer를 삭제했을 때 어떤 일이 일어나는지를 시뮬레이션합니다.

### 연습 문제 - `residual_stack_to_logit_diff` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> Again, make sure you understand what the output of this function represents.
> ```

이 함수는 바로 위의 코드와 매우 유사해야 합니다. `residual_stack` 는 마지막 sequence 위치의 residual stream 값들을 포함하는 `(..., batch, d_model)` 모양의 tensor입니다. 이 값들에 최종 layernorm을 적용한 다음, logit difference 방향으로 project 해야 합니다.

In [ ]:
def residual_stack_to_logit_diff(
    residual_stack: Float[Tensor, "... batch d_model"],
    cache: ActivationCache,
    logit_diff_directions: Float[Tensor, "batch d_model"] = logit_diff_directions,
) -> Float[Tensor, "..."]:
    """
    Gets the avg logit difference between the correct and incorrect answer for a given stack of
    components in the residual stream.
    """
    raise NotImplementedError()


# Test function by checking that it gives the same result as the original logit difference
t.testing.assert_close(residual_stack_to_logit_diff(final_token_residual_stream, cache), original_average_logit_diff)

<details><summary>솔루션</summary>

```python
def residual_stack_to_logit_diff(
    residual_stack: Float[Tensor, "... batch d_model"],
    cache: ActivationCache,
    logit_diff_directions: Float[Tensor, "batch d_model"] = logit_diff_directions,
) -> Float[Tensor, "..."]:
    """
    Gets the avg logit difference between the correct and incorrect answer for a given stack of
    components in the residual stream.
    """
    batch_size = residual_stack.size(-2)
    scaled_residual_stack = cache.apply_ln_to_stack(residual_stack, layer=-1, pos_slice=-1)
    return (
        einops.einsum(scaled_residual_stack, logit_diff_directions, "... batch d_model, batch d_model -> ...")
        / batch_size
    )
```
</details>

솔루션을 구했다면, 결과를 plot 할 수 있습니다.

<details> <summary><code>accumulated_resid</code></summary>에 대한 세부 사항입니다.

아래 plot의 범례입니다: `n_pre`는 layer n 시작 지점의 residual stream을 의미하며, `n_mid`은 layer n의 attention 부분 이후의 residual stream을 의미합니다 (`n_post`은 `n+1_pre`과 동일하므로 포함되지 않았습니다).

* `layer`는 residual stream을 입력하는 layer입니다 (이는 *어떤* layer norm scaling factor를 원하는지 식별하는 데 사용됩니다).
* `incl_mid`은 layer 중간, 즉 attention 이후 및 MLP 이전의 residual stream을 포함할지 여부입니다.
* `pos_slice`은 사용된 position의 subset입니다. 구문(syntax)에 대한 자세한 내용은 `utils.Slice`를 참조하십시오.
* `return_labels`은 반환된 각 component에 대한 label을 함께 반환할지 여부입니다 (plotting 시 유용합니다).
</details>

In [ ]:
accumulated_residual, labels = cache.accumulated_resid(layer=-1, incl_mid=True, pos_slice=-1, return_labels=True)
# accumulated_residual has shape (component, batch, d_model)

logit_lens_logit_diffs: Float[Tensor, " component"] = residual_stack_to_logit_diff(accumulated_residual, cache)

line(
    logit_lens_logit_diffs,
    hovermode="x unified",
    title="Logit Difference From Accumulated Residual Stream",
    labels={"x": "Layer", "y": "Logit Diff"},
    xaxis_tickvals=labels,
    width=800,
)

<details>
<summary>질문 - 이 그래프의 해석은 무엇입니까? 모델이 이 태스크를 어떻게 해결하는지에 대해 무엇을 알려줍니까?</summary>

흥미롭게도, 모델은 layer 7까지는 이 태스크를 전혀 수행하지 못하며, 거의 모든 성능이 attention layer 9에서 나오고, 그 이후부터는 오히려 성능이 *감소*하는 것을 볼 수 있습니다.

이는 IOI 태스크를 해결하기 위해 residual stream에 올바른 방식으로 기록하는 무언가가 (주로 layer 7, 8, 9에서) 일어나고 있음을 알려줍니다. 이를 통해 우리는 분석 범위를 좁힐 수 있으며, 해당 layer에서 어떤 종류의 계산이 일어나고 있는지(예: attention layer와 MLP의 기여도, 그리고 어떤 attention head가 가장 중요한지)에 대한 질문을 시작할 수 있습니다.
</details>

## Layer Attribution

위의 분석을 각 layer에 대해 반복할 수 있습니다 (이는 인접한 residual stream 사이의 차이와 동일합니다).

참고: 용어가 다소 혼란스러울 수 있습니다. transformer의 layer k는 k번째 **transformer block**을 의미하지만, 각 block은 (정보를 이동시키기 위한) **attention layer**와 (정보를 처리하기 위한) **MLP layer**로 구성됩니다.

In [ ]:
per_layer_residual, labels = cache.decompose_resid(layer=-1, pos_slice=-1, return_labels=True)
per_layer_logit_diffs = residual_stack_to_logit_diff(per_layer_residual, cache)

line(
    per_layer_logit_diffs,
    hovermode="x unified",
    title="Logit Difference From Each Layer",
    labels={"x": "Layer", "y": "Logit Diff"},
    xaxis_tickvals=labels,
    width=800,
)

<details>
<summary>질문 - 이 그래프의 해석은 무엇입니까? 모델이 이 태스크를 어떻게 해결하는지에 대해 무엇을 알려줍니까?</summary>

attention 레이어만 중요하다는 것을 알 수 있으며, 이는 타당합니다! IOI 태스크는 정보를 처리하는 것보다 정보를 이동시키는 것(즉, 틀린 이름이 아닌 올바른 이름을 이동시키는 것)에 관한 것이기 때문입니다. 그리고 다시 한번 attention 레이어 9가 성능을 크게 향상시키는 반면, attention 10과 attention 11은 성능을 *저하시킨다*는 점에 주목합니다.
</details>

## Head Attribution

각 attention layer의 출력을 각 attention head 출력의 합으로 더 세분화할 수 있습니다. 각 attention layer는 12개의 head로 구성되며, 각 head는 독립적이고 가산적으로 작동합니다.

<details> <summary>attention 출력을 head들의 합으로 분해하기</summary>

attention layer의 출력을 계산하는 표준적인 방법은 각 head의 mixed value들을 연결(concatenate)한 뒤, 커다란 output weight matrix를 곱하는 것입니다. 하지만 [A Mathematical Framework](https://transformer-circuits.pub/2021/framework/index.html)에서 설명한 것처럼, 이는 output weight matrix를 head별 출력(여기서는 `model.blocks[k].attn.W_O`)으로 나누고 이를 모두 더하는 것(전체 layer에 대한 전체 bias 항 포함)과 동일합니다.
</details>

In [ ]:
per_head_residual, labels = cache.stack_head_results(layer=-1, pos_slice=-1, return_labels=True)
per_head_residual = einops.rearrange(per_head_residual, "(layer head) ... -> layer head ...", layer=model.cfg.n_layers)
per_head_logit_diffs = residual_stack_to_logit_diff(per_head_residual, cache)

imshow(
    per_head_logit_diffs,
    labels={"x": "Head", "y": "Layer"},
    title="Logit Difference From Each Head",
    width=600,
)

단 몇 개의 head만이 실제로 중요하다는 것을 알 수 있습니다. head 9.6과 9.9는 긍정적으로 크게 기여하며(이는 attention layer 9가 왜 그렇게 중요한지를 설명합니다), 반면 head 10.7과 11.10은 부정적으로 크게 기여합니다(이는 attention layer 10과 layer 11이 왜 능동적으로 해로운지를 설명합니다). 이들은 논문에서 논의된 name mover 및 negative name mover의 일부에 해당합니다. 또한 긍정적 또는 부정적으로 기여하지만 그 영향력이 더 적은 여러 head들도 존재합니다(다른 name mover 및 backup name mover들입니다).

여기서 몇 가지 메타 관찰 사항을 짚어볼 가치가 있습니다. 우리 모델은 144개의 head를 가지고 있지만, 단순하고 일반적인 기법들을 사용하여 이러한 동작을 소수의 특정 head로 국한시킬 수 있었습니다. 이는 attention head가 attention을 이해하기 위한 적절한 추상화 수준이라는 [A Mathematical Framework](https://transformer-circuits.pub/2021/framework/index.html) 의 주장을 뒷받침합니다. 또한 *부정적인* head가 존재한다는 점이 매우 놀랍습니다. 예를 들어 10.7은 잘못된 logit이 나올 확률을 7배 *더* 높입니다. 논문에서 몇 가지 가능성을 논의하고는 있지만, 정확히 어떤 일이 일어나고 있는지는 확실하지 않습니다.

## 이 섹션의 유용한 함수들 요약

여기서는 이전에 보지 못했을 수도 있는 transformerlens의 모든 함수들을 정리합니다.

* `cache.apply_ln_to_stack`
    * residual stream 값들의 스택에 layernorm scaling을 적용합니다.
    * 코드가 너무 복잡해지지 않게 하면서, "residual stream의 최종 값"에서 "logit difference 방향으로의 logit 투영"으로 변환하는 데 이 함수를 사용했습니다!
* `cache.accumulated_resid(layer=None)`
    * 레이어 `layer`까지(또는 layer가 None인 경우 residual stream의 최종 값까지) 누적된 residual stream을 반환합니다. 즉, 해당 레이어의 입력까지의 이전 residual stream들의 스택입니다.
    * **logit lens**를 연구할 때 유용합니다.
    * 출력의 첫 번째 차원은 `(0_pre, 0_mid, 1_pre, 1_mid, ..., final_post)` 입니다.
* `cache.decompose_resid(layer)`
    * 레이어 `layer`로 들어오는 residual stream 입력을 이전 레이어들의 출력 스택으로 분해합니다. 이들의 합이 레이어 `layer`의 입력이 됩니다.
    * 출력의 첫 번째 차원은 `(embed, pos_embed, 0_attn_out, 0_mlp_out, ...)` 입니다.
* `cache.stack_head_results(layer)`
    * 레이어 `layer`까지의 모든 head 결과(즉, residual stream 기여도)의 스택을 반환합니다.
    * (즉, `decompose_resid`과 비슷하지만, 각 레이어를 attention/MLP로 나누는 대신 각 attention 레이어를 head별로 나눕니다)
    * 출력의 첫 번째 차원은 `layer * head` 입니다 (이를 시각화하기 위해 `(layer, head)`로 rearrange 해야 했습니다).

## Attention 분석

Attention head는 attention pattern을 직접 확인하고, 정보가 어떤 위치에서 어디로 이동하는지 연구할 수 있기 때문에 특히 연구 가치가 높습니다. 여기서는 logit에 미치는 직접적인 영향을 살펴보고 있으므로, 마지막 token의 attention pattern만 확인하면 되기 때문에 이 방법이 특히 유용합니다.

우리는 attention pattern을 시각화하기 위해 (Anthropic의 PySvelte 라이브러리를 기반으로 개발된) `circuitsvis` 라이브러리를 사용합니다! direct logit attribution 기준 상위 3개의 양수 및 음수 head를 시각화하며, 이를 첫 번째 prompt에 대해 보여줍니다 (예시로서).

<details> <summary>Attention Pattern 해석하기</summary>

Attention pattern을 볼 때 흔히 하는 실수는, 그것이 반드시 바라본 *token*에 대한 정보(아마도 해당 token의 문맥을 고려한 정보)를 전달해야 한다고 생각하는 것입니다. 하지만 실제로 우리가 확신 있게 말할 수 있는 것은, 그것이 해당 입력 token에 대응하는 *residual stream position*으로부터 정보를 이동시킨다는 점뿐입니다. 특히 모델의 후반부로 갈수록, residual stream에는 입력 token과 전혀 상관없는 컴포넌트들이 있을 수 있습니다! 예를 들어, 문장 끝의 마침표는 해당 문장의 요약 정보를 포함하고 있을 수 있으며, head는 그것이 ".", "!" 또는 "?"로 끝나는지에 상관없이 오직 그 요약 정보만을 이동시킬 수 있습니다.
</details>

In [ ]:
def topk_of_Nd_tensor(tensor: Float[Tensor, "rows cols"], k: int):
    """
    Helper function: does same as tensor.topk(k).indices, but works over 2D tensors.
    Returns a list of indices, i.e. shape [k, tensor.ndim].

    Example: if tensor is 2D array of values for each head in each layer, this will
    return a list of heads.
    """
    i = t.topk(tensor.flatten(), k).indices
    return np.array(np.unravel_index(utils.to_numpy(i), tensor.shape)).T.tolist()


k = 3

for head_type in ["Positive", "Negative"]:
    # Get the heads with largest (or smallest) contribution to the logit difference
    top_heads = topk_of_Nd_tensor(per_head_logit_diffs * (1 if head_type == "Positive" else -1), k)

    # Get all their attention patterns
    attn_patterns_for_important_heads: Float[Tensor, "head q k"] = t.stack(
        [cache["pattern", layer][:, head][0] for layer, head in top_heads]
    )

    # Display results
    display(HTML(f"<h2>Top {k} {head_type} Logit Attribution Heads</h2>"))
    display(
        cv.attention.attention_patterns(
            attention=attn_patterns_for_important_heads,
            tokens=model.to_str_tokens(tokens[0]),
            attention_head_names=[f"{layer}.{head}" for layer, head in top_heads],
        )
    )

알림 - 이러한 시각화에는 `attention_patterns` 또는 `attention_heads`을 사용할 수 있습니다. 전자는 실제 값을 볼 수 있게 해주며, 후자는 출력된 문장의 token 위에 마우스를 올릴 수 있게 해줍니다 (또한 token 고정이나 디스플레이의 모든 head 중첩과 같은 다른 유용한 기능들을 제공합니다). 두 방식 모두 서로 다른 상황에서 유용할 수 있습니다 (하지만 보통은 `attention_patterns`를 사용하는 것을 추천하며, 대부분의 경우 attention pattern을 빠르게 파악하는 데 더 유용합니다).

위의 `attention_patterns`을 `attention_heads`로 교체해 보고, 출력 결과를 비교해 보십시오.

<details>
<summary>도움 요청 - 제 <code>attention_heads</code> plot들이 이상하게 작동합니다.</summary>

이는 `circuitsvis`의 버그로 보입니다 - VSCode에서 attention head plot들의 크기가 계속해서 줄어듭니다.

이 문제가 해결될 때까지, 이를 우회하는 한 가지 방법은 브라우저에서 plot을 여는 것입니다. `webbrowser` library를 사용하여 인라인으로 수행할 수 있습니다:

```python
attn_heads = cv.attention.attention_heads(
    attention = attn_patterns_for_important_heads,
    tokens = model.to_str_tokens(tokens[0]),
    attention_head_names = [f"{layer}.{head}" for layer, head in top_heads],
)

path = "attn_heads.html"

with open(path, "w") as f:
    f.write(str(attn_heads))

webbrowser.open(path)
```

정확히 어디에 저장되고 있는지 확인하려면, `os.getcwd()`으로 현재 작업 디렉토리를 출력하면 됩니다.
</details>

이 plot들을 통해, 구현되고 있는 알고리즘에 대해 생각하기 시작할 수 있습니다. 특히, 양의 attribution score가 높은 attention head들의 경우, `" to"`가 어디를 attention 하고 있습니까? 이 head가 logit diff score에 어떤 영향을 미치고 있을까요?

모델이 어떻게 작동하는지에 대한 전체 가설은 다음 섹션의 끝부분까지 아껴두겠습니다.

# 3️⃣ Activation Patching

> ##### 학습 목표
>
> * activation patching의 개념과 활용 방법을 이해합니다.
>     * transformerlens의 activation patching 헬퍼 함수 중 일부를 처음부터 직접 구현합니다 (즉, hook을 사용합니다).
> * activation patching을 사용하여 residual stream 내에서 중요한 정보가 저장되고 처리되는 layer와 sequence position을 추적합니다.
> * 이 섹션을 마칠 때쯤에는 IOI circuit의 대략적인 스케치를 그릴 수 있어야 합니다.

## 소개

위에서 사용한 기법들의 명백한 한계는 circuit의 맨 마지막 부분, 즉 logit에 직접적으로 영향을 주는 부분만 살펴본다는 점입니다. 분명히 이것만으로는 circuit을 이해하기에 충분하지 않습니다! 우리는 요소들이 어떻게 결합하여 이 최종 출력을 만들어내는지 이해하고 싶으며, 이상적으로는 이 동작을 완전히 설명하는 end-to-end circuit을 찾아내고자 합니다.

이를 조사하기 위해 우리가 사용할 기법은 **activation patching**입니다. 이 방법은 [David Bau and Kevin Meng's excellent ROME paper](https://rome.baulab.info/)에서 처음 소개되었으며, 그곳에서는 causal tracing이라고 불립니다.

activation patching의 설정은 두 가지 서로 다른 입력에 대해 모델을 두 번 실행하는 것입니다. 하나는 clean run이고 다른 하나는 corrupted run입니다. clean run은 정답을 출력하고, corrupted run은 그렇지 않습니다. 핵심 아이디어는 모델에 corrupted 입력을 주되, 특정 activation에 **개입(intervene)**하여 clean run의 해당 activation을 **패치(patch)**하고(즉, corrupted activation을 clean activation으로 교체), 실행을 계속하는 것입니다. 그런 다음 출력이 정답 방향으로 얼마나 업데이트되었는지 측정합니다.

이렇게 하여 가능한 많은 activation에 대해 반복하며, 그것들이 corrupted run에 얼마나 영향을 미치는지 살펴볼 수 있습니다. 특정 activation을 패치했을 때 정답의 확률이 유의미하게 증가한다면, 이를 통해 어떤 activation이 중요한지 *국소화(localise)*할 수 있습니다.

다시 말해, 이것은 (주로 **denoising**이었던 지난 섹션과 달리) **noising** 알고리즘입니다.

국소화 능력은 mechanistic interpretability의 핵심적인 단계입니다. 만약 계산이 모델 전체에적으로 퍼져 있다면, 무슨 일이 일어나고 있는지에 대해 깔끔한 mechanistic 이야기를 구성하기가 훨씬 더 어려울 가능성이 큽니다. 하지만 모델의 어느 부분이 중요한지 정확하게 식별할 수 있다면, 그 부분을 확대하여 그것들이 무엇을 나타내고 서로 어떻게 연결되는지 결정할 수 있으며, 궁극적으로 그것들이 나타내는 기저의 circuit을 역공학(reverse engineer)할 수 있습니다.

아래 다이어그램들은 추상적인 신경망에서의 activation patching을 보여줍니다 (노드는 activation을 나타내며, 노드 사이의 화살표는 weight 연결을 나타냅니다).

clean input에 대한 일반적인 forward pass는 다음과 같습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-1c.png" width="300">

그리고 corrupted input(초록색)에서 clean input(검은색)의 forward pass로 activation patching을 수행하는 모습은 다음과 같습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-2c.png" width="440">

여기서 점선은 값의 patching을 나타냅니다 (즉, clean input에 대한 forward pass 도중, 노드 $D$를 corrupted input에서 갖는 값으로 대체합니다). 노드 $H$, $G$ 및 $F$는 주황색으로 표시되어 있으며, 이는 이들이 이제 clean 또는 corrupted와는 다른 분포를 따른다는 것을 나타냅니다.

우리는 다양한 방식으로 transformer에 패칭(patching)을 할 수 있습니다 (예: residual stream의 값, MLP, 또는 attention head의 출력 - 아래 내용을 참조하십시오). 또한 특정 sequence 위치에서 패칭을 함으로써 더욱 세밀하게 분석할 수도 있습니다 (다이어그램에는 표시되지 않음).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-examples.png" width="840">

### Noising vs denoising

우리는 깨끗한 입력(clean input)으로 모델을 실행하고 오염된 입력(corrupted input)에서 값을 패칭(patching)하여 노이즈를 추가하므로, 이 알고리즘을 일종의 **noising**이라고 부를 수 있습니다. 반대로, 오염된 입력으로 모델을 실행하고 깨끗한 입력에서 값을 패칭하여 노이즈를 제거하는 **denoising** 알고리즘도 고려할 수 있습니다.

noising과 denoising 중 언제 무엇을 사용해야 할까요? 이는 목표에 따라 다릅니다. 특정 컴포넌트나 컴포넌트 집합이 태스크를 수행하기에 충분하다는 것을 보여주는 것은 매우 중요한 의미를 가지므로, denoising의 결과가 훨씬 더 강력합니다. 반면, transformer의 복잡성과 컴포넌트 간의 상호 의존성 때문에 모델에 noising을 적용하면 예측 불가능한 결과가 나타날 수 있습니다. 특정 컴포넌트를 ablation 했을 때 loss가 증가한다고 해서, 반드시 그 컴포넌트가 해당 태스크에 필요했다는 것을 의미하지는 않습니다. 예를 들어, gpt2-small에서 MLP0를 ablation 하면 거의 모든 태스크에서 성능이 훨씬 더 나빠지는 것으로 보이지만(이는 MLP0가 일종의 확장된 embedding 역할을 하기 때문이며, 이에 대해서는 이후 연습 문제에서 더 자세히 다룹니다), IOI 태스크에 *특화된* 중요한 역할을 수행하고 있는 것은 아닙니다.

### 예시: residual stream의 denoising

위의 내용은 모두 상당히 추상적이었으므로, Indirect Object Identification을 이해하기 위해 구체적인 예시를 자세히 살펴보겠습니다. 먼저 denoising에 관한 실습으로 시작하겠지만, 이 섹션의 후반부(그리고 path patching에 관한 다음 섹션)에서는 noising으로 넘어가겠습니다.

여기서 clean input은 원래 문장(예: <code>"When Mary and John went to the store, John gave a drink to"</code>)이 되며, corrupted input은 subject token이 바뀐 문장(예: <code>"When Mary and John went to the store, <u>Mary</u> gave a drink to"</code>)이 됩니다. corrupted residual stream 값을 clean 값으로 교체하는 patching은 인과적 개입(causal intervention)이며, 이를 통해 네트워크의 어느 부분이 indirect object를 식별하고 있는지 정확하게 이해할 수 있습니다. 만약 어떤 컴포넌트가 중요하다면, 해당 컴포넌트를 patching(해당 컴포넌트의 corrupted output을 clean output으로 교체)하는 것이 이 컴포넌트가 생성하는 신호를 되돌려 성능을 훨씬 더 좋게 만들 것입니다.

참고 - "noised dataset"이 신호를 지우는 것이 아니라 실제로 **반전**시키기 때문에, noising과 denoising이라는 용어가 여기에는 정확히 맞지 않습니다. 이를 denoising이라고 설명하는 이유는 프레이밍의 문제입니다. 우리는 어떤 컴포넌트/activation이 **필요한지(necessary)** 보다는, 성능을 회복하는 데 **충분한지(sufficient)**를 알아내려고 노력하고 있습니다. 혼란스러울 때 다음과 같은 프레이밍을 기억하면 유용합니다. **noising은 무엇이 필요한지를 알려주고, denoising은 무엇이 충분한지를 알려줍니다.**

질문 - 대신에 corrupted sentence를 <code>"When <u>John</u> and <u>Mary</u> went to the store, <u>Mary</u> gave a drink to"</code> (즉, 문장 내의 이름 3곳을 모두 바꿈)로 설정할 수도 있습니다. 왜 이렇게 하지 않는다고 생각하십니까?

<details>
<summary>힌트</summary>

만약 모델이 prompt `"When Mary and John went to the store, John gave a drink to"`에 대해 forward pass를 수행하는 도중 어느 시점에 **"간접 목적어는 이 시퀀스의 네 번째 token입니다"**라는 정보의 표현을 가지고 있다면 어떻게 될까요?
</details>

<details>
<summary>답변</summary>

모델은 두 가지 서로 다른 방식으로 간접 목적어 `' Mary'`를 가리킬 수 있습니다:

* **token 정보**를 통해, 즉 **"간접 목적어는 token `' Mary'`입니다"**.
* **positional 정보**를 통해, 즉 **"간접 목적어는 이 시퀀스의 네 번째 token입니다"**.

우리는 corrupted dataset이 clean dataset으로 patch될 때 이 두 가지 신호를 모두 반전시키기를 원합니다. 하지만 만약 세 개의 이름을 모두 바꿔서 dataset을 corrupt했다면 다음과 같은 상황이 발생합니다:

* token 정보는 반전됩니다. 왜냐하면 corrupted prompt에 대한 모델 내의 해당 정보는 **"간접 목적어는 token `' Mary'`입니다"**가 될 것이기 때문입니다.
* positional 정보는 반전되지 ***않습니다***. 왜냐하면 해당 정보는 여전히 **"간접 목적어는 이 시퀀스의 네 번째 token입니다"**일 것이기 때문입니다.

실제로, 보너스 섹션에서는 이 사실을 이용하여 모델이 token 정보와 positional 정보 중 무엇을 사용하는지 분리하여 분석해 볼 것입니다 (즉, token 정보만 반전시키고 positional 정보는 유지하거나, 그 반대로 수행함으로써 분석합니다). 스포일러를 드리자면, 모델은 두 가지를 모두 조금씩 사용하고 있는 것으로 나타납니다!
</details>

패치(patch)하기에 자연스러운 대상 중 하나는 특정 레이어와 특정 위치의 residual stream입니다. 예를 들어, 모델은 처음에 `S2` token이 중복되었다는 것을 인식하기 위해 일부 처리를 수행하고, 그 후 attention을 사용하여 해당 정보를 `end` token으로 이동시킬 가능성이 높습니다. 따라서 `end` token의 residual stream을 패치하는 것은 이후 레이어에서는 매우 중요하겠지만, 초기 레이어에서는 전혀 중요하지 않을 것입니다.

우리는 더 세밀하게 들어가서 특정 레이어의 특정 activation을 패치할 수 있습니다. 예를 들어, 마지막 token에 대한 head 9.9의 출력이 logit으로 직접 연결되는 데 중요하다고 생각한다면, 이 head의 출력만 패치해도 성능에 상당한 영향을 미칠 것이라고 예측할 수 있습니다.

이 기법은 circuit의 구성 요소들이 어떻게 연결되어 있는지를 알려주는 것이 아니라, 단지 어떤 구성 요소들이 관여하는지를 알려준다는 점에 유의하시기 바랍니다.

TransformerLens에는 activation patching을 수행하기 위한 유용한 내장 함수들이 있지만, 이 과정을 더 잘 이해하기 위해 이제 일부 함수들을 기본 원리부터(즉, hook만을 사용하여) 직접 구현해 보겠습니다. 구현한 함수들의 출력을 내장 함수와 비교함으로써 제대로 작동하는지 테스트할 수 있습니다.

hook에 대한 복습이 필요하시다면, hook 사용법과 activation 캐싱 방법을 다루는 induction heads 연습 문제로 돌아가 확인하시기 바랍니다.

In [ ]:
from transformer_lens import patching

## 메트릭 생성하기

패칭을 하기 전에, logit 세트를 평가하기 위한 메트릭을 만들어야 합니다. 우리는 **corrupted prompts** (`S2`이 잘못된 이름으로 대체된 프롬프트)를 실행하고 여기에 **clean prompts**를 패칭할 것이므로, 다음과 같은 메트릭을 선택하는 것이 합리적입니다:

* 0이라는 값은 (corrupted prompt에서의 성능과 비교하여) 변화가 없음을 의미합니다.
* 1이라는 값은 clean performance가 완전히 회복되었음을 의미합니다.

예를 들어, clean prompt 전체를 패칭했다면 1이라는 값을 얻게 됩니다. 만약 패칭을 통해 모델이 clean prompt에서의 일반적인 동작보다 태스크를 더 잘 해결하게 된다면 1보다 큰 값을 얻을 수도 있지만, 일반적으로는 0과 1 사이의 값을 기대합니다.

또한 메트릭이 logit 차이의 선형 함수가 되도록 하는 것이 합리적입니다. 이것만으로도 메트릭을 고유하게 지정하기에 충분합니다.

In [ ]:
clean_tokens = tokens
# Swap each adjacent pair to get corrupted tokens
indices = [i + 1 if i % 2 == 0 else i - 1 for i in range(len(tokens))]
corrupted_tokens = clean_tokens[indices]

print(
    "Clean string 0:    ",
    model.to_string(clean_tokens[0]),
    "\nCorrupted string 0:",
    model.to_string(corrupted_tokens[0]),
)

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

clean_logit_diff = logits_to_ave_logit_diff(clean_logits, answer_tokens)
print(f"Clean logit diff: {clean_logit_diff:.4f}")

corrupted_logit_diff = logits_to_ave_logit_diff(corrupted_logits, answer_tokens)
print(f"Corrupted logit diff: {corrupted_logit_diff:.4f}")

### 연습 문제 - 메트릭 생성하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

필요한 메트릭을 생성하기 위해 아래의 함수 `ioi_metric` 을 완성하십시오. 이 섹션 전체에서 동일한 데이터셋을 사용할 것이므로, 이 함수에서 기본 인자(default arguments)를 사용해도 무방합니다.

**중요 참고 사항** - 이 함수는 float가 아닌 스칼라 tensor를 반환해야 합니다. 그렇지 않으면 이후의 일부 patching 함수들이 작동하지 않습니다. 이 함수의 타입 시그니처는 `Float[Tensor, ""]` 입니다.

**두 번째 중요 참고 사항** - 성능이 corrupted input일 때와 동일하면 0, clean input일 때와 동일하면 1이 되도록 정의했습니다. 이는 우리가 **denoising 알고리즘**을 수행하고 있기 때문입니다. 즉, 모델의 성능을 회복하는 데 충분한 activation(corrupted input으로부터 정답을 회복하기에 충분한 정보를 가진 activation)을 찾고 있는 것입니다. 우리의 "귀무 가설"은 해당 컴포넌트가 충분하지 않으며, 따라서 corrupted 값을 clean 값으로 교체하여 patching 하더라도 성능이 회복되지 않는다는 것입니다. 이후 섹션에서는 noising을 수행할 예정이며, 이를 위한 새로운 메트릭 함수를 정의할 것입니다.

In [ ]:
def ioi_metric(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Int[Tensor, "batch 2"] = answer_tokens,
    corrupted_logit_diff: float = corrupted_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
) -> Float[Tensor, ""]:
    """
    Linear function of logit diff, calibrated so that it equals 0 when performance is same as on
    corrupted input, and 1 when performance is same as on clean input.
    """
    raise NotImplementedError()


t.testing.assert_close(ioi_metric(clean_logits).item(), 1.0)
t.testing.assert_close(ioi_metric(corrupted_logits).item(), 0.0)
t.testing.assert_close(ioi_metric((clean_logits + corrupted_logits) / 2).item(), 0.5)

<details><summary>솔루션</summary>

```python
def ioi_metric(
    logits: Float[Tensor, "batch seq d_vocab"],
    answer_tokens: Int[Tensor, "batch 2"] = answer_tokens,
    corrupted_logit_diff: float = corrupted_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
) -> Float[Tensor, ""]:
    """
    Linear function of logit diff, calibrated so that it equals 0 when performance is same as on
    corrupted input, and 1 when performance is same as on clean input.
    """
    patched_logit_diff = logits_to_ave_logit_diff(logits, answer_tokens)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)


t.testing.assert_close(ioi_metric(clean_logits).item(), 1.0)
t.testing.assert_close(ioi_metric(corrupted_logits).item(), 0.0)
t.testing.assert_close(ioi_metric((clean_logits + corrupted_logits) / 2).item(), 0.5)
```
</details>

## Residual Stream Patching

간단한 예시부터 시작하겠습니다. 각 layer의 시작 부분과 각 token position에서 residual stream에 patch를 수행합니다. 직접 함수를 작성하기 전에, TransformerLens의 `patching` module을 사용하여 이 작업이 어떻게 이루어지는지 살펴보겠습니다. 아래 코드를 실행해 주십시오.

In [ ]:
act_patch_resid_pre = patching.get_act_patch_resid_pre(
    model=model,
    corrupted_tokens=corrupted_tokens,
    clean_cache=clean_cache,
    patching_metric=ioi_metric,
)

labels = [f"{tok} {i}" for i, tok in enumerate(model.to_str_tokens(clean_tokens[0]))]

In [ ]:
imshow(
    act_patch_resid_pre,
    labels={"x": "Position", "y": "Layer"},
    x=labels,
    title="resid_pre Activation Patching",
    width=600
)

질문 - 이 그래프의 해석은 무엇입니까? 모델이 이 태스크를 해결하는 방식의 본질에 대해 어떤 중요한 점을 알려줍니까?

<details>
<summary>힌트</summary>

계산의 지역성(locality of computation)에 대해 생각해 보십시오.
</details>

<details>
<summary>정답</summary>

원래 모든 관련 계산은 `S2`에서 일어나며, layer 7과 8에서 정보가 `END`로 이동합니다. residual stream을 올바른 위치로 이동시키는 것만으로 성능이 *거의 정확하게* 회복됩니다!

분명히 말씀드리면, 이 그래프에서 놀라운 점은 첫 번째 행이 `S2`에서만 1이고 나머지는 모두 0이라는 점이나, 끝부분의 행들이 `END`에서만 1이고 나머지는 0으로 수렴한다는 점이 아닙니다. 이 두 가지는 우리가 정확히 예상했던 결과입니다. 정말 놀라운 점은 다음과 같습니다:

* 계산이 매우 지역화되어 있습니다. `S` 대신 `IO`을 선택하는 데 필요한 관련 정보가 처음에는 `S2` token에 저장되었다가, 다른 곳을 거치지 않고 바로 `END` token으로 이동합니다.
* 모델은 기본적으로 layer 8 이후에 작업을 완료하며, 나머지 layer들은 실제로 이 특정 태스크의 성능을 약간 저하시킵니다.

(참고 - 참조를 위해, 첫 번째 prompt의 token들과 그 인덱스가 x축에 표시되어 있습니다. 표기법상의 편의를 위해, 여기서의 차이는 *모든* 8개 prompt에 대해 평균을 낸 것이지만, 레이블은 *첫 번째* prompt에서만 가져왔다는 점에 유의하십시오.)
</details>

### 연습 문제 - head-to-residual patching 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 20-25 minutes on this exercise.
> 
> It's very important to understand how patching works. Many subsequent exercises will build on this one.
> ```

이제 아래의 `get_act_patch_resid_pre` 함수를 구현해야 하며, 이 함수는 위에서 실행한 코드와 동일한 결과를 제공해야 합니다. 이러한 방식으로 hook을 사용하는 방법에 대해 빠르게 복습해 보겠습니다:

* Hook 함수는 `tensor: torch.Tensor` 및 `hook: HookPoint` 인자를 받습니다. 이보다 더 많은 인자를 받는 hook 함수를 정의한 다음, 실제로 hook을 추가할 때 `functools.partial`를 사용하는 것이 더 쉬운 경우가 많습니다.
* `model.run_with_hooks` 함수는 다음 인자들을 받습니다:
    * 실행할 token들 (첫 번째 인자로)
    * `fwd_hooks` - `(hook_name, hook_fn)` 튜플의 리스트입니다. `utils.get_act_name`를 사용하여 hook 이름을 가져올 수 있음을 기억하십시오.
* 팁 - hook을 추가하고 실행하는 함수의 시작 부분에 `model.reset_hooks()`를 두는 것이 좋은 습관입니다. 이는 때때로 hook이 (실행 중 오류를 일으켜) 제대로 제거되지 않는 경우가 있기 때문입니다. 고장 난 hook을 지우지 못한 채, hook 오류를 수정하고도 동일한 오류 메시지를 받는 것만큼 답답한 일은 없습니다!

In [ ]:
def patch_residual_component(
    corrupted_residual_component: Float[Tensor, "batch pos d_model"],
    hook: HookPoint,
    pos: int,
    clean_cache: ActivationCache,
) -> Float[Tensor, "batch pos d_model"]:
    """
    Patches a given sequence position in the residual stream, using the value
    from the clean cache.
    """
    raise NotImplementedError()


def get_act_patch_resid_pre(
    model: HookedTransformer,
    corrupted_tokens: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable[[Float[Tensor, "batch pos d_vocab"]], float],
) -> Float[Tensor, "3 layer pos"]:
    """
    Returns an array of results of patching each position at each layer in the residual
    stream, using the value from the clean cache.

    The results are calculated using the patching_metric function, which should be
    called on the model's logit output.
    """
    raise NotImplementedError()


act_patch_resid_pre_own = get_act_patch_resid_pre(model, corrupted_tokens, clean_cache, ioi_metric)

t.testing.assert_close(act_patch_resid_pre, act_patch_resid_pre_own)

<details><summary>솔루션</summary>

```python
def patch_residual_component(
    corrupted_residual_component: Float[Tensor, "batch pos d_model"],
    hook: HookPoint,
    pos: int,
    clean_cache: ActivationCache,
) -> Float[Tensor, "batch pos d_model"]:
    """
    Patches a given sequence position in the residual stream, using the value
    from the clean cache.
    """
    corrupted_residual_component[:, pos, :] = clean_cache[hook.name][:, pos, :]
    return corrupted_residual_component


def get_act_patch_resid_pre(
    model: HookedTransformer,
    corrupted_tokens: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable[[Float[Tensor, "batch pos d_vocab"]], float],
) -> Float[Tensor, "3 layer pos"]:
    """
    Returns an array of results of patching each position at each layer in the residual
    stream, using the value from the clean cache.

    The results are calculated using the patching_metric function, which should be
    called on the model's logit output.
    """
    model.reset_hooks()
    seq_len = corrupted_tokens.size(1)
    results = t.zeros(model.cfg.n_layers, seq_len, device=device, dtype=t.float32)

    for layer in tqdm(range(model.cfg.n_layers)):
        for position in range(seq_len):
            hook_fn = partial(patch_residual_component, pos=position, clean_cache=clean_cache)
            patched_logits = model.run_with_hooks(
                corrupted_tokens,
                fwd_hooks=[(utils.get_act_name("resid_pre", layer), hook_fn)],
            )
            results[layer, position] = patching_metric(patched_logits)

    return results
```
</details>

테스트를 통과했다면, 결과를 plot 할 수 있습니다.

In [ ]:
imshow(
    act_patch_resid_pre_own,
    x=labels,
    title="Logit Difference From Patched Residual Stream",
    labels={"x": "Sequence Position", "y": "Layer"},
    width=700,
)

## 블록별 residual stream 패칭(Patching)

각 layer의 residual stream에 단순히 patching하는 대신, attention layer 직후 또는 MLP 직후에만 patching할 수도 있습니다. 이를 통해 어떤 token이 언제 중요한지에 대해 조금 더 세밀한 관점을 얻을 수 있습니다.

함수 `patching.get_act_patch_block_every`은 `get_act_patch_resid_pre`과 동일하게 작동하지만, 단순히 residual stream에 patching하는 것이 아니라 `resid_pre`, `attn_out` 그리고 `mlp_out`에 patching하며, `(3, n_layers, seq_len)` 형태의 tensor를 반환합니다.

한 가지 주의할 점은, `resid_pre`, `attn_out` 그리고 `mlp_out`을 순회하며 세 가지를 동시에 patching하는 것이 아니라 한 번에 하나씩만 patching한다는 점입니다.

In [ ]:
act_patch_block_every = patching.get_act_patch_block_every(model, corrupted_tokens, clean_cache, ioi_metric)

In [ ]:
imshow(
    act_patch_block_every,
    x=labels,
    facet_col=0, # This argument tells plotly which dimension to split into separate plots
    facet_labels=["Residual Stream", "Attn Output", "MLP Output"], # Subtitles of separate plots
    title="Logit Difference From Patched Attn Head Output",
    labels={"x": "Sequence Position", "y": "Layer"},
    width=1200,
)

<details>
<summary>질문 - 뒤의 두 그래프는 어떻게 해석해야 합니까?</summary>

여러 attention layer가 유의미하다는 것을 알 수 있지만, residual stream 결과와 마찬가지로 초기 layer는 `S2`에서 중요하고, 후기 layer는 `END`에서 중요하며, 그 외의 다른 token에서는 layer가 기본적으로 중요하지 않습니다. 매우 국소적입니다!

direct logit attribution과 마찬가지로 layer 9는 양수이고 layer 10과 11은 그렇지 않습니다. 이는 후기 layer들이 direct logit 효과에만 영향을 미친다는 것을 시사하지만, layer 7과 8 또한 상당히 중요하다는 것을 알 수 있습니다. 아마도 이들은 어떤 이름이 중복되었는지에 대한 정보를 `S2`에서 `END`으로 이동시키는 head들일 것입니다.

반면, MLP layer들은 별로 중요하지 않습니다. 이는 이 작업이 정보를 처리하는 것보다 이동시키는 것에 더 가깝고, MLP layer는 정보 처리에 특화되어 있기 때문에 타당합니다. 유일한 예외는 매우 중요한 MLP0인데, 이는 이 작업의 circuit에 관한 것이라기보다 MLP0에 대한 일반적인 특성으로 인해 오해를 불러일으키는 결과라고 생각합니다. MLP0에 관한 흥미로운 여담을 위해 계속 읽어주십시오!

</details>


<details>
<summary>MLP의 지식 저장에 관한 여담</summary>

우리는 어느 시점에 "사실"이나 "지식"이 MLP layer에 저장된다고 언급했을 수도 있습니다.
이 주장을 조사하기 위해 이전 함수를 사용한 예시를 들어보겠습니다:
프롬프트 `The White House is where the`이 주어졌을 때, gpt2가 (완성 문구 `The White House is where the president lives.`의 일부로서) 정답을 ` president`라고 추측할 것으로 예상합니다.
그리고 프롬프트 `The Haunted House is where the`가 주어졌을 때, gpt2가 (완성 문구 `The Haunted House is where the ghosts live.`의 일부로서) 정답을 `ghosts`라고 추측할 것으로 예상합니다.

실제로 그렇습니다 (대부분 제가 단일 token 단어를 교체하여 서로 다른 단일 token 정답을 얻을 수 있는 깔끔한 예시를 만들기 위해 이 프롬프트들을 선별했기 때문입니다).
모델은 이를 어떻게 수행할까요? 어딘가에 White House/President 그리고 Haunted House/Ghosts 사이의 연관성이 저장되어 있어야 합니다.

두 프롬프트를 입력하고, token ` president`과 ` ghosts` 사이의 logit 차이를 지표로 사용하여 이를 확인할 수 있습니다.

```python
clean_prompt, clean_answer = "The White House is where the", " president" #Note the space in the answer!
corrupted_prompt, corrupted_answer = "The Haunted House is where the", " ghosts"

clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

assert clean_tokens.shape == corrupted_tokens.shape, "clean and corrupted tokens must have same shape"

clean_token = model.to_single_token(clean_answer)
corrupted_token = model.to_single_token(corrupted_answer)

utils.test_prompt(clean_prompt, clean_answer, model)
utils.test_prompt(corrupted_prompt, corrupted_answer, model)

clean_logits, clean_cache = model.run_with_cache(clean_tokens)

def answer_metric(
    logits: Float[Tensor, "batch seq d_vocab"],
    clean_token: Int = clean_token,
    corrupted_token: Int = corrupted_token,
) -> Float[Tensor, "batch"]:
    return logits[:, -1, clean_token] - logits[:, -1, corrupted_token]

act_patch_block_every = patching.get_act_patch_block_every(model, corrupted_tokens, clean_cache, answer_metric)

imshow(
    act_patch_block_every,
    x=["<endoftext>","The", "White/Haunted", "House", "is", "where", "the"],
    facet_col=0,  # This argument tells plotly which dimension to split into separate plots
    facet_labels=["Residual Stream", "Attn Output", "MLP Output"],  # Subtitles of separate plots
    title="Logit Difference (president - ghosts)",
    labels={"x": "Sequence Position", "y": "Layer"},
    width=1200,
)
```

</details>

### Tied embeddings (MLP0가 수행하는 역할)

GPT-2 Small에서는 MLP0가 매우 중요하며, 이를 제거(ablating)하면 성능이 완전히 파괴된다는 점이 자주 관찰됩니다. 현재 받아들여지는 가설은 첫 번째 MLP layer가 본질적으로 **embedding의 확장** 역할을 하며, 이후의 layer들이 입력 token에 접근하고자 할 때 token embedding보다는 주로 첫 번째 MLP layer의 출력을 읽는다는 것입니다. 이러한 관점에서 보면, 첫 번째 attention layer는 많은 일을 하지 않습니다.

이러한 프레임워크에서는 `S2`에서 MLP0가 중요한 것이 타당합니다. 왜냐하면 그 위치가 입력 token이 다른 유일한 위치이기 때문입니다 (즉, 두 프롬프트 버전 사이에서 해당 위치만 서로 다른 확장된 embedding을 가지며, 다른 모든 token들은 기본적으로 동일한 확장된 embedding을 갖게 됩니다).

왜 이런 일이 발생할까요? 대부분의 효과는 **GPT2-Small의 embedding과 unembedding matrix가 tied 되어 있다**는 사실, 즉 하나가 다른 하나의 transpose와 같다는 점에서 기인하는 것으로 보입니다. 한편으로는 이것이 원칙적으로 보일 수 있습니다. 만약 두 단어가 비슷한 의미를 가진다면 (예: "big"과 "large"), 이들은 서로 대체 가능해야 하며, 즉 비슷한 embedding과 unembedding을 가져야 합니다. 이는 embedding 공간과 unembedding 공간의 기하학적 구조가 서로 연관되어야 함을 시사하는 것처럼 보입니다. 하지만 다른 한편으로는, 이것이 생각만큼 원칙적이지 않은 한 가지 주요 이유가 있습니다. embedding과 unembedding이 함께 direct path를 형성하기 때문입니다 (만약 다른 컴포넌트가 없다면 transformer는 단순히 linear map $x \to x^T W_E W_U$이 될 것입니다). 그리고 우리는 이것이 대칭적이기를 원하지 않는데, 왜냐하면 bigram 예측은 대칭적이지 않기 때문입니다! 예를 들어, $W_E = W_U^T$라면 "Barack Obama"를 확률 높은 bigram으로 예측하기 위해 "Obama Barack" 또한 동일하게 높은 확률로 예측해야 하며, 이는 분명히 일어나서는 안 되는 일입니다. 따라서 첫 번째 MLP layer가 이러한 비대칭성을 극복하기 위해 부분적으로 사용된다는 점이 타당합니다. 이제 우리는 $\operatorname{MLP}_0(x^T W_E) W_U$을 direct path로 생각하며, 이는 $W_E$와 $W_U$가 tied 되어 있더라도 더 이상 대칭적이지 않습니다.

### 연습 문제 (선택 사항) - head-to-block patching 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> 
> Most code can be copied from the last exercise.
> ```

원하신다면 재미로 `get_act_patch_block_every` 함수를 구현해 보셔도 좋습니다. 다만, 이전 연습 문제와 충분히 유사하므로 이를 반드시 수행하실 필요는 없습니다.

In [ ]:
def get_act_patch_block_every(
    model: HookedTransformer,
    corrupted_tokens: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable[[Float[Tensor, "batch pos d_vocab"]], float],
) -> Float[Tensor, "3 layer pos"]:
    """
    Returns an array of results of patching each position at each layer in the residual stream,
    using the value from the clean cache.

    The results are calculated using the patching_metric function, which should be called on the
    model's logit output.
    """
    raise NotImplementedError()


act_patch_block_every_own = get_act_patch_block_every(model, corrupted_tokens, clean_cache, ioi_metric)

t.testing.assert_close(act_patch_block_every, act_patch_block_every_own)

imshow(
    act_patch_block_every_own,
    x=labels,
    facet_col=0,
    facet_labels=["Residual Stream", "Attn Output", "MLP Output"],
    title="Logit Difference From Patched Attn Head Output",
    labels={"x": "Sequence Position", "y": "Layer"},
    width=1200,
)

In [ ]:
imshow(
    act_patch_block_every_own,
    x=labels,
    facet_col=0,
    facet_labels=["Residual Stream", "Attn Output", "MLP Output"],
    title="Logit Difference From Patched Attn Head Output",
    labels={"x": "Sequence Position", "y": "Layer"},
    width=1200
)

<details><summary>솔루션</summary>

```python
def get_act_patch_block_every(
    model: HookedTransformer,
    corrupted_tokens: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable[[Float[Tensor, "batch pos d_vocab"]], float],
) -> Float[Tensor, "3 layer pos"]:
    """
    Returns an array of results of patching each position at each layer in the residual stream,
    using the value from the clean cache.

    The results are calculated using the patching_metric function, which should be called on the
    model's logit output.
    """
    model.reset_hooks()
    results = t.zeros(3, model.cfg.n_layers, tokens.size(1), device=device, dtype=t.float32)

    for component_idx, component in enumerate(["resid_pre", "attn_out", "mlp_out"]):
        for layer in tqdm(range(model.cfg.n_layers)):
            for position in range(corrupted_tokens.shape[1]):
                hook_fn = partial(patch_residual_component, pos=position, clean_cache=clean_cache)
                patched_logits = model.run_with_hooks(
                    corrupted_tokens,
                    fwd_hooks=[(utils.get_act_name(component, layer), hook_fn)],
                )
                results[component_idx, layer, position] = patching_metric(patched_logits)

    return results
```
</details>

## Head Patching

개별 head를 patching함으로써 위의 분석을 더 정교하게 다듬을 수 있습니다! 이제 세 개의 차원이 존재하기 때문에 `(head_index, position and layer)` 이 작업은 다소 더 번거롭습니다.

아래 코드는 모든 sequence position에 대해 특정 head의 output을 patching하고, 그 결과(모델의 각 head에 대해)를 반환합니다.

In [ ]:
act_patch_attn_head_out_all_pos = patching.get_act_patch_attn_head_out_all_pos(
    model, corrupted_tokens, clean_cache, ioi_metric
)

In [ ]:
imshow(
    act_patch_attn_head_out_all_pos,
    labels={"y": "Layer", "x": "Head"},
    title="attn_head_out Activation Patching (All Pos)",
    width=600
)

<details>
<summary>질문 - 이 그래프의 해석은 무엇입니까? 어떤 head가 중요하다고 생각하십니까?</summary>

지난 섹션 마지막의 attention plot에서 관찰했던 일부 head들을 볼 수 있습니다 (예: `9.9`은 큰 양수 점수를 가지고, `10.7`는 큰 음수 점수를 가집니다). 하지만 다른 중요한 head들도 볼 수 있으며, 예를 들어 다음과 같습니다:

* layer 7-8에는 여러 개의 중요한 head가 있습니다. 우리는 이것들이 `S2`에서 `end`으로 정보를 이동시키는 역할을 한다고 추론할 수 있습니다.
* 더 앞쪽 layer들에는 몇 가지 더 중요한 head들이 있습니다 (예: `3.0` 및 `5.5`). 우리는 이것들이 어떤 원시적인 로직을 수행하고 있다고 추측할 수 있으며, 예를 들어 두 번째 `" John"` token이 이전의 자기 자신 인스턴스들에 attend하게 만드는 것일 수 있습니다.

</details>

### 연습 문제 - head-to-head patching 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> 
> Again, it should be similar to the first patching exercise (you can copy code).
> ```

아래에 이 patching 함수의 자신만의 버전을 구현해야 합니다.

새로운 hook 함수를 정의해야 하지만, 이전 연습 문제의 코드 대부분은 재사용 가능할 것입니다.

<details>
<summary>도움말 - patching에 어떤 hook 이름을 사용해야 할지 모르겠습니다.</summary>

다음 위치에서 patch를 수행해야 합니다:

```python
utils.get_act_name("z", layer)
```

이는 value 벡터들의 선형 결합입니다. 즉, residual stream에 다시 더하기 전에 $W_O$와 곱하는 대상입니다. $W_O$ 곱셈 이후에 patch를 하는 것은 효과는 동일하지만 메모리를 더 많이 사용하기 때문에(`d_model`이 `d_head`보다 크기 때문입니다) 의미가 없습니다.
</details>

In [ ]:
def patch_head_vector(
    corrupted_head_vector: Float[Tensor, "batch pos head_index d_head"],
    hook: HookPoint,
    head_index: int,
    clean_cache: ActivationCache,
) -> Float[Tensor, "batch pos head_index d_head"]:
    """
    Patches the output of a given head (before it's added to the residual stream) at every sequence
    position, using the value from the clean cache.
    """
    raise NotImplementedError()


def get_act_patch_attn_head_out_all_pos(
    model: HookedTransformer,
    corrupted_tokens: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable,
) -> Float[Tensor, "layer head"]:
    """
    Returns an array of results of patching at all positions for each head in each layer, using the
    value from the clean cache. The results are calculated using the patching_metric function, which
    should be called on the model's logit output.
    """
    raise NotImplementedError()

In [ ]:
act_patch_attn_head_out_all_pos_own = get_act_patch_attn_head_out_all_pos(
    model, corrupted_tokens, clean_cache, ioi_metric
)

t.testing.assert_close(act_patch_attn_head_out_all_pos, act_patch_attn_head_out_all_pos_own)

In [ ]:
imshow(
    act_patch_attn_head_out_all_pos_own,
    title="Logit Difference From Patched Attn Head Output",
    labels={"x":"Head", "y":"Layer"},
    width=600
)

<details><summary>솔루션</summary>

```python
def patch_head_vector(
    corrupted_head_vector: Float[Tensor, "batch pos head_index d_head"],
    hook: HookPoint,
    head_index: int,
    clean_cache: ActivationCache,
) -> Float[Tensor, "batch pos head_index d_head"]:
    """
    Patches the output of a given head (before it's added to the residual stream) at every sequence
    position, using the value from the clean cache.
    """
    corrupted_head_vector[:, :, head_index] = clean_cache[hook.name][:, :, head_index]
    return corrupted_head_vector


def get_act_patch_attn_head_out_all_pos(
    model: HookedTransformer,
    corrupted_tokens: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable,
) -> Float[Tensor, "layer head"]:
    """
    Returns an array of results of patching at all positions for each head in each layer, using the
    value from the clean cache. The results are calculated using the patching_metric function, which
    should be called on the model's logit output.
    """
    model.reset_hooks()
    results = t.zeros(model.cfg.n_layers, model.cfg.n_heads, device=device, dtype=t.float32)

    for layer in tqdm(range(model.cfg.n_layers)):
        for head in range(model.cfg.n_heads):
            hook_fn = partial(patch_head_vector, head_index=head, clean_cache=clean_cache)
            patched_logits = model.run_with_hooks(
                corrupted_tokens,
                fwd_hooks=[(utils.get_act_name("z", layer), hook_fn)],
                return_type="logits",
            )
            results[layer, head] = patching_metric(patched_logits)

    return results


act_patch_attn_head_out_all_pos_own = get_act_patch_attn_head_out_all_pos(
    model, corrupted_tokens, clean_cache, ioi_metric
)

t.testing.assert_close(act_patch_attn_head_out_all_pos, act_patch_attn_head_out_all_pos_own)

imshow(
    act_patch_attn_head_out_all_pos_own,
    title="Logit Difference From Patched Attn Head Output",
    labels={"x": "Head", "y": "Layer"},
    width=600,
)
```
</details>

## Head 분해하기

마지막으로, activation patching의 또 다른 예시를 살펴보겠습니다.

attention layer를 개별 head 단위의 patching으로 분해한 것만으로도 동작을 국소화하는 데 큰 도움이 되었습니다. 하지만 head를 더 세부적으로 분해함으로써 이를 더욱 깊이 이해할 수 있습니다. 하나의 attention head는 두 개의 반독립적인 연산으로 구성됩니다. 하나는 정보를 *어디서* 어디로 이동시킬지 계산하는 것이고(attention pattern으로 표현되며 QK-circuit을 통해 구현됩니다), 다른 하나는 *어떤* 정보를 이동시킬지 계산하는 것입니다(value 벡터로 표현되며 OV circuit에 의해 구현됩니다). attention pattern*만* 또는 value 벡터*만* patching 함으로써 이 중 어느 것이 중요한지 분리해낼 수 있습니다. 이 분해에 대한 더 자세한 내용은 [A Mathematical Framework](https://transformer-circuits.pub/2021/framework/index.html) 또는 [Neel's walkthrough video](https://www.youtube.com/watch?v=KV5gbOmHbjU) 을 참조하시기 바랍니다.

이를 수행하는 데 유용한 함수는 `get_act_patch_attn_head_all_pos_every` 입니다. 단순히 head output에 패칭하는 것(이전 방식과 같이)이 아니라, 다음 항목들에 패칭합니다:
* Output (이는 head가 residual stream에 쓰는 값을 패칭하는 것과 동일합니다)
* Queries (즉, key나 value 벡터는 변경하지 않고 query 벡터만 패칭합니다)
* Keys
* Values
* Patterns (즉, attention patterns를 패칭합니다).

다시 한번 말씀드리지만, 이 함수는 여러 항목을 동시에 패칭하는 것이 아닙니다. 이 다섯 가지 항목을 각각 루프하며 하나씩 패칭한 결과를 얻는 것입니다.

In [ ]:
act_patch_attn_head_all_pos_every = patching.get_act_patch_attn_head_all_pos_every(
    model, corrupted_tokens, clean_cache, ioi_metric
)

In [ ]:
imshow(
    act_patch_attn_head_all_pos_every,
    facet_col=0,
    facet_labels=["Output", "Query", "Key", "Value", "Pattern"],
    title="Activation Patching Per Head (All Pos)",
    labels={"x": "Head", "y": "Layer"},
)

### 연습 문제 (선택 사항) - head-to-head-input patching 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> Most code can be copied from the last exercise.
> ```

마찬가지로, 이를 직접 구현하고 싶다면 아래에서 진행하실 수 있습니다. 하지만 이전 연습 문제들과 개념적으로 다르지 않기 때문에 필수 과제는 아닙니다. 직접 구현하지 않더라도, 어떤 일이 일어나고 있는지 확실히 이해하기 위해 솔루션을 확인하시기 바랍니다.

In [ ]:
def patch_attn_patterns(
    corrupted_head_vector: Float[Tensor, "batch head_index pos_q pos_k"],
    hook: HookPoint,
    head_index: int,
    clean_cache: ActivationCache,
) -> Float[Tensor, "batch pos head_index d_head"]:
    """
    Patches the attn patterns of a given head at every sequence position, using the value from the
    clean cache.
    """
    raise NotImplementedError()


def get_act_patch_attn_head_all_pos_every(
    model: HookedTransformer,
    corrupted_tokens: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable,
) -> Float[Tensor, "layer head"]:
    """
    Returns an array of results of patching at all positions for each head in each layer (using the
    value from the clean cache) for output, queries, keys, values and attn pattern in turn.

    The results are calculated using the patching_metric function, which should be called on the
    model's logit output.
    """
    raise NotImplementedError()


act_patch_attn_head_all_pos_every_own = get_act_patch_attn_head_all_pos_every(
    model, corrupted_tokens, clean_cache, ioi_metric
)

t.testing.assert_close(act_patch_attn_head_all_pos_every, act_patch_attn_head_all_pos_every_own)

In [ ]:
imshow(
    act_patch_attn_head_all_pos_every_own,
    facet_col=0,
    facet_labels=["Output", "Query", "Key", "Value", "Pattern"],
    title="Activation Patching Per Head (All Pos)",
    labels={"x": "Head", "y": "Layer"},
    width=1200
)

<details><summary>솔루션</summary>

```python
def patch_attn_patterns(
    corrupted_head_vector: Float[Tensor, "batch head_index pos_q pos_k"],
    hook: HookPoint,
    head_index: int,
    clean_cache: ActivationCache,
) -> Float[Tensor, "batch pos head_index d_head"]:
    """
    Patches the attn patterns of a given head at every sequence position, using the value from the
    clean cache.
    """
    corrupted_head_vector[:, head_index] = clean_cache[hook.name][:, head_index]
    return corrupted_head_vector


def get_act_patch_attn_head_all_pos_every(
    model: HookedTransformer,
    corrupted_tokens: Float[Tensor, "batch pos"],
    clean_cache: ActivationCache,
    patching_metric: Callable,
) -> Float[Tensor, "layer head"]:
    """
    Returns an array of results of patching at all positions for each head in each layer (using the
    value from the clean cache) for output, queries, keys, values and attn pattern in turn.

    The results are calculated using the patching_metric function, which should be called on the
    model's logit output.
    """
    results = t.zeros(5, model.cfg.n_layers, model.cfg.n_heads, device=device, dtype=t.float32)
    # Loop over each component in turn
    for component_idx, component in enumerate(["z", "q", "k", "v", "pattern"]):
        for layer in tqdm(range(model.cfg.n_layers)):
            for head in range(model.cfg.n_heads):
                # Get different hook function if we're doing attention probs
                hook_fn_general = patch_attn_patterns if component == "pattern" else patch_head_vector
                hook_fn = partial(hook_fn_general, head_index=head, clean_cache=clean_cache)
                # Get patched logits
                patched_logits = model.run_with_hooks(
                    corrupted_tokens,
                    fwd_hooks=[(utils.get_act_name(component, layer), hook_fn)],
                    return_type="logits",
                )
                results[component_idx, layer, head] = patching_metric(patched_logits)

    return results


act_patch_attn_head_all_pos_every_own = get_act_patch_attn_head_all_pos_every(
    model, corrupted_tokens, clean_cache, ioi_metric
)

t.testing.assert_close(act_patch_attn_head_all_pos_every, act_patch_attn_head_all_pos_every_own)

imshow(
    act_patch_attn_head_all_pos_every_own,
    facet_col=0,
    facet_labels=["Output", "Query", "Key", "Value", "Pattern"],
    title="Activation Patching Per Head (All Pos)",
    labels={"x": "Head", "y": "Layer"},
    width=1200,
)
```
</details>

참고 - 우리는 이를 훨씬 더 세밀한 방식으로 수행할 수 있습니다. `patching.get_act_patch_attn_head_by_pos_every` 함수(즉, 위와 동일하지만 `all_pos`을 `by_pos`로 대체)를 사용하면 layer, head, component뿐만 아니라 sequence position별로도 동일한 분해 결과를 얻을 수 있습니다. 앞서 언급한 `patching.get_act_patch_attn_head_out_all_pos` 함수에 대해서도 마찬가지입니다(`all_pos`를 `by_pos`로 대체). 다만, 당연하게도 이 함수들은 상당히 느립니다!

이 그래프에는 몇 가지 눈에 띄는 특징이 있습니다. 예를 들어, 최소 세 가지의 서로 다른 head 그룹이 있다는 것을 보여줍니다:

* 초기 head들 (`3.0`, `5.5`, `6.9`)은 attention pattern(구체적으로는 query 벡터) 때문에 중요합니다.
* 7번 및 8번 layer의 중간 head들 (`7.3`, `7.9`, `8.6`, `8.10`)은 value 벡터 때문에 더 중요한 것으로 보입니다.
* logit difference를 개선하는 후기 head들 (`9.9`, `10.0`)은 query 벡터 때문에 중요합니다.

질문 - 중간 head들(즉, 7번 및 8번 layer의 중요한 head들)의 결과가 갖는 의미는 무엇입니까? 특히, value patching이 다른 두 가지 형태의 patching보다 훨씬 더 큰 영향을 미친다는 사실을 어떻게 해석해야 합니까?

*힌트 - 혼란스럽다면 head `7.3`, `7.9`, `8.6`, `8.10`의 attention pattern을 그려보십시오. attention head의 출력을 표시했을 때 사용한 위의 코드를 대부분 재사용할 수 있습니다.*

<details>
<summary>attention head를 그리기 위한 코드</summary>

```python
# Get the heads with largest value patching
# (we know from plot above that these are the 4 heads in layers 7 & 8)
k = 4
top_heads = topk_of_Nd_tensor(act_patch_attn_head_all_pos_every[3], k=k)

# Get all their attention patterns
attn_patterns_for_important_heads: Float[Tensor, "head q k"] = t.stack([
    cache["pattern", layer][:, head].mean(0)
        for layer, head in top_heads
])

# Display results
display(HTML(f"<h2>Top {k} Logit Attribution Heads (from value-patching)</h2>"))
display(cv.attention.attention_patterns(
    attention = attn_patterns_for_important_heads,
    tokens = model.to_str_tokens(tokens[0]),
    attention_head_names = [f"{layer}.{head}" for layer, head in top_heads],
))
```
</details>

<details>
<summary>답변</summary>

attention pattern을 보면 이 head들이 `END`에서 `S2`로 attend한다는 것을 알 수 있으며, 따라서 이들이 정답을 결정하는 데 사용되는 정보를 `S2`에서 `END`로 이동시키는 역할을 한다고 추측할 수 있습니다. 이는 대부분의 정보가 7번 및 8번 layer를 통해 이동한다는 것을 확인했던 이전 결과와 일치합니다.

value patching이 이들에게 가장 중요하다는 사실은, 흥미로운 계산이 **왜 `end`이 `S2`에 attend하는가**보다는 **`S2`에서 `end`으로 어떤 정보를 이동시키는가**에 들어있음을 시사합니다. 왜 이러한 추론을 내릴 수 있는지 혼란스럽다면 아래 다이어그램을 참조하십시오.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/k-vs-v-patching-explained.png" width="900">

</details>

## 이해 다지기

이제 시야를 넓혀 내용을 다시 정리해 보겠습니다. 지금까지 우리가 발견한 가장 중요한 관찰 결과들은 다음과 같습니다:

* Head `9.9`, `9.6`, `10.0`는 residual stream에 직접적으로 쓰는 관점에서 가장 중요한 head들입니다. 이 모든 head에서 `END`은 `IO`에 강하게 attend합니다.
    * 우리는 각 layer의 각 head가 residual stream에 쓰는 값들을 가져와 `residual_stack_to_logit_diff`를 사용하여 logit diff 방향으로 투영함으로써 이를 발견했습니다. 또한 `circuitsvis`을 사용하여 attention pattern을 살펴보았습니다.
    * <span style="color:darkorange">**이는 이 head들이 `IO`을 `end`로 복사하여, 이를 예측된 다음 token으로 사용하고 있음을 시사합니다.**</span>
    * 그렇다면 질문은 *"이 head들이 어떻게 `S`이 아니라 이 token에 attend해야 하는지 어떻게 아는가?"*가 됩니다.

<br>

* 모든 동작은 layer 7까지는 `S2`에서 일어나다가 이후 `END`로 전이됩니다. 그리고 attention layer는 매우 중요하지만, MLP layer는 (확장된 embedding 역할을 하는 MLP0를 제외하고는) 그리 중요하지 않습니다.
    * 우리는 `resid_pre`, `attn_out`, `mlp_out`에 대해 **activation patching**을 수행하여 이를 발견했습니다.
    * <span style="color:darkorange">**이는 layer 7과 8에 `S2`에서 `END`으로 정보를 이동시키는 head 클러스터가 있음을 시사합니다. 우리는 이 정보가 head `9.9`, `9.6`, `10.0`이 `IO`에 attend하도록 만드는 방법이라고 추론합니다.**</span>
    * 그렇다면 질문은 *"이 정보는 무엇이며, 어떻게 `S2` token에 도달하게 되고, `END`은 어떻게 그것에 attend해야 하는지 아는가?"*가 됩니다.

<br>

* layer 7과 8에서 중요한 head들은 `7.3`, `7.9`, `8.6`, `8.10`입니다. 이 head들은 value 벡터에 대해 높은 activation patching 값을 가지며, query와 key에 대해서는 상대적으로 낮습니다.
    * 우리는 이 head들의 value 입력에 대해 **activation patching**을 수행하여 이를 발견했습니다.
    * <span style="color:darkorange">**이는 이전의 관찰 결과를 뒷받침하며, 흥미로운 계산이 `END`이 `S2`에 attend한다는 사실보다는 `S2`에서 `END`로 *무엇이 이동되는가*에 들어있음을 알려줍니다.**</span>.
    * 우리는 여전히 다음을 알지 못합니다: *"이 정보는 무엇이며, 어떻게 `S2` token에 도달하게 되는가?"*

<br>

* 위에서 언급한 2개의 head 클러스터 외에도, 세 번째 중요한 head 클러스터가 있습니다. 바로 query 벡터가 좋은 성능을 내는 데 특히 중요한 초기 head들(예: `3.0`, `5.5`, `6.9`)입니다.
    * 우리는 이 head들의 query 입력에 대해 **activation patching**을 수행하여 이를 발견했습니다.

이 모든 점을 고려했을 때, 이 세 개의 head가 무엇을 하고 있는지에 대한 이론을 세우고 전체 circuit에 대한 간단한 모델을 만들어 보시겠습니까?

*힌트 - 여전히 어려우시다면, head `3.0`의 attention pattern을 그려보십시오. `5.5`과 `6.9`의 패턴은 처음에는 다소 혼란스러울 수 있습니다 (이들은 circuit이 작동하는 "가장 단순한 그림"에 복잡함을 더합니다). 이들이 circuit의 핵심을 이해하는 데 방해가 되지 않도록 나중에 논의하겠습니다.*

<details>
<summary>정답 (및 circuit의 간단한 다이어그램)</summary>

head `3.0`의 attention pattern을 그렸다면, `S2`이 `S1`에 attention을 기울이는 것을 확인하셨을 것입니다. 이는 초기 head들이 destination token이 중복되었는지를 감지하고 있음을 시사합니다. 따라서 subject가 중복되었다는 정보가 `S2`에 저장됩니다.

subject token이 중복되었다는 정보가 어떻게 `end` 이후의 token을 예측하는 데 도움이 될까요? 정답(`IO` token)은 중복되지 않은 token입니다. 따라서 subject token이 중복되었다는 정보가 후기 head들이 중복된 token에 기울이는 attention을 *억제(inhibit)*하는 데 사용되며, 대신 중복되지 않은 token에 attention을 기울이게 된다고 추론할 수 있습니다.

circuit의 후반부를 요약하자면, 이 중복된 token에 대한 정보는 중간 head 클러스터인 `7.3`, `7.9`, `8.6` 및 `8.10`에 의해 `S2`에서 `end`으로 이동하며, 이 정보는 후기 head인 `9.9`, `9.6` 및 `10.0`의 query로 들어가 이들이 중복된 token에 대한 attention을 *억제*하게 만듭니다. 대신, 이들은 `IO`에 attention을 기울입니다 (이 token을 logit으로 직접 복사합니다).

이 circuit의 그림은 대체로 정확한 것으로 밝혀졌습니다. 곧 논의할 몇 가지 세부 사항이 빠져 있긴 하지만, 머릿속에 그려두기에 좋은 대략적인 그림입니다. 이를 다음과 같이 도식화할 수 있습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ioi-main-simple.png" width="1000">

설명:

* 초기 head들을 **DTH** (duplicate token heads)라고 부르며, 이들의 역할은 `S2`이 중복되었음을 감지하는 것입니다.
* 두 번째 head 그룹은 **SIH** (S-inhibition heads)라고 하며, 이들의 역할은 중복된 token 정보를 `S2`에서 `END`로 이동시키는 것입니다. 여기서는 positional 정보를 이동시키는 것으로 묘사했지만, 원칙적으로 이는 token embedding 정보일 수도 있습니다 (마지막 섹션에서 더 자세히 다룹니다).
* 마지막 head 그룹은 **NMH** (name mover heads)라고 하며, 이들의 역할은 `IO` token을 `END` token으로 복사하여 이를 예측된 다음 token으로 사용하는 것입니다 (S-inhibition heads 덕분에, 이 head들은 `S` token에 attention을 기울이지 않습니다).

참고 - 이 다이어그램을 어떻게 해석해야 할지 여전히 혼란스럽지만 induction circuit과 그 작동 방식을 이해하고 계신다면, 제가 [induction circuits](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ih-simple.png)를 위해 동일한 스타일로 작성한 다이어그램과 비교해 보는 것이 도움이 될 수 있습니다. 또한, 저의 induction heads [LessWrong post](https://www.lesswrong.com/posts/TvrfY4c9eaGLeyDkE/induction-heads-illustrated)를 읽으셨고 이 스타일의 다이어그램이 그것과 어떻게 다른지 혼란스러우시다면, [here](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ih-compared.png)은 두 다이어그램(induction heads용)을 비교하고 차이점을 설명하는 이미지입니다.

</details>

이제 우리의 결과와 논문의 결과를 비교하여 이 그림을 조금 더 구체화해 보겠습니다. 아래는 위 드롭다운에 있던 다이어그램의 더 복잡한 버전이며, 중요한 head들도 표시되어 있습니다. 이 다이어그램은 논문의 [original diagram](https://res.cloudinary.com/lesswrong-2-0/image/upload/v1672942728/mirroredImages/3ecs6duLmTfyra3Gp/h5icqzpyuhu4mqvfjhvw.png)를 기반으로 합니다. 이 다이어그램의 모든 내용을 이해하지 못하더라도 걱정하지 마십시오. circuit의 경계는 모호하며, 이 circuit 내의 모든 head의 "역할"은 leaky abstraction입니다. 그보다는, 이 다이어그램은 여러분이 이 circuit을 더 잘 이해할 수 있도록 직관을 올바른 방향으로 안내하기 위한 것입니다.

<details>
<summary>대규모 circuit 다이어그램</summary>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ioi-main-full-c.png" width="1250">
</details>

위의 방식과 다른 주요 차이점은 다음과 같습니다:

#### Induction heads

첫 번째 head 클러스터에는 단순히 duplicate token head만 있는 것이 아니라, previous token head와 induction head라는 두 가지 다른 유형의 head가 더 있습니다. induction head는 induction 메커니즘을 통해 duplicate token head와 동일한 역할을 수행합니다. 이들은 token `S2`이 `S1+1`에 attention 하도록 만들며(previous token head에 의해 매개됨), 그 출력은 `S1`에 대한 pointer이자 `S1`이 복제되었다는 신호로 모두 사용됩니다 (이 두 가지의 차이점에 대해서는 아래 "Position vs token information being moved" 단락에서 더 자세히 다룹니다).

*(참고 - 원본 논문의 다이어그램은 induction head와 duplicate token head가 서로 결합되는 것처럼 묘사하고 있습니다. 이는 오해의 소지가 있으며, 실제로는 그렇지 않습니다.)*

왜 이 circuit에서 induction head가 사용될까요? 보너스 섹션에서 더 자세히 살펴보겠지만, 한 가지 가능성은 induction head가 학습 초기에 기본적으로 형성되는 특성이라서, 모델이 이미 존재하는 이 메커니즘을 해당 작업에 맞게 재활용하는 것이 효율적이기 때문일 수 있습니다. induction head에 대한 더 자세한 내용과 그것들이 어떻게, 왜 형성되는지에 대해서는 [this paper](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html)를 참조하십시오.

#### Negative & Backup name mover heads

앞서 우리는 후반부 레이어의 일부 head들이 실제로 성능을 저하시키고 있다는 것을 확인했습니다. 이 head들은 name mover head와 매우 유사한 동작을 하지만, 반대 방향으로 작동(즉, 정답을 억제)하는 것으로 나타났습니다. 모델이 왜 이렇게 동작하는지는 명확하지 않으나, 논문에서는 이러한 head들이 모델이 실수를 할 때 높은 cross-entropy loss를 피하기 위해 "위험을 분산(hedge)"하는 데 도움을 줄 수 있다고 추측합니다.

Backup name mover head들은 아마 더 기이할 것입니다. name mover head들을 **ablate** 했을 때, 이 head들이 그 빈자리를 채워 어떻게든 작업을 수행한다는 것이 밝혀졌습니다 (NMH가 ablate 되지 않았을 때는 이 작업을 수행하지 않는 것처럼 보임에도 불구하고 말입니다). 이는 모델 내의 **내장된 중복성(built-in redundancy)**의 한 예시입니다. 한 가지 가능한 설명은 모델이 dropout과 함께 학습되었기 때문이라는 것이지만, 이 설명이 완전히 만족스럽지는 않습니다 (dropout 없이 학습된 모델들도 여전히 BNMH를 가지고 있는 것으로 보이며, 다만 이 모델만큼 강력하지는 않습니다). induction head와 마찬가지로, 마지막 섹션에서 이에 대해 더 자세히 살펴보겠습니다.

#### Positional 정보 vs token 정보

다이어그램에는 두 가지 종류의 S-inhibition head가 표시되어 있습니다. positional 정보에 기반하여 억제하는 head(분홍색)와 token 정보에 기반하여 억제하는 head(보라색)입니다. 어떤 head가 어떤 역할을 수행하는지는 명확하지 않으며, 실제로 일부 head는 두 가지 역할을 모두 수행할 수도 있습니다.

해당 논문은 어떤 S-inhibition head가 어떤 유형의 정보를 사용하는지 구분해내는 독창적인 방법을 제시하며, 이에 대해서는 마지막 섹션에서 논의하겠습니다.

#### S-inhibition head에서의 K-composition

S-inhibition head의 key와 value에 대해 activation patching을 수행했을 때, value는 중요했지만 key는 그렇지 않다는 것을 발견했습니다. 우리는 이 head들에서 K-composition이 실제로 일어나지 않으며, `END`이 `S2`에 attention을 기울이는 이유는 중복 token 정보 외의 다른 이유(예: 단순히 가장 가까운 이름에 attention을 기울이거나, 쉼표로 구분되지 않은 모든 이름에 attention을 기울이는 것일 수 있습니다) 때문이라고 결론지었습니다. 이것이 대체로 사실이긴 하지만, 이 head들에서도 약간의 K-composition이 일어나고 있음이 밝혀졌습니다. 우리는 이를 duplicate token head가 residual stream에 "중복됨" 플래그를 쓰고(이 token의 정체나 위치에 대한 정보는 포함하지 않고), 이 플래그를 S-inhibition head의 key가 사용하여 `END`가 `S2`에 attention을 기울이게 만드는 것으로 생각할 수 있습니다. 다이어그램에서 이는 (단순화된 버전의 연한 회색 박스뿐만 아니라) 진한 회색 박스로 표시됩니다. 아직 이것이 일어난다는 증거를 보지는 못했지만, 다음 섹션(path patching을 살펴볼 때)에서 확인하게 될 것입니다.

참고 - 초기 head들이 residual stream에 위치 정보나 "중복 플래그" 정보를 쓰는지 여부는 해당 head가 induction head인지 또는 duplicate token head인지와 반드시 연관되는 것은 아닙니다. 원칙적으로 두 유형의 head 모두 두 가지 유형의 정보를 모두 쓸 수 있습니다.

# 4️⃣ Path Patching

> ##### 학습 목표
>
> * path patching의 개념과 activation patching과의 차이점을 이해합니다.
> * hook을 사용하여 path patching을 처음부터 구현합니다.
> * [IOI paper](https://arxiv.org/abs/2211.00593)의 여러 결과들을 재현합니다.

이 섹션은 이전 두 섹션보다 개념적이고 탐색적인 성격이 훨씬 적으며, 훨씬 더 기술적이고 엄격하게 진행됩니다. path patching이 무엇인지, 그리고 어떻게 작동하는지 배우게 되며, 이를 사용하여 논문의 많은 결과(뿐만 아니라 path patching과 관련 없는 다른 논문의 결과들)를 재현하게 됩니다.

## 설정

여기서는 앞선 몇 가지 섹션에서 사용했던 간이적인 탐색 방식보다는, 논문 저자들이 사용한 설정을 더 밀접하게 따를 것입니다. 분명히 말씀드리자면, 모델의 circuit을 막 조사하기 시작한 단계라면 여기서 사용하는 엄격한 설정의 상당 부분은 필수적이지 않습니다. 이러한 엄격함은 논문을 발표할 때는 필요하지만, 많은 시간과 노력이 소요될 수 있습니다!

In [ ]:
from part41_indirect_object_identification.ioi_dataset import NAMES, IOIDataset

우리가 사용할 데이터셋은 `IOIDataset`의 인스턴스이며, 이는 `NAMES` 리스트에서 이름을 무작위로 선택하여 생성됩니다 (문장 템플릿과 객체 또한 서로 다른 리스트에서 선택됩니다). 이것이 어떻게 수행되는지에 대한 자세한 내용은 `ioi_dataset.py` 파일을 통해 확인하실 수 있습니다.

(참고 - 이 코드를 실행하는 동안 메모리 오류가 발생한다면 `N`을 줄이셔도 됩니다. 만약 `N = 10`에서도 여전히 메모리 오류가 발생한다면, Colab으로 전환하거나 Lambda Labs와 같은 가상 머신을 사용하시는 것을 권장합니다.)

In [ ]:
N = 25
ioi_dataset = IOIDataset(
    prompt_type="mixed",
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
    seed=1,
    device=str(device),
)

이 데이터셋에는 몇 가지 유용한 속성과 메서드가 있습니다. 이번 실습을 위해 숙지해야 할 주요 항목들은 다음과 같습니다:

* `toks`은 token ID를 포함하는 `(batch_size, max_seq_len)` 모양의 tensor입니다 (즉, 모델에 입력으로 전달하는 값입니다).
* `s_tokenIDs`와 `io_tokenIDs`은 각각 subject와 object의 token ID를 포함하는 리스트입니다.
* `sentences`는 문장들(문자열 형태)을 포함하는 리스트입니다.
* `word_idx`는 단어 유형(예: `"S1"`, `"S2"`, `"IO"` 또는 `"end"`)을 데이터셋의 각 시퀀스 내 해당 단어들의 위치를 포함하는 tensor로 매핑하는 dictionary입니다.
    * 이전 섹션들과 달리 subject, indirect object, 그리고 end token의 위치가 모든 문장에서 동일하지 않기 때문에, 이는 인덱싱을 할 때 특히 유용합니다.

먼저, patching을 위해 어떤 데이터셋을 사용해야 할까요? 이전 섹션에서는 단순히 subject와 indirect object token의 위치를 바꿨으며, 이는 신호의 방향이 뒤집혔음을 의미했습니다. 하지만 여기서 우리가 할 작업은 조금 더 원칙적입니다. IOI 신호를 뒤집는 대신, 이를 지워버릴 것입니다. 우리는 `ioi_dataset`에서 모든 이름을 서로 다른 무작위 이름으로 대체하여 새로운 데이터셋을 구축함으로써 이를 수행합니다. 이렇게 하면 문장 구조는 그대로 유지되지만, 실제 indirect object identification 작업과 관련된 모든 정보(즉, 반복되는 이름의 정체와 위치)가 삭제됩니다.

예를 들어, 문장 `"When John and Mary went to the shops, John gave the bag to Mary"`이 주어졌을 때, ABC 데이터셋의 대응하는 문장은 `"When Edward and Laura went to the shops, Adam gave the bag to Mary"`가 될 수 있습니다. 우리는 후자의 prompt에 대한 residual stream이 IOI 작업을 해결하는 데 도움이 될 만한 token이나 positional 정보(즉, `John`보다 `Mary`을 선호하거나, 4번째 token보다 2번째 token을 선호하는 정보)를 전혀 포함하지 않을 것이라고 기대합니다.

이 데이터셋을 아래에 정의합니다. `gen_flipped_prompts` 메서드의 구문에 주목하십시오. 알파벳은 시퀀스 내의 이름을 어떻게 대체할지를 알려줍니다. 예를 들어, `ABB->XYZ`은 `"When Mary and John went to the store, John gave a drink to Mary"` 형태의 문장에서 `"When [X] and [Y] went to the store, [Z] gave a drink to Mary"`을 사용하여 독립적으로 무작위 선택된 세 이름 `[X]`, `[Y]`, `[Z]`로 대체하라는 의미입니다. 우리는 이 함수를 보너스 섹션에서 positional 신호와 token 신호를 분리하려고 할 때 더 많이 사용할 것입니다 (예를 들어 `ABB->BAB`와 같이 처음 두 이름을 바꾸는 재미있는 작업도 가능하기 때문입니다).

In [ ]:
abc_dataset = ioi_dataset.gen_flipped_prompts("ABB->XYZ, BAB->XYZ")

이 데이터셋을 살펴보겠습니다. 행이 아닌 열을 입력받아 테이블을 출력하는 헬퍼 함수 `make_table`를 정의하겠습니다 (구문에 대해서는 걱정하지 마십시오, 중요하지 않습니다).

In [ ]:
def format_prompt(sentence: str) -> str:
    """Format a prompt by underlining names (for rich print)"""
    return re.sub("(" + "|".join(NAMES) + ")", lambda x: f"[u bold dark_orange]{x.group(0)}[/]", sentence) + "\n"


def make_table(cols, colnames, title="", n_rows=5, decimals=4):
    """Makes and displays a table, from cols rather than rows (using rich print)"""
    table = Table(*colnames, title=title)
    rows = list(zip(*cols))
    f = lambda x: x if isinstance(x, str) else f"{x:.{decimals}f}"
    for row in rows[:n_rows]:
        table.add_row(*list(map(f, row)))
    rprint(table)


make_table(
    colnames=["IOI prompt", "IOI subj", "IOI indirect obj", "ABC prompt"],
    cols=[
        map(format_prompt, ioi_dataset.sentences),
        model.to_string(ioi_dataset.s_tokenIDs).split(),
        model.to_string(ioi_dataset.io_tokenIDs).split(),
        map(format_prompt, abc_dataset.sentences),
    ],
    title="Sentences from IOI vs ABC distribution",
)

다음으로, 이전 섹션에서 다루었던 것과 유사한 함수들을 정의하겠습니다. 여러분이 직접 작성하는 과정을 반복하게 하는 대신 함수들을 제공해 드립니다 (하지만 이 함수들을 이전에 작성하신 것과 비교해 보고, 어떻게 작동하는지 반드시 이해하시기 바랍니다).

namespace가 오염되지 않도록, 이 함수들의 이름은 약간 다르게 지정하겠습니다.

In [ ]:
def logits_to_ave_logit_diff_2(
    logits: Float[Tensor, "batch seq d_vocab"],
    ioi_dataset: IOIDataset = ioi_dataset,
    per_prompt=False,
) -> Float[Tensor, "*batch"]:
    """
    Returns logit difference between the correct and incorrect answer.

    If per_prompt=True, return the array of differences rather than the average.
    """
    # Only the final logits are relevant for the answer
    # Get the logits corresponding to the indirect object / subject tokens respectively
    io_logits: Float[Tensor, "batch"] = logits[
        range(logits.size(0)), ioi_dataset.word_idx["end"], ioi_dataset.io_tokenIDs
    ]
    s_logits: Float[Tensor, "batch"] = logits[
        range(logits.size(0)), ioi_dataset.word_idx["end"], ioi_dataset.s_tokenIDs
    ]
    # Find logit difference
    answer_logit_diff = io_logits - s_logits
    return answer_logit_diff if per_prompt else answer_logit_diff.mean()


model.reset_hooks(including_permanent=True)

ioi_logits_original, ioi_cache = model.run_with_cache(ioi_dataset.toks)
abc_logits_original, abc_cache = model.run_with_cache(abc_dataset.toks)

ioi_per_prompt_diff = logits_to_ave_logit_diff_2(ioi_logits_original, per_prompt=True)
abc_per_prompt_diff = logits_to_ave_logit_diff_2(abc_logits_original, per_prompt=True)

ioi_average_logit_diff = logits_to_ave_logit_diff_2(ioi_logits_original).item()
abc_average_logit_diff = logits_to_ave_logit_diff_2(abc_logits_original).item()

print(f"Average logit diff (IOI dataset): {ioi_average_logit_diff:.4f}")
print(f"Average logit diff (ABC dataset): {abc_average_logit_diff:.4f}")

make_table(
    colnames=["IOI prompt", "IOI logit diff", "ABC prompt", "ABC logit diff"],
    cols=[
        map(format_prompt, ioi_dataset.sentences),
        ioi_per_prompt_diff,
        map(format_prompt, abc_dataset.sentences),
        abc_per_prompt_diff,
    ],
    title="Sentences from IOI vs ABC distribution",
)

우리는 항상 ***ABC 데이터셋이 아니라 IOI 데이터셋의 정답을 기준으로*** 성능을 측정하고 있다는 점에 유의하십시오. 이는 ABC 데이터셋이 IOI 태스크에 도움이 되는 어떠한 정보도 포함하지 않기를 원하기 때문입니다 (따라서 이를 patching 하면 정답과 전혀 상관관계가 없는 신호를 얻게 됩니다). 예를 들어, 모델은 당연히 `"When Max and Victoria got a snack at the store, Clark decided to give it to"`와 같은 문장을 `"Tyler"`라는 이름으로 완성하지 않을 것입니다.

마지막으로, 새로운 데이터에 적용 가능한 새로운 `ioi_metric` 함수를 정의해 보겠습니다.

논문의 결과와 일치시키기 위해, 여기서는 다른 관례를 사용하겠습니다. 0은 성능이 IOI 데이터셋과 동일함(즉, 어떤 방식으로도 손상되지 않음)을 의미하며, -1은 성능이 ABC 데이터셋과 동일함(즉, 모델이 주어와 간접 목적어를 구분하는 능력을 완전히 상실함)을 의미합니다.

마찬가지로, 이 함수에는 약간 다른 이름을 붙이겠습니다.

In [ ]:
def ioi_metric_2(
    logits: Float[Tensor, "batch seq d_vocab"],
    clean_logit_diff: float = ioi_average_logit_diff,
    corrupted_logit_diff: float = abc_average_logit_diff,
    ioi_dataset: IOIDataset = ioi_dataset,
) -> float:
    """
    We calibrate this so that the value is 0 when performance isn't harmed (i.e. same as IOI
    dataset), and -1 when performance has been destroyed (i.e. is same as ABC dataset).
    """
    patched_logit_diff = logits_to_ave_logit_diff_2(logits, ioi_dataset)
    return (patched_logit_diff - clean_logit_diff) / (clean_logit_diff - corrupted_logit_diff)


print(f"IOI metric (IOI dataset): {ioi_metric_2(ioi_logits_original):.4f}")
print(f"IOI metric (ABC dataset): {ioi_metric_2(abc_logits_original):.4f}")

## path patching이란 무엇입니까?

이전 섹션에서는 activation patching에 대해 살펴보았습니다. 이는 *다른 모든 것을 동일하게 유지하면서, 특정 attention head가 residual stream에 쓰는 값을 다른 분포 하에서 썼을 값으로 교체한다면 어떤 일이 일어날까?* 와 같은 질문에 답하는 방법입니다. 이는 attention head와 같은 개별 컴포넌트의 역할을 조사하는 좋은 방법임이 증명되었으며, key / query / value를 차례로 patching 하여 어떤 것이 어떤 head에 더 중요한지 파악하는 것과 같은 더 세밀한 분석을 가능하게 했습니다.

하지만 circuit을 연구할 때, 단순히 attention head 전체를 교체하는 대신 다음과 같이 더 미묘한 질문을 던지고 싶을 수 있습니다. *다른 모든 것을 동일하게 유지하면서, attention head $A$에서 head $B$로($B$가 $A$ 다음에 오는 경우) 전달되는 직접적인 입력값을 다른 분포 하에서의 값으로 교체한다면 어떤 일이 일어날까?*. 이는 attention head가 일반적으로 얼마나 중요한가라는 질문보다는, 이 두 attention head를 연결하여 형성된 circuit이 얼마나 중요한가라는 더 구체적인 질문에 답합니다. Path patching은 바로 이러한 질문들에 답하기 위해 설계되었습니다.

다음 다이어그램들이 transformer에서 activation patching과 path patching의 차이점을 설명하는 데 도움이 될 것입니다. activation patching은 다음과 같았음을 상기해 보십시오:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-2c.png" width="420">

여기서 검은색과 초록색 분포는 각각 clean 데이터셋과 corrupted 데이터셋입니다 (따라서 이는 `ioi_dataset` 및 `abc_dataset`가 됩니다). 이와 대조적으로, path patching은 **node**가 아닌 **edge**를 교체하는 것을 포함합니다. 아래 다이어그램에서, 우리는 edge $D \to G$를 corrupted 분포에서의 값으로 교체하고 있습니다. 따라서 patched run에서 $G$는 clean 분포에서와 동일하게 계산되지만, $D$로부터 오는 **직접적인** 입력이 대신 corrupted 분포에서 온 것처럼 처리됩니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-3c.png" width="560">

안타깝게도 transformer의 경우, 이를 실제로 구현하는 것보다 설명하는 것이 더 쉽습니다. 그 이유는 "노드"가 attention head이고, "엣지"들이 residual stream 안에서 모두 얽혀 있기 때문입니다 (즉, 해당 엣지를 포함하는 모든 경로에 영향을 주지 않고 어떻게 하나의 엣지 값만 변경할 수 있을지가 불분명합니다). 해결책은 아래 다이어그램(오른쪽에서 왼쪽으로 읽습니다)에 표시된 3단계 알고리즘을 사용하는 것입니다.

용어 참고 - head $D$를 **sender node**라고 부르고, head $G$를 **receiver node**라고 부릅니다. 또한, 노드를 "freezing"한다는 것은 "입력과 동일한 값으로 patch하는 것"을 의미합니다. 예를 들어, 아래 2단계에서 head $H$를 freeze하지 않는다면, head $D$의 오염된 값에 영향을 받아 다른 값을 가지게 될 것입니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simpler-patching-4c.png" width="900">

이를 구체화하기 위해, 레이어당 2개의 head를 가진 간단한 3-layer transformer를 예로 들어보겠습니다. head `0.0`에서 `2.0`로 가는 edge에 대해 path patching을 수행해 보겠습니다 (용어 참고: `0.0`가 **sender**이고, `2.0`이 **receiver**입니다). 여기서 "direct paths"는 다른 attention head를 거치지 않는 모든 경로를 의미합니다 (따라서 어떤 조합의 MLP를 거쳐도 무방합니다). 직관적으로, 노드(attention head)만이 모델 내에서 정보를 이동시킬 수 있는 유일한 요소이며, 이것이 우리가 연구하고자 하는 대상입니다. 반면, MLP는 단순히 정보 처리를 수행하므로 이 작업에서는 그리 흥미롭지 않습니다.

우리의 3단계 프로세스는 아래 다이어그램과 같습니다 (초록색은 corrupted, 회색은 clean임을 기억하십시오).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/path-patching-alg-transformers-6.png" width="700">

(참고 - 이 다이어그램에서 색이 칠해지지 않은 노드는 patching을 수행하지 않음을 나타냅니다. 즉, 해당 노드의 downstream에 있는 노드들의 값으로부터 계산되도록 둡니다.)

이것이 왜 작동할까요? 위의 가운데 그림을 충분히 오래 살펴본다면, `0.0` $\to$ `2.0` 로 향하는 모든 비직접 경로(non-direct path)의 기여도는 clean distribution에서와 동일한 반면, 모든 직접 경로(direct path)의 기여도는 corrupted distribution에서와 동일하다는 것을 깨닫게 될 것입니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/path-patching-decomp-four.png" width="850">

### 왜 MLP를 사용합니까?

왜 MLP를 direct path의 일부로 포함하는지 궁금하실 수 있습니다. 짧게 대답하자면, 이는 IOI 논문에서 사용한 방식이며 저희는 이를 재현하려고 하기 때문입니다! 조금 더 자세히 설명하자면, 이 방법과 MLP를 direct path로 간주하지 않는 방법 모두 정당화될 수 있습니다.

한 가지 예를 들어보겠습니다. head `0.0`의 출력이 head `2.0`에 의해 직접 사용되지만, 그 사이에서 하나의 MLP가 매개체 역할을 한다고 가정해 봅시다. 지나치게 단순화해서 생각하면, `0.0`가 벡터 $v$을 residual stream에 쓰고, 어떤 neuron이 $v$를 감지하여 $w$를 residual stream에 쓰며, `2.0`이 $w$을 감지하는 상황을 상상할 수 있습니다. 만약 MLP를 direct path로 계산하지 않는다면, 이러한 인과 관계를 포착할 수 없을 것입니다. 단점은 상황이 조금 더 복잡해진다는 점입니다. 이제 우리는 기본적으로 MLP에 "가짜 입력(fake input)"을 전달하게 되며, 이전에 설명한 것처럼 깨끗한 연산(벡터 $v$, $w$를 사용한 연산)이 이러한 새로운 상황에서도 여전히 일어날 것이라고 가정하는 것은 위험하기 때문입니다.

또한, MLP를 direct path의 일부로 둔다고 해서 MLP가 circuit에서 어떤 역할을 하는지 이해하는 데 도움이 되지는 않으며, 단지 일부 MLP가 중요하다는 것만 알려줄 뿐입니다! 다행히 IOI circuit에서는 (MLP0를 제외하고) MLP가 중요하지 않으므로, 이 두 가지 형태의 path patching을 모두 수행해도 꽤 비슷한 결과가 나옵니다. 선택 사항으로, 이 다른 형태의 path patching을 사용하여 다음 몇 섹션의 결과를 재현해 볼 수 있습니다. 사실 이 방식이 알고리즘적으로 구현하기 더 쉬운데, 두 번의 forward pass 대신 한 번의 forward pass만 필요하기 때문입니다. 그 이유를 알 수 있을까요?

<details>
<summary>정답</summary>

이전 버전의 알고리즘에서는 MLP가 sender와 receiver 사이의 direct path 일부였기 때문에, receiver에 패칭할 값을 찾기 위해 forward pass를 수행해야 했습니다. 하지만 MLP가 direct path의 일부가 아니라면, receiver 노드에 무엇을 패칭할지 직접 계산할 수 있습니다:

```
orig_receiver_input <- orig_receiver_input + (new_sender_output - old_sender_output)
```

MLP를 포함하지 않는 direct path 다이어그램:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/path-patching-decomp-one.png" width="1200">

</details>

## Path Patching: Name Mover Heads

우리는 가장 간단한 형태의 path patching부터 시작하겠습니다. 여기서는 단 하나의 receiver node, 즉 residual stream의 최종 값을 사용합니다. 지금까지는 receiver node가 다른 attention head인 경우만 논의했지만, 동일한 원리가 어떤 receiver node를 선택하더라도 적용됩니다.

<details>
<summary>질문 - attention head에서 residual stream으로의 path patching과, 해당 attention head에 대한 activation patching의 차이점을 설명해 주실 수 있나요?</summary>

Activation patching은 해당 head의 값과, 그 head에 의존하는 모든 후속 layer의 값을 변경합니다.

Path patching은 다음과 같은 질문에 답합니다. "만약 head가 residual stream에 직접 쓴 값은 $x_{new}$ 때와 같지만, 이 head에서 residual stream으로 가는 모든 비직접 경로(즉, 다른 head를 거치는 경로)의 값은 $x_{orig}$ 때와 같다면 어떻게 될까?"
</details>

이 patching은 [the paper](https://arxiv.org/pdf/2211.00593.pdf) (5페이지)의 섹션 3.1 시작 부분에 설명되어 있습니다. 3단계 프로세스는 다음과 같습니다:

1. clean input과 corrupted input으로 모델을 실행합니다. head output들을 cache합니다.
2. clean input으로 모델을 실행하되, sender head는 corrupted input의 값으로 **patched**하고, 다른 모든 head는 clean input일 때의 값으로 **frozen**합니다. residual stream의 최종 값(즉, 최종 layer의 `resid_post`)을 cache합니다.
3. 보통은 clean input으로 모델을 다시 실행하고 최종 residual stream의 cached 값을 patch하겠지만, 이 경우에는 다른 forward pass를 실행할 필요 없이 residual stream의 최종 값을 직접 unembed할 수 있으므로 그럴 필요가 없습니다.

다음은 2-layer transformer에 대한 예시 그림입니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/path-patching-residpost-newest.png" width="680">

### 연습 문제 - 최종 residual stream 값에 대한 path patching 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 30-45 minutes on this exercise.
> Path patching is a very challenging algorithm with many different steps.
> ```

위에서(그리고 논문에서) 설명한 대로, head에서 residual stream으로 이어지는 path patching을 구현해야 합니다.

이 연습 문제는 여러 가지 구성 요소가 포함되어 있어 상당히 어려울 것으로 예상됩니다. 저희는 의도적으로 함수와 docstring만 제공하고 나머지는 비워두어 매우 개방적인 형태로 구성했습니다.

진행을 위한 몇 가지 힌트와 팁은 다음과 같습니다:

* 함수를 3개 부분(위의 각 단계에 하나씩)으로 나누고, 각 섹션을 하나씩 작성하십시오.
* 새로운 hook 함수가 필요합니다: 알고리즘의 2단계에서 freezing / patching을 수행하는 함수입니다.
* activation patching 함수에서 사용한 코드의 상당 부분을 재사용할 수 있습니다.
* `model.run_with_cache`을 호출할 때, 이름에서 boolean으로 매핑되는 함수인 키워드 인자 `names_filter`를 사용할 수 있습니다. 이 인자를 사용하면, 모델은 이 필터를 통과하는 이름을 가진 activation만 캐시합니다 (예를 들어, query 벡터만 캐시하기 위해 `names_filter = lambda name: name.endswith("q")`와 같이 사용할 수 있습니다).

더 많은 힌트와 가이드가 필요하다면 드롭다운 메뉴를 확인하십시오 (예: 함수 docstring부터 시작하고 싶은 경우).

결과를 그래프로 그리고 [the paper](https://arxiv.org/pdf/2211.00593.pdf)의 Figure 3(b) (6페이지 상단)를 재현할 수 있다면 성공한 것입니다.

**참고 - 만약 `model.add_hook` 다음에 `model.run_with_cache`을 사용한다면, `add_hook` 메서드에 `level=1` 인자를 전달해야 할 수도 있습니다. 왜 이 작업을 하지 않으면 함수가 가끔 실패하는지 정확한 이유는 모르겠습니다 (이 버그는 연습 문제가 작성된 이후에 나타나기 시작했습니다). 이를 추적할 시간이 없었지만, 원인을 찾아내시는 분께는 추가 점수를 드리겠습니다 (-:**

<details>
<summary>메인 함수에 대한 docstring을 확인하려면 여기를 클릭하십시오.</summary>

```python
def get_path_patch_head_to_final_resid_post(
    model: HookedTransformer,
    patching_metric: Callable,
    new_dataset: IOIDataset = abc_dataset,
    orig_dataset: IOIDataset = ioi_dataset,
    new_cache: ActivationCache | None = abc_cache,
    orig_cache: ActivationCache | None = ioi_cache,
) -> Float[Tensor, "layer head"]:
    '''
    Performs path patching (see algorithm in appendix B of IOI paper), with:

        sender head = (each head, looped through, one at a time)
        receiver node = final value of residual stream

    Returns:
        tensor of metric values for every possible sender head
    '''
    pass
```
</details>

<details>
<summary>메인 함수에 대한 docstring과 일부 주석 및 함수 구조를 확인하려면 여기를 클릭하십시오.</summary>

```python
def get_path_patch_head_to_final_resid_post(
    model: HookedTransformer,
    patching_metric: Callable,
    new_dataset: IOIDataset = abc_dataset,
    orig_dataset: IOIDataset = ioi_dataset,
    new_cache: ActivationCache | None = abc_cache,
    orig_cache: ActivationCache | None = ioi_cache,
) -> Float[Tensor, "layer head"]:
    '''
    Performs path patching (see algorithm in appendix B of IOI paper), with:

        sender head = (each head, looped through, one at a time)
        receiver node = final value of residual stream

    Returns:
        tensor of metric values for every possible sender head
    '''
    model.reset_hooks()
    results = t.zeros(model.cfg.n_layers, model.cfg.n_heads, device=device, dtype=t.float32)

    # ========== Step 1 ==========
    # Gather activations on x_orig and x_new

    # YOUR CODE HERE


    # Using itertools to loop gives us a smoother progress bar (using nested for loops is also fine)
    for (sender_layer, sender_head) in tqdm_notebook(list(itertools.product(
        range(model.cfg.n_layers),
        range(model.cfg.n_heads)
    ))):
        pass

        # ========== Step 2 ==========
        # Run on x_orig, with sender head patched from x_new, every other head frozen

        # YOUR CODE HERE


        # ========== Step 3 ==========
        # Unembed the final residual stream value, to get our patched logits

        # YOUR CODE HERE


        # Save the results
        results[sender_layer, sender_head] = patching_metric(patched_logits)


    return results
```
</details>

In [ ]:
def patch_or_freeze_head_vectors(
    orig_head_vector: Float[Tensor, "batch pos head_index d_head"],
    hook: HookPoint,
    new_cache: ActivationCache,
    orig_cache: ActivationCache,
    head_to_patch: tuple[int, int],
) -> Float[Tensor, "batch pos head_index d_head"]:
    """
    This helps implement step 2 of path patching. We freeze all head outputs (i.e. set them to their
    values in orig_cache), except for head_to_patch (if it's in this layer) which we patch with the
    value from new_cache.

    head_to_patch: tuple of (layer, head)
    """
    # Setting using ..., otherwise changing orig_head_vector will edit cache value too
    orig_head_vector[...] = orig_cache[hook.name][...]
    if head_to_patch[0] == hook.layer():
        orig_head_vector[:, :, head_to_patch[1]] = new_cache[hook.name][:, :, head_to_patch[1]]
    return orig_head_vector


def get_path_patch_head_to_final_resid_post(
    model: HookedTransformer,
    patching_metric: Callable,
    new_dataset: IOIDataset = abc_dataset,
    orig_dataset: IOIDataset = ioi_dataset,
    new_cache: ActivationCache | None = abc_cache,
    orig_cache: ActivationCache | None = ioi_cache,
) -> Float[Tensor, "layer head"]:
    """
    Performs path patching (see algorithm in appendix B of IOI paper), with:

        sender head = (each head, looped through, one at a time)
        receiver node = final value of residual stream

    Returns:
        tensor of metric values for every possible sender head
    """
    raise NotImplementedError()


path_patch_head_to_final_resid_post = get_path_patch_head_to_final_resid_post(model, ioi_metric_2)

In [ ]:
imshow(
    100 * path_patch_head_to_final_resid_post,
    title="Direct effect on logit difference",
    labels={"x": "Head", "y": "Layer", "color": "Logit diff. variation"},
    coloraxis=dict(colorbar_ticksuffix="%"),
    width=600,
)

<details>
<summary>도움말 - heatmap의 모든 값이 동일하게 나옵니다.</summary>

여기에는 몇 가지 가능한 이유가 있을 수 있습니다. 흔한 원인 중 하나는 텐서의 값만 바꾸는 것이 아니라 실제 텐서 자체를 변경하고 있는 경우입니다. 이는 하나의 텐서가 변경될 때 다른 텐서도 함께 변경됨을 의미합니다. 예를 들어, 다음과 같이 작성한 경우:

```python
x = t.zeros(3)
y = x
x[0] = 1
print(y)
```

그러면 `y` 또한 `[1, 0, 0]`가 됩니다. 이를 방지하려면 "이 텐서의 모든 값을 다른 텐서의 값으로 설정하라"는 의미인 `...` 구문을 사용할 수 있습니다. 예를 들어, 다음과 같이 작성하면:

```python
x = t.zeros(3)
y = t.zeros(3)
x[...] = y
x[0] = 1
print(y)
```

그러면 `y`은 여전히 `[0, 0, 0]`로 유지됩니다.

`x[:] = y`을 사용하는 것도 방법입니다.

---

또 다른 가능한 설명은 알고리즘의 어느 시점에서 잘못된 입력 값이나 cache를 전달했거나, 잘못된 값으로 freeze한 경우입니다. 다이어그램에서 회색은 원래 값(clean)을 나타내고 파란색은 새로운 값(corrupted)을 나타낸다는 점을 기억하십시오. 예를 들어, 단계 2에서는 모델을 `orig_dataset`(= IOI dataset)으로 실행하고, 모든 non-sender head를 `orig_cache`의 값으로 freeze해야 합니다.

---

마지막으로, sender patching을 덮어쓰지 않는 방식으로 head를 freeze하고 있지는 않은지 확인하십시오! 하나의 hook point에 여러 개의 hook 함수가 추가되면, 추가된 순서대로 실행됩니다 (마지막 함수가 이전 함수들을 덮어쓸 수 있습니다).

</details>


<details><summary>해결책</summary>

```python
def patch_or_freeze_head_vectors(
    orig_head_vector: Float[Tensor, "batch pos head_index d_head"],
    hook: HookPoint,
    new_cache: ActivationCache,
    orig_cache: ActivationCache,
    head_to_patch: tuple[int, int],
) -> Float[Tensor, "batch pos head_index d_head"]:
    """
    This helps implement step 2 of path patching. We freeze all head outputs (i.e. set them to their
    values in orig_cache), except for head_to_patch (if it's in this layer) which we patch with the
    value from new_cache.

    head_to_patch: tuple of (layer, head)
    """
    # Setting using ..., otherwise changing orig_head_vector will edit cache value too
    orig_head_vector[...] = orig_cache[hook.name][...]
    if head_to_patch[0] == hook.layer():
        orig_head_vector[:, :, head_to_patch[1]] = new_cache[hook.name][:, :, head_to_patch[1]]
    return orig_head_vector


def get_path_patch_head_to_final_resid_post(
    model: HookedTransformer,
    patching_metric: Callable,
    new_dataset: IOIDataset = abc_dataset,
    orig_dataset: IOIDataset = ioi_dataset,
    new_cache: ActivationCache | None = abc_cache,
    orig_cache: ActivationCache | None = ioi_cache,
) -> Float[Tensor, "layer head"]:
    """
    Performs path patching (see algorithm in appendix B of IOI paper), with:

        sender head = (each head, looped through, one at a time)
        receiver node = final value of residual stream

    Returns:
        tensor of metric values for every possible sender head
    """
    model.reset_hooks()
    results = t.zeros(model.cfg.n_layers, model.cfg.n_heads, device=device, dtype=t.float32)

    resid_post_hook_name = utils.get_act_name("resid_post", model.cfg.n_layers - 1)
    resid_post_name_filter = lambda name: name == resid_post_hook_name

    # ========== Step 1 ==========
    # Gather activations on x_orig and x_new

    # Note the use of names_filter for the run_with_cache function. Using it means we
    # only cache the things we need (in this case, just attn head outputs).
    z_name_filter = lambda name: name.endswith("z")
    if new_cache is None:
        _, new_cache = model.run_with_cache(new_dataset.toks, names_filter=z_name_filter, return_type=None)
    if orig_cache is None:
        _, orig_cache = model.run_with_cache(orig_dataset.toks, names_filter=z_name_filter, return_type=None)

    # Looping over every possible sender head (the receiver is always the final resid_post)
    for sender_layer, sender_head in tqdm(list(product(range(model.cfg.n_layers), range(model.cfg.n_heads)))):
        # ========== Step 2 ==========
        # Run on x_orig, with sender head patched from x_new, every other head frozen

        hook_fn = partial(
            patch_or_freeze_head_vectors,
            new_cache=new_cache,
            orig_cache=orig_cache,
            head_to_patch=(sender_layer, sender_head),
        )
        model.add_hook(z_name_filter, hook_fn)

        _, patched_cache = model.run_with_cache(
            orig_dataset.toks, names_filter=resid_post_name_filter, return_type=None
        )
        assert set(patched_cache.keys()) == {resid_post_hook_name}

        # ========== Step 3 ==========
        # Unembed the final residual stream value, to get our patched logits

        patched_logits = model.unembed(model.ln_final(patched_cache[resid_post_hook_name]))

        # Save the results
        results[sender_layer, sender_head] = patching_metric(patched_logits)

    return results


path_patch_head_to_final_resid_post = get_path_patch_head_to_final_resid_post(model, ioi_metric_2)
```
</details>

이 그래프의 해석은 무엇입니까? activation patching을 통해 얻은 동일한 그래프와 비교했을 때 어떤 차이가 있습니까? (우리의 metric이 다른 방식으로 정의되었으므로, 두 결과 사이에 부호 차이가 있을 것으로 예상해야 함을 기억하십시오.)

<details>
<summary>몇 가지 생각들</summary>

이 그래프는 사실 activation patching으로 얻은 그래프와 거의 동일합니다 (새로운 metric으로 인해 결과의 부호가 반전된 점만 제외하면 말입니다).

이는 타당합니다. activation patching이 path patching과 다른 결과를 내는 유일한 이유는 `Mary - John` 방향으로 쓰는 head의 출력이 이후의 다른 head에 의해 사용될 때뿐입니다 (이 경우 activation patching에서는 이것이 반영되지만, path patching은 residual stream에 미치는 직접적인 영향만을 격리하기 때문입니다). attention head의 주된 목적이 모델 내에서 정보를 이동시키는 것이라는 점을 고려하면, 이런 일이 일어나지 않을 가능성이 높다고 추측하는 것이 합리적입니다.

하지만 걱정하지 마십시오. 다음 연습 세트에서는 더 흥미로운 path patching을 수행할 것이며, activation patching 결과와 유의미하게 다른 결과들을 얻게 될 것입니다.
</details>

## Path Patching: S-Inhibition Heads

path patching의 첫 번째 섹션에서는 attention head의 출력에서 residual stream의 최종 값으로 이어지는 단순한 형태의 patching을 수행했습니다. 여기서는 조금 더 흥미로운 작업을 수행하여, 한 head의 출력에서 이후에 나오는 head의 입력으로 patching을 진행하겠습니다. 이것의 목적은 두 head가 정확히 어떻게 composition되는지, 그리고 composition된 head들이 모델의 출력에 어떤 영향을 미치는지 조사하는 것입니다.

우리는 이전 섹션에서 S-inhibition head들의 value를 patching하여 그것들이 중요하다는 힌트를 얻었습니다. 하지만 이는 이러한 value 벡터들의 어떤 입력이 중요한지는 알려주지 않았습니다. 우리는 모델의 앞부분에 대한 분석을 바탕으로 이에 대해 추측해야만 했습니다. path patching을 통해, 우리는 어떤 head들이 중요한지 찾아내기 위해 더 정밀한 테스트를 수행할 수 있습니다.

path patching을 통한 논문의 결과는 7페이지의 그림 5(b)에 나와 있습니다.

### 연습 문제 - head에서 head로의 path patching 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 20-25 minutes on this exercise.
> 
> You'll need a new hook function, but copying code from the previous exercise should make this one easier.
> ```

아래의 함수 `get_path_patch_head_to_head`을 완성해야 합니다. 이 함수는 receiver node의 리스트(그리고 input의 타입 - keys, queries, 또는 values)를 인자로 받으며, 모델의 특정 head에서 모든 receiver head로 3단계 path patching 알고리즘을 적용한 후, 모델 출력에 대해 patching metric을 실행한 결과가 담긴 `(layer, head)` 모양의 tensor를 반환합니다. 논문의 결과(그림 5(b))를 재현할 수 있어야 합니다.

\*실제로 모든 layer를 반환할 필요는 없습니다. receiver head의 마지막 layer와 같거나 그 이후 layer에 있는 sender head로부터의 causal effect는 반드시 0이 되기 때문입니다.

더 많은 안내가 필요하다면, 아래의 드롭다운을 통해 이 함수가 첫 번째 path patching 함수와 어떻게 달라야 하는지 확인할 수 있습니다 (대부분의 방식은 비슷하므로, 해당 함수를 복사하는 것부터 시작하면 됩니다).

<details>
<summary>첫 번째 path patching 함수와의 차이점</summary>

1단계는 두 함수 모두 동일합니다 - 모든 observation을 수집합니다.

2단계는 매우 유사합니다. 유일한 차이점은 다른 activation 세트(receiver head들)를 캐싱한다는 점입니다.

3단계에서는 receiver node가 모델의 맨 끝이 아니라 중간에 위치하므로, 최종 residual stream의 patched value로부터 logit 출력을 직접 계산하는 대신, 이 node들을 patch한 상태로 모델을 다시 실행해야 합니다. 이를 위해 attention head의 input을 patch하는 새로운 hook 함수를 작성해야 합니다 (아직 작성하지 않은 경우).
</details>

In [ ]:
def patch_head_input(
    orig_activation: Float[Tensor, "batch pos head_idx d_head"],
    hook: HookPoint,
    patched_cache: ActivationCache,
    head_list: list[tuple[int, int]],
) -> Float[Tensor, "batch pos head_idx d_head"]:
    """
    Function which can patch any combination of heads in layers,
    according to the heads in head_list.
    """
    heads_to_patch = [head for layer, head in head_list if layer == hook.layer()]
    orig_activation[:, :, heads_to_patch] = patched_cache[hook.name][:, :, heads_to_patch]
    return orig_activation


def get_path_patch_head_to_heads(
    receiver_heads: list[tuple[int, int]],
    receiver_input: str,
    model: HookedTransformer,
    patching_metric: Callable,
    new_dataset: IOIDataset = abc_dataset,
    orig_dataset: IOIDataset = ioi_dataset,
    new_cache: ActivationCache | None = None,
    orig_cache: ActivationCache | None = None,
) -> Float[Tensor, "layer head"]:
    """
    Performs path patching (see algorithm in appendix B of IOI paper), with:

        sender head = (each head, looped through, one at a time)
        receiver node = input to a later head (or set of heads)

    The receiver node is specified by receiver_heads and receiver_input, for example if
    receiver_input = "v" and receiver_heads = [(8, 6), (8, 10), (7, 9), (7, 3)], we're doing path
    patching from each head to the value inputs of the S-inhibition heads.

    Returns:
        tensor of metric values for every possible sender head
    """
    model.reset_hooks()

    raise NotImplementedError()


model.reset_hooks()

s_inhibition_value_path_patching_results = get_path_patch_head_to_heads(
    receiver_heads=[(8, 6), (8, 10), (7, 9), (7, 3)],
    receiver_input="v",
    model=model,
    patching_metric=ioi_metric_2,
)

In [ ]:
imshow(
    100 * s_inhibition_value_path_patching_results,
    title="Direct effect on S-Inhibition Heads' values",
    labels={"x": "Head", "y": "Layer", "color": "Logit diff.<br>variation"},
    width=600,
    coloraxis=dict(colorbar_ticksuffix="%"),
)

<details>
<summary>질문 - 이 그래프의 해석은 무엇입니까? </summary>

이 그래프는 S-inhibition head의 value 벡터가 중요하다는 이전의 관찰 결과를 확인해 줍니다. 더 나아가, S-inhibition head의 value 벡터가 주로 head `0.1`, `3.0`, `5.5` 및 `6.9` (논문에서 각각 가장 중요한 두 개의 duplicate token head와 가장 중요한 두 개의 induction head로 찾아낸 head들입니다)의 출력에 의해 공급된다는 우리의 가설을 확인해 줍니다.
</details>


<details><summary>정답</summary>

```python
def patch_head_input(
    orig_activation: Float[Tensor, "batch pos head_idx d_head"],
    hook: HookPoint,
    patched_cache: ActivationCache,
    head_list: list[tuple[int, int]],
) -> Float[Tensor, "batch pos head_idx d_head"]:
    """
    Function which can patch any combination of heads in layers,
    according to the heads in head_list.
    """
    heads_to_patch = [head for layer, head in head_list if layer == hook.layer()]
    orig_activation[:, :, heads_to_patch] = patched_cache[hook.name][:, :, heads_to_patch]
    return orig_activation


def get_path_patch_head_to_heads(
    receiver_heads: list[tuple[int, int]],
    receiver_input: str,
    model: HookedTransformer,
    patching_metric: Callable,
    new_dataset: IOIDataset = abc_dataset,
    orig_dataset: IOIDataset = ioi_dataset,
    new_cache: ActivationCache | None = None,
    orig_cache: ActivationCache | None = None,
) -> Float[Tensor, "layer head"]:
    """
    Performs path patching (see algorithm in appendix B of IOI paper), with:

        sender head = (each head, looped through, one at a time)
        receiver node = input to a later head (or set of heads)

    The receiver node is specified by receiver_heads and receiver_input, for example if
    receiver_input = "v" and receiver_heads = [(8, 6), (8, 10), (7, 9), (7, 3)], we're doing path
    patching from each head to the value inputs of the S-inhibition heads.

    Returns:
        tensor of metric values for every possible sender head
    """
    model.reset_hooks()

    assert receiver_input in ("k", "q", "v")
    receiver_layers = set(next(zip(*receiver_heads)))
    receiver_hook_names = [utils.get_act_name(receiver_input, layer) for layer in receiver_layers]
    receiver_hook_names_filter = lambda name: name in receiver_hook_names

    results = t.zeros(max(receiver_layers), model.cfg.n_heads, device=device, dtype=t.float32)

    # ========== Step 1 ==========
    # Gather activations on x_orig and x_new

    # Note the use of names_filter for the run_with_cache function. Using it means we
    # only cache the things we need (in this case, just attn head outputs).
    z_name_filter = lambda name: name.endswith("z")
    if new_cache is None:
        _, new_cache = model.run_with_cache(new_dataset.toks, names_filter=z_name_filter, return_type=None)
    if orig_cache is None:
        _, orig_cache = model.run_with_cache(orig_dataset.toks, names_filter=z_name_filter, return_type=None)

    # Note, the sender layer will always be before the final receiver layer, otherwise there will
    # be no causal effect from sender -> receiver. So we only need to loop this far.
    for sender_layer, sender_head in tqdm(list(product(range(max(receiver_layers)), range(model.cfg.n_heads)))):
        # ========== Step 2 ==========
        # Run on x_orig, with sender head patched from x_new, every other head frozen

        hook_fn = partial(
            patch_or_freeze_head_vectors,
            new_cache=new_cache,
            orig_cache=orig_cache,
            head_to_patch=(sender_layer, sender_head),
        )
        model.add_hook(z_name_filter, hook_fn, level=1)

        _, patched_cache = model.run_with_cache(
            orig_dataset.toks, names_filter=receiver_hook_names_filter, return_type=None
        )
        # model.reset_hooks(including_permanent=True)
        assert set(patched_cache.keys()) == set(receiver_hook_names)

        # ========== Step 3 ==========
        # Run on x_orig, patching in the receiver node(s) from the previously cached value

        hook_fn = partial(
            patch_head_input,
            patched_cache=patched_cache,
            head_list=receiver_heads,
        )
        patched_logits = model.run_with_hooks(
            orig_dataset.toks,
            fwd_hooks=[(receiver_hook_names_filter, hook_fn)],
            return_type="logits",
        )

        # Save the results
        results[sender_layer, sender_head] = patching_metric(patched_logits)

    return results


model.reset_hooks()

s_inhibition_value_path_patching_results = get_path_patch_head_to_heads(
    receiver_heads=[(8, 6), (8, 10), (7, 9), (7, 3)],
    receiver_input="v",
    model=model,
    patching_metric=ioi_metric_2,
)
```
</details>

# 5️⃣ 전체 복제: 최소 회로 및 그 외

> ##### 학습 목표
>
> * [IOI paper](https://arxiv.org/abs/2211.00593)의 다른 결과물 대부분을 복제합니다.
> * 가이드가 적은, 더 개방적인 코딩을 연습합니다.

이 섹션은 훨씬 더 개방적이고 도전적일 것입니다. 연습 문제에서 제공되는 가이드가 다소 적을 것입니다.

## 복사 및 쓰기 방향 결과

이 섹션에서는 **name mover heads**와 **negative name mover heads**에 대한 논문의 분석을 재현하며 시작하겠습니다. 이전 분석을 통해 이러한 head들이 간접 목적어 token을 복사하거나 부정적으로 복사하고 있다는 점이 어느 정도 납득되었겠지만, 여기서 보여드릴 결과는 이를 조금 더 엄밀하게 증명합니다.

### 연습 문제 - writing direction 결과 재현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 20-25 minutes on this exercise.
> These exercises are much more challenging than they are conceptually important.
> ```

논문의 그림 3(c)를 살펴보겠습니다. 이 그래프는 가장 강한 name mover 및 negative name mover head의 출력을 `END`이 `IO` 또는 `S`에 attention을 주는 확률(색상으로 구분)에 대해 플롯한 것입니다.

몇 가지 명확한 설명은 다음과 같습니다:
* 여기서 "Projection"은 "dot product"와 동의어로 사용됩니다.
* 우리는 name embedding, 즉 attention을 받고 있는 token의 embedding 벡터로 projection을 수행합니다. 이는 logit diff(`IO`와 `S`의 unembedding 벡터 간의 차이로 head의 출력을 projection하여 얻은 값)와는 다릅니다.
    * 이렇게 하는 이유는 우리가 답하고자 하는 질문이 *"attention head가 자신이 attention을 주는 이름을 복사(또는 반대로 복사)하는가?"*이기 때문입니다.

아래 셀에 논문의 결과를 재현하는 코드를 작성해야 합니다. 특정 head의 결과(즉, `IO` 및 `S` token에 대한 projection과 attention 확률)를 저장하는 4개의 1D tensor가 주어졌을 때, 논문에 나오는 것과 유사한 plot을 생성하는 코드를 제공해 드렸습니다. 마찬가지로, 논문의 결과와 유사한 결과를 얻는다면 코드가 제대로 작동한 것입니다.

In [ ]:
def scatter_embedding_vs_attn(
    attn_from_end_to_io: Float[Tensor, "batch"],
    attn_from_end_to_s: Float[Tensor, "batch"],
    projection_in_io_dir: Float[Tensor, "batch"],
    projection_in_s_dir: Float[Tensor, "batch"],
    layer: int,
    head: int,
):
    scatter(
        x=t.concat([attn_from_end_to_io, attn_from_end_to_s], dim=0),
        y=t.concat([projection_in_io_dir, projection_in_s_dir], dim=0),
        color=["IO"] * N + ["S"] * N,
        title=f"Projection of the output of {layer}.{head} along the name<br>embedding vs attention probability on name",
        title_x=0.5,
        labels={"x": "Attn prob on name", "y": "Dot w Name Embed", "color": "Name type"},
        color_discrete_sequence=["#72FF64", "#C9A5F7"],
        width=650,
    )


def calculate_and_show_scatter_embedding_vs_attn(
    layer: int,
    head: int,
    cache: ActivationCache = ioi_cache,
    dataset: IOIDataset = ioi_dataset,
) -> None:
    """
    Creates and plots a figure equivalent to 3(c) in the paper.

    This should involve computing the four 1D tensors:
        attn_from_end_to_io
        attn_from_end_to_s
        projection_in_io_dir
        projection_in_s_dir
    and then calling the scatter_embedding_vs_attn function.
    """
    raise NotImplementedError()

In [ ]:
calculate_and_show_scatter_embedding_vs_attn(9, 9)  # name mover head 9.9

calculate_and_show_scatter_embedding_vs_attn(11, 10)  # negative name mover head 11.10

<details><summary>솔루션</summary>

```python
def scatter_embedding_vs_attn(
    attn_from_end_to_io: Float[Tensor, "batch"],
    attn_from_end_to_s: Float[Tensor, "batch"],
    projection_in_io_dir: Float[Tensor, "batch"],
    projection_in_s_dir: Float[Tensor, "batch"],
    layer: int,
    head: int,
):
    scatter(
        x=t.concat([attn_from_end_to_io, attn_from_end_to_s], dim=0),
        y=t.concat([projection_in_io_dir, projection_in_s_dir], dim=0),
        color=["IO"] * N + ["S"] * N,
        title=f"Projection of the output of {layer}.{head} along the name<br>embedding vs attention probability on name",
        title_x=0.5,
        labels={"x": "Attn prob on name", "y": "Dot w Name Embed", "color": "Name type"},
        color_discrete_sequence=["#72FF64", "#C9A5F7"],
        width=650,
    )


def calculate_and_show_scatter_embedding_vs_attn(
    layer: int,
    head: int,
    cache: ActivationCache = ioi_cache,
    dataset: IOIDataset = ioi_dataset,
) -> None:
    """
    Creates and plots a figure equivalent to 3(c) in the paper.

    This should involve computing the four 1D tensors:
        attn_from_end_to_io
        attn_from_end_to_s
        projection_in_io_dir
        projection_in_s_dir
    and then calling the scatter_embedding_vs_attn function.
    """
    # Get the value written to the residual stream at the end token by this head
    z = cache[utils.get_act_name("z", layer)][:, :, head]  # [batch seq d_head]
    N = z.size(0)
    output = z @ model.W_O[layer, head]  # [batch seq d_model]
    output_on_end_token = output[t.arange(N), dataset.word_idx["end"]]  # [batch d_model]

    # Get the directions we'll be projecting onto
    io_unembedding = model.W_U.T[dataset.io_tokenIDs]  # [batch d_model]
    s_unembedding = model.W_U.T[dataset.s_tokenIDs]  # [batch d_model]

    # Get the value of projections, by multiplying and summing over the d_model dimension
    projection_in_io_dir = (output_on_end_token * io_unembedding).sum(-1)  # [batch]
    projection_in_s_dir = (output_on_end_token * s_unembedding).sum(-1)  # [batch]

    # Get attention probs, and index to get the probabilities from END -> IO / S
    attn_probs = cache["pattern", layer][:, head]  # [batch seqQ seqK]
    attn_from_end_to_io = attn_probs[t.arange(N), dataset.word_idx["end"], dataset.word_idx["IO"]]  # [batch]
    attn_from_end_to_s = attn_probs[t.arange(N), dataset.word_idx["end"], dataset.word_idx["S1"]]  # [batch]

    # Show scatter plot
    scatter_embedding_vs_attn(
        attn_from_end_to_io,
        attn_from_end_to_s,
        projection_in_io_dir,
        projection_in_s_dir,
        layer,
        head,
    )

```
</details>

### 연습 문제 - copying score 결과 재현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 30-40 minutes on this exercise.
> These exercises are much more challenging than they are conceptually important.
> ```

이제 OV circuit을 직접 살펴봄으로써 name mover head의 copying을 다른 방식으로 테스트해 보겠습니다.

논문의 6페이지에서 발췌한 내용입니다:

> Name Mover Head가 일반적으로 이름을 복사하는지 확인하기 위해, 우리는 head의 OV matrix를 통해 어떤 값들이 쓰이는지 연구했습니다. 구체적으로, 먼저 첫 번째 MLP layer 이후 각 name token 위치에서의 residual stream 상태를 얻었습니다. 그런 다음, 이를 Name Mover Head의 OV matrix와 곱하고(head가 해당 token에 완벽하게 attention을 기울였을 때 일어날 일을 시뮬레이션함), unembedding matrix를 곱한 뒤, 최종 layer norm을 적용하여 logit 확률을 얻었습니다. 우리는 상위 5개 logit(N = 1000)에 입력 name token이 포함된 샘플의 비율을 계산하고 이를 copy score라고 부릅니다. 세 개의 Name Mover Head 모두 95% 이상의 copy score를 보였으며, 이는 평균적인 head의 20% 미만과 대조적입니다.
>
> Negative Name Mover Head는 ... 큰 음수 copy score를 가집니다. 즉, OV matrix의 음수 값을 사용하여 계산한 copy score가 높게 나타났습니다(평균 head의 12% 대비 98%).

이들의 방법과 이전 연습 문제에서 induction head의 copying을 연구했던 방식 사이의 유사점에 주목하십시오. 하지만 차이점도 존재합니다 (예를 들어, 여기서는 token 일반이 아니라 head가 이름을 복사하는지 여부만을 살펴보고 있습니다).

아래의 `get_copying_scores` 함수를 완성하여 이 결과들을 재현해야 합니다.

`ioi_cache`에서 인덱싱하여 이를 수행할 수도 있지만, 훨씬 더 원칙적인 대안은 `NAMES` 리스트에 있는 모든 이름을 embedding하고 MLP, layernorm, OV matrix와 같은 연산들을 수동으로 적용하는 것입니다. 이것이 정답 코드에서 사용하는 방식입니다.

몇 가지 참고 사항입니다:

* 이름을 token으로 변환하기 위해 `model.to_tokens`을 사용할 수 있습니다. 이름을 embedding하기 위해 token들만 필요하므로 `prepend_bos=False`를 사용하는 것을 기억하십시오. 이 함수는 이름 리스트를 단일 token 입력의 batch로 처리하며, 이는 우리의 목적에 적합합니다.
* 모델의 block들을 인덱싱함으로써 MLP와 layernorm을 함수로 적용할 수 있습니다 (예: `model.blocks[i].mlp` 또는 `model.blocks[j].ln1`을 함수로 사용). `ln1`은 attention 앞에 오는 layernorm이며, `ln2`은 MLP 앞에 온다는 점을 기억하십시오.
* OV matrix를 적용하기 전에 MLP0를 적용해야 한다는 점을 기억하십시오 (이것이 우리가 score에서 0번째 layer를 제외하는 이유입니다). 그 이유는 gpt2-small에서 MLP0를 ablate하는 것이 다른 MLP들을 ablate하는 것에 비해 이상할 정도로 큰 영향을 미치기 때문이며, 이는 아마도 MLP0가 확장된 embedding 역할을 하기 때문일 것입니다 (설명은 [here](https://www.lesswrong.com/s/yivyHaCAmMJ3CqSyj/p/XNjRwEX9kxbpzWFWd#:~:text=GPT%2D2%20Small%E2%80%99s%20performance%20is%20ruined%20if%20you%20ablate%20MLP0)를 참조하십시오).

또한, 이 실험의 일부 설정이 매우 약간 다르게 구성되었기 때문에 논문과 완전히 동일한 결과를 얻을 것이라고 기대해서는 안 됩니다. 하지만 아마도 10% 이상의 오차는 발생하지 않을 것입니다.

In [ ]:
def get_copying_scores(model: HookedTransformer, k: int = 5, names: list = NAMES) -> Float[Tensor, "2 layer-1 head"]:
    """
    Gets copying scores (both positive and negative) as described in page 6 of the IOI paper, for
    every (layer, head) pair in the model.

    Returns these in a 3D tensor (the first dimension is for positive vs negative).

    Omits the 0th layer, because this is before MLP0 (which we're claiming acts as an extended
    embedding).
    """
    raise NotImplementedError()

In [ ]:
copying_results = get_copying_scores(model)

imshow(
    copying_results,
    facet_col=0,
    facet_labels=["Positive copying scores", "Negative copying scores"],
    title="Copying scores of attention heads' OV circuits",
    width=800
)

In [ ]:
heads = {"name mover": [(9, 9), (10, 0), (9, 6)], "negative name mover": [(10, 7), (11, 10)]}

for i, name in enumerate(["name mover", "negative name mover"]):
    make_table(
        title=f"Copying Scores ({name} heads)",
        colnames=["Head", "Score"],
        cols=[
            list(map(str, heads[name])) + ["[dark_orange bold]Average"],
            [f"{copying_results[i, layer-1, head]:.2%}" for (layer, head) in heads[name]] + [f"[dark_orange bold]{copying_results[i].mean():.2%}"]
        ]
    )

<details><summary>솔루션</summary>

```python
def get_copying_scores(model: HookedTransformer, k: int = 5, names: list = NAMES) -> Float[Tensor, "2 layer-1 head"]:
    """
    Gets copying scores (both positive and negative) as described in page 6 of the IOI paper, for
    every (layer, head) pair in the model.

    Returns these in a 3D tensor (the first dimension is for positive vs negative).

    Omits the 0th layer, because this is before MLP0 (which we're claiming acts as an extended
    embedding).
    """
    results = t.zeros((2, model.cfg.n_layers, model.cfg.n_heads), device=device)

    # Define components from our model (for typechecking, and cleaner code)
    embed: Embed = model.embed
    mlp0: MLP = model.blocks[0].mlp
    ln0: LayerNorm = model.blocks[0].ln2
    unembed: Unembed = model.unembed
    ln_final: LayerNorm = model.ln_final

    # Get embeddings for the names in our list
    name_tokens: Int[Tensor, "batch 1"] = model.to_tokens(names, prepend_bos=False)
    name_embeddings: Int[Tensor, "batch 1 d_model"] = embed(name_tokens)

    # Get residual stream after applying MLP
    resid_after_mlp1 = name_embeddings + mlp0(ln0(name_embeddings))

    # Loop over all (layer, head) pairs
    for layer in range(1, model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            # Get W_OV matrix
            W_OV = model.W_V[layer, head] @ model.W_O[layer, head]

            # Get residual stream after applying W_OV or -W_OV respectively
            # (note, because of bias b_U, it matters that we do sign flip here, not later)
            resid_after_OV_pos = resid_after_mlp1 @ W_OV
            resid_after_OV_neg = resid_after_mlp1 @ -W_OV

            # Get logits from value of residual stream
            logits_pos = unembed(ln_final(resid_after_OV_pos)).squeeze()  # [batch d_vocab]
            logits_neg = unembed(ln_final(resid_after_OV_neg)).squeeze()  # [batch d_vocab]

            # Check how many are in top k
            topk_logits: Int[Tensor, "batch k"] = t.topk(logits_pos, dim=-1, k=k).indices
            in_topk = (topk_logits == name_tokens).any(-1)
            # Check how many are in bottom k
            bottomk_logits: Int[Tensor, "batch k"] = t.topk(logits_neg, dim=-1, k=k).indices
            in_bottomk = (bottomk_logits == name_tokens).any(-1)

            # Fill in results
            results[:, layer - 1, head] = t.tensor([in_topk.float().mean(), in_bottomk.float().mean()])

    return results


copying_results = get_copying_scores(model)

imshow(
    copying_results,
    facet_col=0,
    facet_labels=["Positive copying scores", "Negative copying scores"],
    title="Copying scores of attention heads' OV circuits",
    width=900,
)

heads = {"name mover": [(9, 9), (10, 0), (9, 6)], "negative name mover": [(10, 7), (11, 10)]}

for i, name in enumerate(["name mover", "negative name mover"]):
    make_table(
        title=f"Copying Scores ({name} heads)",
        colnames=["Head", "Score"],
        cols=[
            list(map(str, heads[name])) + ["[dark_orange bold]Average"],
            [f"{copying_results[i, layer - 1, head]:.2%}" for (layer, head) in heads[name]]
            + [f"[dark_orange bold]{copying_results[i].mean():.2%}"],
        ],
    )
```
</details>

## 초기 head들의 검증

circuit의 초기에 나타나는 세 가지 서로 다른 종류의 head가 있으며, 이는 단순한 무작위 token 시퀀스에 대한 attention pattern을 살펴봄으로써 검증할 수 있습니다. 이 세 가지 유형이 무엇인지, 그리고 이러한 방식으로 어떻게 검증할 수 있는지 알아낼 수 있습니까?

<details>
<summary>정답</summary>

Previous token head, induction head, 그리고 duplicate token head입니다.

우리는 `n` 개의 무작위 token 뒤에 동일한 `n` 개의 무작위 token이 반복되는 시퀀스를 사용하여 이들을 동시에 검증할 수 있습니다. 작동 방식은 다음과 같습니다:

* Prev token head: 오프셋이 1인 attention pattern(즉, 대각선 바로 아래)을 측정함으로써 확인합니다.
* Induction head: 오프셋이 `n-1` 인 attention pattern(즉, token의 두 번째 인스턴스가 첫 번째 인스턴스 다음의 token에 attention을 기울이는 경우)을 측정함으로써 확인합니다.
* Duplicate token head: 오프셋이 `n` 인 attention pattern(즉, token이 자신의 이전 인스턴스에 attention을 기울이는 경우)을 측정함으로써 확인합니다.

세 가지 경우 모두, head가 이러한 지표에서 1에 가까운 점수를 기록한다면, 해당 head가 이 유형으로 작동하고 있다는 강력한 증거가 됩니다.

참고로, 우리는 특정 분포에서만 이를 관찰하고 있기 때문에 "head X는 induction head이다"라고 말하는 것은 leaky abstraction입니다. 예를 들어, 중복된 token이 없을 때 induction head와 duplicate token head의 역할이 무엇인지는 명확하지 않습니다 (이론적으로는 완전히 다른 일을 할 수도 있습니다).
</details>

### 연습 문제 - head validation 수행하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 20-30 minutes on this exercise.
> Understanding how to identify certain types of heads by their characteristic attention patterns is important.
> ```

위의 드롭다운에서 정답을 읽으셨다면, 이제 이 validation을 수행해야 합니다. 결과는 논문의 Figure 18을 재현한 모습이어야 합니다.

이 함수를 위한 템플릿을 제공했습니다. 인자가 다음 옵션 중 하나여야 함을 나타내는 `typing.Literal`의 사용법에 유의하시기 바랍니다.

또한 헬퍼 함수 `generate_repeated_tokens`(연습 문제 세트 1.2에서 사용한 것과 유사하지만, 논문과 일치시키기 위해 start token이 없습니다)와, `get_attn_scores` 함수를 호출하여 결과를 플롯하는(Figure 18과 유사하게 보이도록 하는) 헬퍼 함수 `plot_early_head_validation_results`을 제공했습니다. 따라서 여러분이 작성해야 할 부분은 `get_attn_scores` 함수뿐입니다.

In [ ]:
def generate_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch: int = 1
) -> Float[Tensor, "batch 2*seq_len"]:
    """
    Generates a sequence of repeated random tokens (no start token).
    """
    rep_tokens_half = t.randint(0, model.cfg.d_vocab, (batch, seq_len), dtype=t.int64)
    rep_tokens = t.cat([rep_tokens_half, rep_tokens_half], dim=-1).to(device)
    return rep_tokens


def get_attn_scores(
    model: HookedTransformer,
    seq_len: int,
    batch: int,
    head_type: Literal["duplicate", "prev", "induction"],
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Returns attention scores for sequence of duplicated tokens, for every head.
    """
    raise NotImplementedError()


def plot_early_head_validation_results(seq_len: int = 50, batch: int = 50):
    """
    Produces a plot that looks like Figure 18 in the paper.
    """
    head_types = ["duplicate", "prev", "induction"]

    results = t.stack([get_attn_scores(model, seq_len, batch, head_type=head_type) for head_type in head_types])

    imshow(
        results,
        facet_col=0,
        facet_labels=[
            f"{head_type.capitalize()} token attention prob.<br>on sequences of random tokens"
            for head_type in head_types
        ],
        labels={"x": "Head", "y": "Layer"},
        width=1300,
    )

In [ ]:
model.reset_hooks()
plot_early_head_validation_results()

<details><summary>솔루션</summary>

```python
def generate_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch: int = 1
) -> Float[Tensor, "batch 2*seq_len"]:
    """
    Generates a sequence of repeated random tokens (no start token).
    """
    rep_tokens_half = t.randint(0, model.cfg.d_vocab, (batch, seq_len), dtype=t.int64)
    rep_tokens = t.cat([rep_tokens_half, rep_tokens_half], dim=-1).to(device)
    return rep_tokens


def get_attn_scores(
    model: HookedTransformer,
    seq_len: int,
    batch: int,
    head_type: Literal["duplicate", "prev", "induction"],
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Returns attention scores for sequence of duplicated tokens, for every head.
    """
    rep_tokens = generate_repeated_tokens(model, seq_len, batch)

    _, cache = model.run_with_cache(rep_tokens, return_type=None, names_filter=lambda name: name.endswith("pattern"))

    # Get the right indices for the attention scores

    if head_type == "duplicate":
        src_indices = range(seq_len)
        dest_indices = range(seq_len, 2 * seq_len)
    elif head_type == "prev":
        src_indices = range(seq_len)
        dest_indices = range(1, seq_len + 1)
    elif head_type == "induction":
        dest_indices = range(seq_len, 2 * seq_len)
        src_indices = range(1, seq_len + 1)

    results = t.zeros(model.cfg.n_layers, model.cfg.n_heads, device=device, dtype=t.float32)
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attn_scores = cache["pattern", layer]  # [batch seqQ seqK]
            avg_attn_on_duplicates = attn_scores[:, head, dest_indices, src_indices].mean().item()
            results[layer, head] = avg_attn_on_duplicates

    return results


def plot_early_head_validation_results(seq_len: int = 50, batch: int = 50):
    """
    Produces a plot that looks like Figure 18 in the paper.
    """
    head_types = ["duplicate", "prev", "induction"]

    results = t.stack([get_attn_scores(model, seq_len, batch, head_type=head_type) for head_type in head_types])

    imshow(
        results,
        facet_col=0,
        facet_labels=[
            f"{head_type.capitalize()} token attention prob.<br>on sequences of random tokens"
            for head_type in head_types
        ],
        labels={"x": "Head", "y": "Layer"},
        width=1300,
    )


model.reset_hooks()
plot_early_head_validation_results()
```
</details>

참고 - 이 수치들은 모델의 head들에 대한 "위키"를 구축하는 것이 유용한 인프라가 될 수 있음을 시사합니다. 여기서 살펴본 것과 같이 head 기능에 관한 특정 메트릭에 따른 점수를 제공하는 방식입니다. `HookedTransformer`를 사용하면 이를 매우 쉽게 만들 수 있습니다. `HookedTransformer.from_pretrained`에 입력하는 이름만 바꾸면 동일한 아키텍처 내에서 다른 모델이 되므로, 동일한 코드가 작동해야 합니다. 만약 이것을 만들고 싶으시다면, 꼭 보고 싶습니다!

개념 증명(proof of concept)으로서, [I made a mosaic of all induction heads across the 40 models then in HookedTransformer](https://www.neelnanda.io/mosaic).

## 최소 회로 (Minimal Circuit)

### 배경: faithfulness, completeness, 그리고 minimality

저자들은 circuit 설명의 타당성을 검증하기 위해 faithfulness, completeness, minimality라는 세 가지 기준을 개발했습니다. 정의는 다음과 같습니다:

* **Faithful** = circuit이 전체 모델만큼 성능을 낼 수 있음
* **Complete** = circuit이 태스크 수행에 사용되는 모든 노드를 포함함
* **Minimal** = circuit이 태스크와 무관한 노드를 포함하지 않음

이 세 가지 기준을 모두 충족한다면, 해당 circuit은 모델 동작에 대한 신뢰할 수 있는 설명으로 간주됩니다.

연습 문제 - 왜 이러한 각 기준이 중요한지 이해하시겠습니까? 각 기준의 쌍에 대해, circuit이 두 가지는 충족하지만 세 번째는 충족하지 못하는 것이 가능할까요? (가능하다면 예시를 들어주실 수 있습니까?)

<details>
<summary>정답</summary>

naive circuit(전체 모델을 포함하는 경우)은 당연히 faithful하고 complete하지만, 분명히 minimal하지는 않습니다. 일반적으로 non-minimal circuit의 문제는 기계론적으로 이해하기 어려울 수 있다는 점이며, 이는 이러한 종류의 circuit 분석 목적에 어긋납니다.

completeness는 당연히 faithfulness를 함의합니다. 왜냐하면 어떤 노드가 태스크에 관여하지 않는다면, 해당 태스크에 대한 모델의 성능을 향상시킬 수 없기 때문입니다.

처음에는 faithfulness가 completeness를 함의한다고 생각할 수 있지만, 이는 사실이 아닙니다. backup name mover heads가 이 점을 잘 보여줍니다. 이들은 태스크에 사용되며, 이들이 수행하는 역할을 이해하지 못하면 현실에 대해 잘못된 모델을 갖게 됩니다 (예를 들어, name mover heads를 ablate하면 성능이 파괴될 것이라고 생각하겠지만, 실제로는 그렇지 않습니다). 따라서 backup name mover heads를 포함하지 않는 circuit을 정의한다면, 이는 faithful하겠지만 (backup name mover heads가 사용되지 않으므로) complete하지는 않을 것입니다.

요약:

* **Faithful & complete, not minimal** = 가능 (예시: naive circuit)
* **Faithful & minimal, not complete** = 가능 (예시: backup name mover heads가 누락된 circuit)
* **Complete & minimal, not faithful** = 불가능 (completeness가 faithfulness를 함의하므로)

논문에서 저자들은 이러한 개념들을 공식화합니다. Faithfulness는 $|F(C) - F(M)|$이 작은 것과 동일하며 (여기서 $C$는 우리의 circuit, $M$는 모델, $F$은 성능 측정 함수입니다), completeness는 **모든 부분 집합 $K \subset C$에 대해** $|F(C\backslash K) - F(M\backslash K)|$이 작은 것과 동일합니다 ($K$가 공집합인 경우를 포함하며, 이는 completeness가 faithfulness를 함의함을 보여줍니다). backup name mover heads가 누락된 우리의 circuit이 어떻게 이 조건을 위반하는지 알 수 있겠습니까?

<details>
<summary>정답</summary>

$K$이 name mover heads의 집합일 때 이 조건을 위반합니다. $C \backslash K$는 $M \backslash K$보다 성능이 떨어지는데, 그 이유는 후자는 backup name mover heads를 포함하고 있는 반면 전자는 name mover heads와 backup name mover heads를 ***모두*** 잃었기 때문입니다.
</details>
</details>

이제 회로의 대부분의 구성 요소를 분석했고 그것들이 어떻게 작동하는지에 대한 대략적인 아이디어를 얻었으므로, 다음 단계는 핵심 구성 요소들을 제외한 모든 것을 ablate하고 모델이 여전히 잘 작동하는지 확인하는 것입니다.

이 ablation은 상당히 대규모로 진행됩니다. 단일 시퀀스 위치(예를 들어 DTH의 경우 `S2` token, SIH의 경우 `end` token)에서 각 핵심 attention head(예: duplicate token head 또는 S-inhibition head)의 출력을 제외한 모든 것을 ablate합니다. 우리의 핵심 회로가 총 26개의 head를 가지고 있고 시퀀스 길이가 평균적으로 약 20인 점을 고려하면, 이는 attention head 출력의 $(26 / 144) / 20 \approx 1\%$를 제외한 모든 것을 ablate한다는 의미입니다 (그리고 모델을 통과하는 가능한 경로의 수는 이보다 ***훨씬*** 더 많이 감소합니다).

어떻게 ablation을 수행할까요? zero-ablation을 사용할 수 있지만, 이는 사실 몇 가지 명확하지 않은 문제점들이 있습니다. 직관적으로 설명하자면, head들이 0이 아닌 입력을 "기대"하고 있을 수 있으며, 입력을 0으로 설정하는 것은 본질적으로 임의적인 선택이 되어 이를 distribution 밖으로 밀어내는 결과가 됩니다. 이는 해당 head에 bias 항을 추가하는 것과 같다고 생각할 수 있으며, 이는 후속 계산을 망가뜨려 노이즈가 섞인 결과로 이어질 수 있습니다. mean-ablation(즉, head의 출력을 `ioi_dataset`에 걸친 평균 출력으로 설정)을 사용할 수도 있지만, 여기서 문제는 이 데이터셋에 대해 평균을 내는 것이 IOI task를 해결하는 데 필요한 관련 정보를 포함하고 있을 수 있다는 점입니다. 예를 들어, `S2`에 기록되는 `is_duplicated` flag는 모든 sequence에 존재하므로, 평균을 내더라도 이 정보가 제거되지 않습니다.

이 문제를 해결할 방법을 생각해보시겠습니까? 고민해 보신 후, 아래 드롭다운을 통해 저자들이 이 문제를 어떻게 처리했는지 확인해 보시기 바랍니다.

<details>
<summary>정답</summary>

IOI 데이터셋이 아니라 ABC 데이터셋의 평균을 사용하여 ablation을 수행합니다. 이렇게 하면 평균값에 IOI task 해결을 위한 관련 정보가 여전히 포함되어 있는 문제를 제거할 수 있습니다.

</details>

또 다른 복잡한 점이 있습니다. 문장들이 서로 다른 template을 가지고 있으며, `S` 및 `IO`과 같은 token들의 위치가 이러한 template 전반에 걸쳐 일관되지 않습니다 (이전 연습 문제에서는 모든 중요한 token들이 동일한 index를 갖는 매우 작은 문장 집합을 선택함으로써 이 문제를 피했습니다). 위치가 서로 다른 두 template의 예시는 다음과 같습니다:

```
"Then, [B] and [A] had a long argument and after that [B] said to [A]"
"After the lunch [B] and [A] went to the [PLACE], and [B] gave a [OBJECT] to [A]"
```

저자들이 이 문제를 해결하기 위해 무엇을 했는지 추측할 수 있습니까?

<details>
<summary>정답</summary>

저자들은 전체 데이터셋이 아니라 각 template에 대해 평균을 구했고, 이 값들을 사용하여 ablation을 수행했습니다.

다시 말해, head의 출력(shape `(batch, seq, d_model)`)을 patching하여 ablation을 수행할 때, 이 tensor의 `(i, j, k)`번째 요소에 patching되는 값은 배치 내의 `i`번째 문장과 동일한 template을 가진 모든 문장에 대해, sequence 위치 `j`에 있는 벡터의 `k`번째 요소의 평균값이 됩니다.
</details>

### 연습 문제 - 최소 회로(minimal circuit) 구성하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵⚪⚪⚪
> 
> This exercise is expected to take a long time; at least an hour. It is probably the most challenging exercise in this notebook.
> ```

이제 모델에 ablation을 수행하여 최소 회로(minimal circuit)를 얻기 위한 충분한 정보를 가지고 있습니다. 아래에서 이를 직접 구현해 보실 수 있습니다.

이 실습은 기술적으로 매우 도전적인 과제이므로, 흥미가 없으시다면 건너뛰셔도 좋습니다. 하지만 이 ablation이 어떻게 작동하는지 대략적인 윤곽을 이해하기 위해 솔루션을 읽어보시는 것을 권장합니다.

이 과제에 도전하고 싶으시다면, 아래 코드로 시작하실 수 있습니다. 여기서는 두 개의 딕셔너리를 정의합니다. 하나는 head 타입과 해당 타입에 속하는 모델 내의 head들을 매핑하는 것이고, 다른 하나는 head 타입과 해당 타입의 head에 대해 ablation을 수행하지 *않을* 시퀀스 위치를 매핑하는 것입니다.

In [ ]:
CIRCUIT = {
    "name mover": [(9, 9), (10, 0), (9, 6)],
    "backup name mover": [(10, 10), (10, 6), (10, 2), (10, 1), (11, 2), (9, 7), (9, 0), (11, 9)],
    "negative name mover": [(10, 7), (11, 10)],
    "s2 inhibition": [(7, 3), (7, 9), (8, 6), (8, 10)],
    "induction": [(5, 5), (5, 8), (5, 9), (6, 9)],
    "duplicate token": [(0, 1), (0, 10), (3, 0)],
    "previous token": [(2, 2), (4, 11)],
}

SEQ_POS_TO_KEEP = {
    "name mover": "end",
    "backup name mover": "end",
    "negative name mover": "end",
    "s2 inhibition": "end",
    "induction": "S2",
    "duplicate token": "S2",
    "previous token": "S1+1",
}

명확히 하자면, 우리가 mean-ablation을 수행할 대상은 다음과 같습니다:

* `CIRCUIT` dict에 포함되지 않은 모든 head
* `CIRCUIT` dict에 포함된 head들의 모든 sequence position 중, `SEQ_POS_TO_KEEP` dict에 의해 지정된 sequence position을 제외한 나머지

그리고 우리는 head의 출력을 `abc_dataset`에 대한 평균 출력으로 대체함으로써 mean-ablation을 수행하며, 이때 배치의 문장과 동일한 template을 가진 모든 문장에 대해 평균을 냅니다. `dataset.groups` attribute를 사용하여 데이터셋의 template에 접근할 수 있으며, 이는 (동일한 template을 공유하는 배치의 sequence 인덱스를 포함하는) tensor 리스트를 반환합니다.

이제 다음 함수를 완성해 보십시오. 이 함수는 모델이 `ioi_dataset`에서 실행될 때마다 이 ablation을 수행하는 ***permanent hook***을 추가해야 합니다 (이 hook은 모델이 이 데이터셋에서 실행될 때만 의미가 있으므로, 다른 데이터셋에서 실행할 경우 hook을 리셋해야 한다는 점에 유의하십시오).

Permanent hook은 transformerlens의 유용한 기능입니다. 이는 일반적인 hook과 동일하게 작동하지만, 모델을 실행할 때(예: `model.run_with_cache` 또는 `model.run_with_hooks` 사용 시) 제거되지 않는다는 점이 다릅니다. 이를 제거하는 유일한 방법은 다음과 같습니다:

```python
model.reset_hooks(including_permanent=True)
```

Permanent hook은 다음과 같이 추가할 수 있습니다:

```python
model.add_hook(hook_name, hook_fn, is_permanent=True)
```

여기서 `hook_name`는 문자열이거나 문자열을 boolean으로 매핑하는 필터 함수일 수 있습니다.

In [ ]:
def add_mean_ablation_hook(
    model: HookedTransformer,
    means_dataset: IOIDataset,
    circuit: dict[str, list[tuple[int, int]]] = CIRCUIT,
    seq_pos_to_keep: dict[str, str] = SEQ_POS_TO_KEEP,
    is_permanent: bool = True,
) -> HookedTransformer:
    """
    Adds a permanent hook to the model, which ablates according to the circuit and seq_pos_to_keep
    dictionaries.

    In other words, when the model is run on ioi_dataset, every head's output will be replaced with
    the mean over means_dataset for sequences with the same template, except for a subset of heads
    and sequence positions as specified by the circuit and seq_pos_to_keep dicts.
    """
    raise NotImplementedError()

<details>
<summary>힌트 (메인 함수에 유용하게 사용될 일부 함수들의 docstring입니다)</summary>

```python
def compute_means_by_template(
    means_dataset: IOIDataset,
    model: HookedTransformer
) -> Float[Tensor, "layer batch seq head_idx d_head"]:
    '''
    Returns the mean of each head's output over the means dataset. This mean is
    computed separately for each group of prompts with the same template (these
    are given by means_dataset.groups).
    '''
    pass


def get_heads_and_posns_to_keep(
    means_dataset: IOIDataset,
    model: HookedTransformer,
    circuit: dict[str, list[tuple[int, int]]],
    seq_pos_to_keep: dict[str, str],
) -> dict[int, Bool[Tensor, "batch seq head"]]:
    '''
    Returns a dictionary mapping layers to a boolean mask giving the indices of the
    z output which *shouldn't* be mean-ablated.

    The output of this function will be used for the hook function that does ablation.
    '''
    pass


def hook_fn_mask_z(
    z: Float[Tensor, "batch seq head d_head"],
    hook: HookPoint,
    heads_and_posns_to_keep: dict[int, Bool[Tensor, "batch seq head"]],
    means: Float[Tensor, "layer batch seq head d_head"],
) -> Float[Tensor, "batch seq head d_head"]:
    '''
    Hook function which masks the z output of a transformer head.

    heads_and_posns_to_keep
        dict created with the get_heads_and_posns_to_keep function. This tells
        us where to mask.

    means
        Tensor of mean z values of the means_dataset over each group of prompts
        with the same template. This tells us what values to mask with.
    '''
    pass
```

이 세 가지 함수를 모두 작성하고 나면, 메인 함수를 완성하는 것은 간단합니다. 메인 함수는 다음과 같이 동작해야 합니다:

* `compute_means_by_template`를 사용하여 평균(means)을 구합니다.
* `get_heads_and_posns_to_keep`를 사용하여 boolean mask를 구합니다.
* 앞선 두 함수의 출력값을 사용하여 `hook_fn_mask_z`에 `functools.partial`을 적용함으로써, mean ablation을 수행하는 hook 함수를 생성합니다.
* 이 hook 함수를 모델에 permanent hook으로 추가합니다.

</details>


<details><summary>정답</summary>

```python
def get_heads_and_posns_to_keep(
    means_dataset: IOIDataset,
    model: HookedTransformer,
    circuit: dict[str, list[tuple[int, int]]],
    seq_pos_to_keep: dict[str, str],
) -> dict[int, Bool[Tensor, "batch seq head"]]:
    """
    Returns a dictionary mapping layers to a boolean mask giving the indices of the z output which
    *shouldn't* be mean-ablated.

    The output of this function will be used for the hook function that does ablation.
    """
    heads_and_posns_to_keep = {}
    batch_size, seq_len, n_heads = len(means_dataset), means_dataset.max_len, model.cfg.n_heads

    for layer in range(model.cfg.n_layers):
        mask = t.zeros(size=(batch_size, seq_len, n_heads))

        for head_type, head_list in circuit.items():
            seq_pos = seq_pos_to_keep[head_type]
            indices = means_dataset.word_idx[seq_pos]
            for layer_idx, head_idx in head_list:
                if layer_idx == layer:
                    mask[range(batch_size), indices, head_idx] = 1

        heads_and_posns_to_keep[layer] = mask.bool()

    return heads_and_posns_to_keep


def hook_fn_mask_z(
    z: Float[Tensor, "batch seq head d_head"],
    hook: HookPoint,
    heads_and_posns_to_keep: dict[int, Bool[Tensor, "batch seq head"]],
    means: Float[Tensor, "layer batch seq head d_head"],
) -> Float[Tensor, "batch seq head d_head"]:
    """
    Hook function which masks the z output of a transformer head.

    heads_and_posns_to_keep
        dict created with the get_heads_and_posns_to_keep function. This tells us where to mask.

    means
        Tensor of mean z values of the means_dataset over each group of prompts with the same
        template. This tells us
        what values to mask with.
    """
    # Get the mask for this layer, and add d_head=1 dimension so it broadcasts correctly
    mask_for_this_layer = heads_and_posns_to_keep[hook.layer()].unsqueeze(-1).to(z.device)

    # Set z values to the mean
    z = t.where(mask_for_this_layer, z, means[hook.layer()])

    return z


def compute_means_by_template(
    means_dataset: IOIDataset, model: HookedTransformer
) -> Float[Tensor, "layer batch seq head_idx d_head"]:
    """
    Returns the mean of each head's output over the means dataset. This mean is computed separately
    for each group of prompts with the same template (these are given by means_dataset.groups).
    """
    # Cache the outputs of every head
    _, means_cache = model.run_with_cache(
        means_dataset.toks.long(),
        return_type=None,
        names_filter=lambda name: name.endswith("z"),
    )
    # Create tensor to store means
    n_layers, n_heads, d_head = model.cfg.n_layers, model.cfg.n_heads, model.cfg.d_head
    batch, seq_len = len(means_dataset), means_dataset.max_len
    means = t.zeros(size=(n_layers, batch, seq_len, n_heads, d_head), device=model.cfg.device)

    # Get set of different templates for this data
    for layer in range(model.cfg.n_layers):
        z_for_this_layer = means_cache[utils.get_act_name("z", layer)]  # [batch seq head d_head]
        for template_group in means_dataset.groups:
            z_for_this_template = z_for_this_layer[template_group]
            z_means_for_this_template = einops.reduce(
                z_for_this_template, "batch seq head d_head -> seq head d_head", "mean"
            )
            means[layer, template_group] = z_means_for_this_template

    return means


def add_mean_ablation_hook(
    model: HookedTransformer,
    means_dataset: IOIDataset,
    circuit: dict[str, list[tuple[int, int]]] = CIRCUIT,
    seq_pos_to_keep: dict[str, str] = SEQ_POS_TO_KEEP,
    is_permanent: bool = True,
) -> HookedTransformer:
    """
    Adds a permanent hook to the model, which ablates according to the circuit and seq_pos_to_keep
    dictionaries.

    In other words, when the model is run on ioi_dataset, every head's output will be replaced with
    the mean over means_dataset for sequences with the same template, except for a subset of heads
    and sequence positions as specified by the circuit and seq_pos_to_keep dicts.
    """
    model.reset_hooks(including_permanent=True)

    # Compute the mean of each head's output on the ABC dataset, grouped by template
    means = compute_means_by_template(means_dataset, model)

    # Convert this into a boolean map
    heads_and_posns_to_keep = get_heads_and_posns_to_keep(means_dataset, model, circuit, seq_pos_to_keep)

    # Get a hook function which will patch in the mean z values for each head, at
    # all positions which aren't important for the circuit
    hook_fn = partial(hook_fn_mask_z, heads_and_posns_to_keep=heads_and_posns_to_keep, means=means)

    # Apply hook
    model.add_hook(lambda name: name.endswith("z"), hook_fn, is_permanent=is_permanent)

    return model
```
</details>

작성하신 함수가 제대로 작동하는지 테스트하려면, 제공된 함수를 사용하여 구현하신 circuit의 logit 차이가 이 결과와 일치하는지 확인하시면 됩니다:

In [ ]:
import part41_indirect_object_identification.ioi_circuit_extraction as ioi_circuit_extraction

model = ioi_circuit_extraction.add_mean_ablation_hook(
    model,
    means_dataset=abc_dataset,
    circuit=CIRCUIT,
    seq_pos_to_keep=SEQ_POS_TO_KEEP,
)
ioi_logits_minimal = model(ioi_dataset.toks)

print(f"""
Avg logit diff (IOI dataset, using entire model): {logits_to_ave_logit_diff_2(ioi_logits_original):.4f}
Avg logit diff (IOI dataset, only using circuit): {logits_to_ave_logit_diff_2(ioi_logits_minimal):.4f}
""")

In [ ]:
model = add_mean_ablation_hook(
    model,
    means_dataset=abc_dataset,
    circuit=CIRCUIT,
    seq_pos_to_keep=SEQ_POS_TO_KEEP,
)
ioi_logits_minimal = model(ioi_dataset.toks)

print(f"""
Avg logit diff (IOI dataset, using entire model): {logits_to_ave_logit_diff_2(ioi_logits_original):.4f}
Avg logit diff (IOI dataset, only using circuit): {logits_to_ave_logit_diff_2(ioi_logits_minimal):.4f}
""")

logit 차이가 아주 조금만 감소하며, S보다 IO token을 선호하는 높은 likelihood ratio를 나타낼 만큼 여전히 충분히 높다는 것을 확인하실 수 있습니다.

### 연습 문제 - minimality 점수 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵⚪⚪⚪
> 
> This exercise is expected to take a long time; at least an hour.
> It is probably the second most challenging exercise in this notebook.
> ```

이 섹션은 모델의 minimality score를 보여주는 논문의 그림 7을 재현하며 마무리하겠습니다.

다시 한번 말씀드리지만, 이 실습은 매우 도전적이며 최소한의 가이드만으로 수행하도록 설계되었습니다. 이 그래프를 설명하는 논문의 관련 섹션인 섹션 4(experimental validation)의 시작부터 섹션 4.2 끝까지를 읽어야 합니다. 10페이지에 설명된 sampling 알고리즘은 수행할 필요가 없는데, 이는 아래 딕셔너리 형태로 각 컴포넌트에 대한 집합 $K$ 을 제공하기 때문입니다 (이는 논문의 그림 20인 "minimality sets" 표에 제공된 정보에 기반합니다).

In [ ]:
K_FOR_EACH_COMPONENT = {
    (9, 9): set(),
    (10, 0): {(9, 9)},
    (9, 6): {(9, 9), (10, 0)},
    (10, 7): {(11, 10)},
    (11, 10): {(10, 7)},
    (8, 10): {(7, 9), (8, 6), (7, 3)},
    (7, 9): {(8, 10), (8, 6), (7, 3)},
    (8, 6): {(7, 9), (8, 10), (7, 3)},
    (7, 3): {(7, 9), (8, 10), (8, 6)},
    (5, 5): {(5, 9), (6, 9), (5, 8)},
    (5, 9): {(11, 10), (10, 7)},
    (6, 9): {(5, 9), (5, 5), (5, 8)},
    (5, 8): {(11, 10), (10, 7)},
    (0, 1): {(0, 10), (3, 0)},
    (0, 10): {(0, 1), (3, 0)},
    (3, 0): {(0, 1), (0, 10)},
    (4, 11): {(2, 2)},
    (2, 2): {(4, 11)},
    (11, 2): {(9, 9), (10, 0), (9, 6)},
    (10, 6): {(9, 9), (10, 0), (9, 6), (11, 2)},
    (10, 10): {(9, 9), (10, 0), (9, 6), (11, 2), (10, 6)},
    (10, 2): {(9, 9), (10, 0), (9, 6), (11, 2), (10, 6), (10, 10)},
    (9, 7): {(9, 9), (10, 0), (9, 6), (11, 2), (10, 6), (10, 10), (10, 2)},
    (10, 1): {(9, 9), (10, 0), (9, 6), (11, 2), (10, 6), (10, 10), (10, 2), (9, 7)},
    (11, 9): {(9, 9), (10, 0), (9, 6), (9, 0)},
    (9, 0): {(9, 9), (10, 0), (9, 6), (11, 9)},
}

또한, head와 score를 매핑하는 딕셔너리 `minimality_scores` 가 주어졌을 때, 다음 코드는 논문에 나온 것과 유사한 플롯을 생성합니다:

In [ ]:
def plot_minimal_set_results(minimality_scores: dict[tuple[int, int], float]):
    """
    Plots the minimality results, in a way resembling figure 7 in the paper.

    minimality_scores:
        dict with elements like (9, 9): minimality score for head 9.9 (as described
        in section 4.2 of the paper)
    """

    CIRCUIT_reversed = {head: k for k, v in CIRCUIT.items() for head in v}
    colors = [CIRCUIT_reversed[head].capitalize() + " head" for head in minimality_scores.keys()]
    color_sequence = [px.colors.qualitative.Dark2[i] for i in [0, 1, 2, 5, 3, 6]] + ["#BAEA84"]

    bar(
        list(minimality_scores.values()),
        x=list(map(str, minimality_scores.keys())),
        labels={"x": "Attention head", "y": "Change in logit diff", "color": "Head type"},
        color=colors,
        template="ggplot2",
        color_discrete_sequence=color_sequence,
        bargap=0.02,
        yaxis_tickformat=".0%",
        legend_title_text="",
        title="Plot of minimality scores (as percentages of full model logit diff)",
        width=800,
        hovermode="x unified",
    )

이제 `minimality_scores` dictionary를 생성하고, 위에서 제공된 plot 함수를 사용하여 결과를 시각화해야 합니다:

In [ ]:
minimality_scores = {(9, 9): ...}
plot_minimal_set_results(minimality_scores)
# YOUR CODE HERE - create the `minimality_scores` dictionary, to be used in the plot function given above

<details>
<summary>힌트 (메인 함수에 유용하게 사용될 일부 함수들의 docstring입니다)</summary>

```python
def get_score(
    model: HookedTransformer,
    ioi_dataset: IOIDataset,
    abc_dataset: IOIDataset,
    K: set[tuple[int, int]],
    C: dict[str, list[tuple[int, int]]],
) -> float:
    '''
    Returns the value F(C \ K), where F is the logit diff, C is the
    core circuit, and K is the set of circuit components to remove.
    '''
    pass


def get_minimality_score(
    model: HookedTransformer,
    ioi_dataset: IOIDataset,
    abc_dataset: IOIDataset,
    v: tuple[int, int],
    K: set[tuple[int, int]],
    C: dict[str, list[tuple[int, int]]] = CIRCUIT,
) -> float:
    '''
    Returns the value | F(C \ K_union_v) - F(C | K) |, where F is
    the logit diff, C is the core circuit, K is the set of circuit
    components to remove, and v is a head (not in K).
    '''
    pass


def get_all_minimality_scores(
    model: HookedTransformer,
    ioi_dataset: IOIDataset = ioi_dataset,
    abc_dataset: IOIDataset = abc_dataset,
    k_for_each_component: dict = K_FOR_EACH_COMPONENT
) -> dict[tuple[int, int], float]:
    '''
    Returns dict of minimality scores for every head in the model (as
    a fraction of F(M), the logit diff of the full model).

    Warning - this resets all hooks at the end (including permanent).
    '''
    pass


minimality_scores = get_all_minimality_scores(model)

plot_minimal_set_results(minimality_scores)
```

세 번째 함수의 출력은 위에서 제공된 plotting 함수를 사용하여 시각화할 수 있습니다.

</details>


<details><summary>정답</summary>

```python
def get_score(
    model: HookedTransformer,
    ioi_dataset: IOIDataset,
    abc_dataset: IOIDataset,
    K: set[tuple[int, int]],
    C: dict[str, list[tuple[int, int]]],
) -> float:
    """
    Returns the value F(C \ K), where F is the logit diff, C is the core circuit, and K is the set
    of circuit components to remove.
    """
    C_excl_K = {k: [head for head in v if head not in K] for k, v in C.items()}
    model = add_mean_ablation_hook(model, abc_dataset, C_excl_K, SEQ_POS_TO_KEEP)
    logits = model(ioi_dataset.toks)
    score = logits_to_ave_logit_diff_2(logits, ioi_dataset).item()

    return score


def get_minimality_score(
    model: HookedTransformer,
    ioi_dataset: IOIDataset,
    abc_dataset: IOIDataset,
    v: tuple[int, int],
    K: set[tuple[int, int]],
    C: dict[str, list[tuple[int, int]]] = CIRCUIT,
) -> float:
    """
    Returns the value | F(C \ K_union_v) - F(C | K) |, where F is the logit diff, C is the core
    circuit, K is the set of circuit components to remove, and v is a head (not in K).
    """
    assert v not in K
    K_union_v = K | {v}
    C_excl_K_score = get_score(model, ioi_dataset, abc_dataset, K, C)
    C_excl_Kv_score = get_score(model, ioi_dataset, abc_dataset, K_union_v, C)

    return abs(C_excl_K_score - C_excl_Kv_score)


def get_all_minimality_scores(
    model: HookedTransformer,
    ioi_dataset: IOIDataset = ioi_dataset,
    abc_dataset: IOIDataset = abc_dataset,
    k_for_each_component: dict = K_FOR_EACH_COMPONENT,
) -> dict[tuple[int, int], float]:
    """
    Returns dict of minimality scores for every head in the model (as a fraction of F(M), the
    logit diff of the full model).

    Warning - this resets all hooks at the end (including permanent).
    """
    # Get full circuit score F(M), to divide minimality scores by
    model.reset_hooks(including_permanent=True)
    logits = model(ioi_dataset.toks)
    full_circuit_score = logits_to_ave_logit_diff_2(logits, ioi_dataset).item()

    # Get all minimality scores, using the `get_minimality_score` function
    minimality_scores = {}
    for v, K in tqdm(k_for_each_component.items()):
        score = get_minimality_score(model, ioi_dataset, abc_dataset, v, K)
        minimality_scores[v] = score / full_circuit_score

    model.reset_hooks(including_permanent=True)

    return minimality_scores

minimality_scores = get_all_minimality_scores(model)
```
</details>

참고 - 무작위 오차로 인해 결과가 논문과 완전히 일치하지 않을 수 있습니다 (예를 들어, 각 카테고리 내 head의 중요도 순서가 다를 수 있으며, 특히 backup name mover head와 같이 모델에 미치는 영향이 작은 head에서 더욱 그렇습니다). 하지만 중요한 특징들에 있어서는 상당히 유사할 것입니다.

# ☆ 보너스 / 이상 현상 탐색하기

> ##### 학습 목표
>
> * 모델의 다른 부분들을 탐색합니다 (예: negative name mover heads 및 induction heads).
> * 모델 circuit에 존재하는 미묘한 차이점들과, 초기 조사 후에 명백해 보이는 것보다 더 많은 부분이 circuit에 포함되어 있는 경우가 많다는 사실을 이해합니다.
> * 논문에서 사용된 세 가지 정량적 기준인 **faithfulness**, **completeness**, **minimality**의 중요성을 이해합니다.

여기에서는 아직 자세히 살펴보지 않은 circuit의 조금 더 생소한 부분들을 탐구합니다. 구체적으로, 탐구할 세 가지 부분이 있습니다:

* Early induction heads
* Backup name mover heads
* 이동되는 positional 정보와 token 정보의 비교

이 세 섹션은 모두 선택 사항이며, 원하는 만큼 (원하는 순서대로) 진행하시면 됩니다. 또한 이 섹션의 끝에는 저자나 Neel이 제안한 추가 조사 방향들이 제시되어 있습니다.

## 초기 induction heads

우리가 논의했듯이, 정말 이상한 관찰 결과는 중복된 token을 감지하는 초기 head 중 일부가 단순한 direct duplicate token head가 아니라 induction head라는 점입니다. 이는 매우 이상합니다! 어떻게 이런 일이 가능할까요?

먼저, induction head가 무엇인지 다시 한번 정리해 보겠습니다. induction head는 반복되는 시퀀스를 감지하고 이어갈 수 있는 중요한 attention head 유형입니다. 이는 두 개의 head로 구성된 induction circuit의 두 번째 head로, 현재 token의 이전 복사본을 찾고 그 *다음* token에 attention을 기울인 뒤, 이를 현재 위치로 복사하여 다음에 올 token으로 예측합니다. 이는 [Anthropic wrote a whole paper on them](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) 일 정도로 매우 중요한 역할을 합니다.

아래 다이어그램은 단어 `"Dursley"`이 두 번째로 등장했을 때, `" D"` 다음에 token `"urs"`이 온다고 예측하는 작동 방식을 보여줍니다 (모델이 Harry Potter 데이터로 학습되지 않았다고 가정하므로, 이는 in-context learning의 예시입니다).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ih-simple.png" width="650">

여기서 induction head가 나타나는 것이 왜 놀라운 일일까요? 이는 과한 설정(overkill)처럼 느껴지기 때문에 놀라운 일입니다. 모델은 subject의 첫 번째 복사본 뒤에 *어떤* token이 오는지에는 관심이 없으며, 단지 그것이 중복되었다는 사실에만 관심이 있습니다. 그리고 모델은 이미 더 단순한 duplicate token head를 가지고 있습니다. 제 추측으로는, 이미 주변에 induction head가 존재하고 있었고, 그것들이 주 기능 외에도 중복된 token에서만 *또한* 활성화되었기 때문인 것 같습니다. 따라서 기존의 메커니즘을 재활용하는 것이 유용했을 것입니다.

이는 우리가 더 큰 모델에서 circuit을 찾을 때, 단순한 circuit의 구성 요소들이 재활용되고 그 위에 새로운 기능이 구축됨에 따라 상황이 점점 더 복잡해질 수 있음을 시사합니다.

먼저, 아래 셀에서 반복되는 token이 포함된 시퀀스에 대해 induction head(`5.5` 및 `6.9`)의 attention pattern을 시각화하고, 이들이 동일한 token의 이전 인스턴스 *다음*에 오는 token을 attention하고 있는지 확인해야 합니다. "Validation of duplicate token heads" 섹션에서 작성했던 코드를 다시 사용하시면 됩니다.

In [ ]:
model.reset_hooks(including_permanent=True)

attn_heads = [(5, 5), (6, 9)]

# Get repeating sequences (note we could also take mean over larger batch)
batch = 1
seq_len = 15
rep_tokens = generate_repeated_tokens(model, seq_len, batch)

# Run cache (we only need attention patterns for layers 5 and 6)
_, cache = model.run_with_cache(
    rep_tokens,
    return_type=None,
    names_filter=lambda name: name.endswith("pattern") and any(f".{layer}." in name for layer, head in attn_heads),
)

# Display results
attn = t.stack([cache["pattern", layer][0, head] for (layer, head) in attn_heads])
cv.attention.attention_patterns(
    tokens=model.to_str_tokens(rep_tokens[0]),
    attention=attn,
    attention_head_names=[f"{layer}.{head}" for (layer, head) in attn_heads],
)

이것이 시사하는 바 중 하나는, 더 복잡한 circuit을 찾을 때 쉽게 찾을 수 있도록 head들을 더 단순한 circuit에 속하는지 여부에 따라 분류하는 것이 유용하다는 점입니다. 여기서는 이를 쉽게 수행할 수 있습니다! induction head에 관한 흥미로운 사실은, 이들이 반복되는 무작위 token 시퀀스에서도 작동한다는 점입니다. 이는 GPT-2가 학습한 자연어와는 완전히 다른 off distribution 데이터라는 점에서 주목할 만합니다. off distribution에서 모델의 동작을 예측할 수 있다는 것은 mechanistic interpretability의 성공을 보여주는 좋은 척도입니다! 이는 특정 head가 induction head인지 아닌지를 확인하는 좋은 sanity check가 됩니다.

우리는 무작위 token 시퀀스를 한 번 반복해서 제공하고, token의 두 번째 복사본에서 첫 번째 복사본 바로 다음 token으로 향하는 평균 attention을 측정함으로써 induction head의 특성을 정의할 수 있습니다. 동시에, token의 두 번째 복사본에서 첫 번째 복사본으로 향하는 평균 attention(이는 해당 head가 duplicate token head일 때 보여줄 attention입니다)과, previous token head를 찾기 위해 이전 token으로 향하는 평균 attention도 함께 측정할 수 있습니다.

참고로, 이는 어떤 것이 induction head인지에 대한 표면적인 연구입니다. 우리는 그것이 실제로 올바른 token을 boost 하는지, 혹은 단일 previous head와 어떻게 결합하는지에 대한 문제는 완전히 무시합니다. 특히, 때때로 induction-y token을 억제하는 anti-induction head가 발견되기도 하는데(이유는 알 수 없습니다!), 이 기법은 그러한 head들도 함께 찾아낼 것입니다.

### 연습 문제 - patching을 통한 prev token head 검증

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> This just involves performing a specific kind of patching, with functions you've already written.
> ```

논문에서는 head `2.2`과 `4.11`가 previous token head라고 언급합니다. 이전 섹션에서 previous token score를 플롯함으로써(Figure 18 재현 과정에서) 이미 이를 검증하셨기를 바랍니다. 하지만 이번에는, 이 head들이 우리 circuit 다이어그램에서 암시된 방식으로 previous token head로서 작동하고 있음을 증명하기 위해 특정한 종류의 path patching을 수행할 것입니다.

<details>
<summary>질문 - 어떤 종류의 path patching을 수행해야 할까요?</summary>

이 head들이 induction circuit에서 prev token head로 작동하고 있음을 보여주기 위해, 우리는 key-patching을 수행하고자 합니다 (즉, prev token head의 출력에서 induction head의 key 입력으로 이어지는 경로를 patch합니다).

우리는 이것이 성능을 악화시킬 것으로 예상합니다. 왜냐하면 induction head가 제공하는 duplicate token 신호를 방해하기 때문입니다.
</details>

In [ ]:
model.reset_hooks(including_permanent=True)

# YOUR CODE HERE - create `induction_head_key_path_patching_results`

In [ ]:
imshow(
    100 * induction_head_key_path_patching_results,
    title="Direct effect on Induction Heads' keys",
    labels={"x": "Head", "y": "Layer", "color": "Logit diff.<br>variation"},
    coloraxis=dict(colorbar_ticksuffix="%"),
    width=600,
)

<details><summary>솔루션</summary>

```python
model.reset_hooks(including_permanent=True)

induction_head_key_path_patching_results = get_path_patch_head_to_heads(
    receiver_heads=[(5, 5), (6, 9)], receiver_input="k", model=model, patching_metric=ioi_metric_2
)

```
</details>

논문의 다이어그램에 있는 induction heads 박스 안에 대괄호로 표시된 다른 두 개의 head(head `5.8` 및 `5.9`)가 있다는 점을 눈치채셨을 것입니다. 이 head들이 무엇을 하고 있는지, 그리고 왜 대괄호 안에 있는지 직접 알아내 보시겠습니까?

<details>
<summary>힌트</summary>

attention head의 출력에서 S-inhibition head의 value 입력으로 patching을 적용했던 path patching 섹션을 떠올려 보십시오 (논문의 그림 4b를 재현하셨기를 바랍니다). 이번에는 S-inhibition head의 **keys**에 patching을 시도해 보십시오. 여기서 head `5.8`와 `5.9`가 눈에 띄나요?
</details>

<details>
<summary>정답</summary>

힌트에서 제안한 플롯을 그려보면, `5.8`와 `5.9`에서 S-inhibition head로 patching을 할 때 각각 logit difference에 약 5%의 부정적인 영향을 미치는 것을 발견할 수 있습니다. 우리는 이것들이 induction head이며, `end`가 `S2`에 기울이는 attention을 증가시키는 역할을 한다고 결론짓고 싶을 것입니다. 불행히도, 논문의 그림 18 결과("validation of early heads" 섹션에서 재현한 내용)에 따르면 이들은 induction head가 아닌 것으로 보입니다.

알고 보니 이들은 이 분포에서는 induction head처럼 작동하지만, 일반적인 경우는 아닙니다. 이 가설을 뒷받침하는 간단한 방법은 attention pattern을 살펴보는 것입니다 (우리는 head `5.8`와 `5.9` 모두에서 `S2`가 `S1+1`에 attention을 기울이는 것을 발견합니다).
</details>

<details>
<summary>여담 - 비선형성에 대한 교훈</summary>

위의 드롭다운에서 저는 `5.8`와 `5.9`가 이 분포에서 induction을 수행하고 있다고 주장했습니다. 이를 테스트하는 더 엄격한 방법은 이 head들의 keys로 path patch를 수행하여, 이전 token head들 중 어느 하나가 큰 영향을 미치는지 확인하는 것입니다. 만약 그렇다면, 이는 induction에 대한 매우 강력한 증거가 됩니다. 하지만 결과적으로 prev token head들 중 어느 것도 fuzzy induction head를 통한 경로를 통해 모델의 IOI 성능에 영향을 주지 않는 것으로 나타났습니다. 이것이 우리의 가설을 무효화할까요?

사실, 비선형성 때문에 그렇지 않습니다. 두 prev token head `2.2`와 `4.11`는 확률에 직접 더해지는 것이 아니라, `S2`에서 `S1+1`로의 attention probability를 계산하는 데 사용되는 **logits**에 더해지게 됩니다. 따라서 하나의 head만으로는 attention probability를 많이 증가시키지 못하더라도, 두 head가 함께 작동하면 이를 상당히 증가시킬 수 있습니다. (동기 부여 예시: 가능한 source token이 2개만 있다고 가정해 봅시다. 어떤 head도 작동하지 않을 때 logits는 `[0.0, -8.0]`이며, 각 head의 역할은 두 번째 logit에 `4.0`를 더하는 것입니다. 하나의 head만 작동하면 두 번째 token의 확률은 거의 0에서 `0.018`로 변하며 (이는 attention head의 출력에 매우 작은 영향을 줍니다), 하지만 두 head가 모두 작동하면 logits가 같아져 확률이 `0.5`가 됩니다 (이는 분명히 훨씬 더 큰 영향을 미칩니다)).

실제로 제가 여기서 발견한 것이 바로 이것입니다. 하나 이상의 sender head를 허용하도록 path patching 함수를 수정한 후, `2.2`와 `4.11` 모두에서 `5.8`와 `5.9`로 patching을 하는 것이 단일 head에서 patching을 하는 것보다 모델의 IOI 성능에 더 큰 영향을 미친다는 것을 발견했습니다 (개별적으로는 거의 0%인 반면, 둘 다 적용했을 때는 성능이 약 1% 하락했습니다). 이러한 효과는 이 head들에서 네 개의 모든 induction head로 patching을 할 때도 발견되며, 이때는 더욱 두드러집니다 (개별 head의 경우 각각 3%와 15% 하락, 둘 다 적용 시 31% 하락).
</details>

## Backup name mover heads

또 다른 흥미로운 이상 현상은 **backup name mover heads**입니다. 모델 내부를 해석할 때 적용하는 표준적인 기법은 ablation 또는 knock-out입니다. 모델을 실행하되 특정 head를 0으로 설정하도록 개입하면 어떤 일이 일어날까요? 만약 모델이 이러한 개입에도 강건하다면, 단순하게 생각해서 해당 head가 중요한 일을 하고 있지 않다고 확신할 수 있으며, 반대로 모델의 작업 성능이 훨씬 떨어진다면 해당 head가 중요했음을 시사합니다. 이러한 접근 방식에는 몇 가지 개념적인 결함이 있어 증거가 암시적인 수준에 그칠 수 있습니다. 예를 들어, head의 평균 출력이 0에서 멀리 떨어져 있을 수 있으며, 따라서 knockout이 이를 예상 activation에서 멀어지게 만들어 *어떤* 작업에서든 내부 구조를 망가뜨릴 수 있습니다. 하지만 여전히 일부 데이터를 얻기 위해 적용하기에 유용한 기법입니다.

하지만 논문에서 발견된 놀라운 점은 모델에 **내장된 중복성(built in redundancy)**이 있다는 것입니다. 만약 name mover 중 하나를 knockout 하면, 이후 레이어에 있는 일부 backup name mover들이 *동작을 변경하여* 원래 name mover head가 하던 일의 (일부를) 수행합니다. 이는 단순한 knockout 방식이 name mover의 중요성을 상당히 과소평가하게 된다는 것을 의미합니다.

한번 테스트해 보겠습니다! 아래 코드에서 확인한 것처럼 가장 중요한 name mover인 `9.9`를 `end` token에서만 ablate하고, 성능을 비교해 보겠습니다.

In [ ]:
model.reset_hooks(including_permanent=True)

ioi_logits, ioi_cache = model.run_with_cache(ioi_dataset.toks)
original_average_logit_diff = logits_to_ave_logit_diff_2(ioi_logits)

s_unembeddings = model.W_U.T[ioi_dataset.s_tokenIDs]
io_unembeddings = model.W_U.T[ioi_dataset.io_tokenIDs]
logit_diff_directions = io_unembeddings - s_unembeddings  # [batch d_model]

per_head_residual, labels = ioi_cache.stack_head_results(layer=-1, return_labels=True)
per_head_residual = einops.rearrange(
    per_head_residual[:, t.arange(len(ioi_dataset)).to(device), ioi_dataset.word_idx["end"].to(device)],
    "(layer head) batch d_model -> layer head batch d_model",
    layer=model.cfg.n_layers,
)

per_head_logit_diffs = residual_stack_to_logit_diff(per_head_residual, ioi_cache, logit_diff_directions)

top_layer, top_head = topk_of_Nd_tensor(per_head_logit_diffs, k=1)[0]
print(f"Top Name Mover to ablate: {top_layer}.{top_head}")

# Getting means we can use to ablate
abc_means = ioi_circuit_extraction.compute_means_by_template(abc_dataset, model)[top_layer]


# Define hook function and add to model
def ablate_top_head_hook(z: Float[Tensor, "batch pos head_index d_head"], hook):
    """
    Ablates hook by patching in results
    """
    z[range(len(ioi_dataset)), ioi_dataset.word_idx["end"], top_head] = abc_means[
        range(len(ioi_dataset)), ioi_dataset.word_idx["end"], top_head
    ]
    return z


model.add_hook(utils.get_act_name("z", top_layer), ablate_top_head_hook)

# Run the model, temporarily adds caching hooks and then removes *all* hooks after running,
# including the ablation hook.
ablated_logits, ablated_cache = model.run_with_cache(ioi_dataset.toks)
rprint(
    "\n".join(
        [
            f"{original_average_logit_diff:.4f} = Original logit diff",
            f"{per_head_logit_diffs[top_layer, top_head]:.4f} = Direct Logit Attribution of top name mover head",
            f"{original_average_logit_diff - per_head_logit_diffs[top_layer, top_head]:.4f} = Naive prediction of post ablation logit diff",
            f"{logits_to_ave_logit_diff_2(ablated_logits):.4f} = Logit diff after ablating L{top_layer}H{top_head}",
        ]
    )
)

여기서 무슨 일이 일어나고 있는 것일까요? 전체 모델에 대한 logit diff를 계산하고, 그중 얼마만큼이 head `9.9` 에서 직접적으로 오는지 계산합니다. 이를 바탕으로, 이 head를 ablate 했을 때 logit diff가 어느 정도까지 떨어질지에 대한 추정치를 도출합니다. 실제로 성능은 이러한 단순한 예측보다 **훨씬** 더 좋습니다.

왜 이런 현상이 발생하는 것일까요? 이전과 마찬가지로, 각 head의 direct logit attribution을 살펴봄으로써 어떤 일이 일어나고 있는지 파악할 수 있습니다.

In [ ]:
per_head_ablated_residual, labels = ablated_cache.stack_head_results(layer=-1, return_labels=True)
per_head_ablated_residual = einops.rearrange(
    per_head_ablated_residual[:, t.arange(len(ioi_dataset)).to(device), ioi_dataset.word_idx["end"].to(device)],
    "(layer head) batch d_model -> layer head batch d_model",
    layer=model.cfg.n_layers,
)
per_head_ablated_logit_diffs = residual_stack_to_logit_diff(
    per_head_ablated_residual, ablated_cache, logit_diff_directions
)
per_head_ablated_logit_diffs = per_head_ablated_logit_diffs.reshape(model.cfg.n_layers, model.cfg.n_heads)

In [ ]:
imshow(
    t.stack(
        [
            per_head_logit_diffs,
            per_head_ablated_logit_diffs,
            per_head_ablated_logit_diffs - per_head_logit_diffs,
        ]
    ),
    title="Direct logit contribution by head, pre / post ablation",
    labels={"x": "Head", "y": "Layer"},
    facet_col=0,
    facet_labels=["No ablation", "9.9 is ablated", "Change in head contribution post-ablation"],
    width=1200,
)

scatter(
    y=per_head_logit_diffs.flatten(),
    x=per_head_ablated_logit_diffs.flatten(),
    hover_name=labels,
    range_x=(-1, 1),
    range_y=(-2, 2),
    labels={"x": "Ablated", "y": "Original"},
    title="Original vs Post-Ablation Direct Logit Attribution of Heads",
    width=600,
    add_line="y=x",
)

첫 번째 그래프들은 head `9.9`를 ablate한 후, 해당 head가 logit diff에 주는 직접적인 기여도는 (당연하게도) 감소하지만, 다른 head들(특히 layer 10에 있는 head들)의 기여도는 실제로 많이 증가한다는 것을 보여줍니다. 두 번째 그래프는 이를 다른 방식으로 보여줍니다 (오른쪽 heatmap에서 눈에 띄는 head들은 scatter plot에서 y=x 선보다 훨씬 아래에 위치한 head들과 동일합니다).

한 가지 자연스러운 가설은 final LayerNorm scaling이 변경되어 final residual stream의 크기가 커지거나 작아졌기 때문이라는 것입니다. 이는 어느 정도 사실이며, 일반적인 head들이 x=y 선에서 약간 벗어나 있는 것을 볼 수 있습니다. 하지만 평균 LN scaling ratio는 1.04이며, 이는 *모든* head를 동일한 비율로 균일하게 변화시켜야 하므로, 이것만으로는 충분한 설명이 될 수 없습니다.

In [ ]:
ln_scaling_no_ablation = ioi_cache["ln_final.hook_scale"][
    t.arange(len(ioi_dataset)), ioi_dataset.word_idx["end"]
].squeeze()
ln_scaling_ablated = ablated_cache["ln_final.hook_scale"][
    t.arange(len(ioi_dataset)), ioi_dataset.word_idx["end"]
].squeeze()

In [ ]:
scatter(
    y=ln_scaling_ablated,
    x=ln_scaling_no_ablation,
    labels={"x": "No ablation", "y": "Ablation"},
    title=f"Final LN scaling factors compared (ablation vs no ablation)<br>Average ratio = {(ln_scaling_no_ablation / ln_scaling_ablated).mean():.4f}",
    width=700,
    add_line="y=x"
)

**독자 연습 문제:** 이 분석을 마무리하실 수 있습니까? 여기서 어떤 일이 일어나고 있습니까? 왜 backup name mover들이 행동을 바꾸고 있습니까? 왜 하나의 negative name mover가 현저하게 덜 중요해지고 있습니까?

## 이동되는 positional 정보 vs token 정보

부록의 A 섹션(**Disentangling token and positional signal in the output of S-Inhibition Heads**)에서 저자들은 S-Inhibition head가 `S1`에 가해지는 attention을 억제하기 위해 token 정보와 positional 정보 중 무엇을 사용하는지 알아내고자 합니다. 이는 저의 IOI 다이어그램에서 보라색 상자와 분홍색 상자로 구분되어 설명되어 있습니다.

저자들이 어떤 정보가 무엇인지 찾아내는 방식은 매우 독창적입니다. 그들은 원본 IOI 데이터셋에서 일부 신호를 삭제하거나 반전시켜 데이터셋을 구성합니다. 예를 들어, S-inhibition head가 기록한 token 정보는 유지하면서 positional 정보를 반전시켰을 때의 효과를 조사하고 싶다면, 다음과 같은 문장을:

```
When Mary and John went to the store, John gave a drink to Mary
```

다음과 같이 대체할 수 있습니다:

```
When John and Mary went to the store, John gave a drink to Mary
```

이 방식이 왜 작동하는지 정확히 짚어보겠습니다. S-inhibition head에 의해 `end` token 위치에 기록되는 정보는 "중복된 token에 attention을 기울이지 마라"라는 신호와 "중복된 token과 동일한 위치에 있는 token에 attention을 기울이지 마라"라는 신호의 조합일 것입니다. 만약 모델을 위의 첫 번째 문장으로 실행한 뒤 두 번째 문장을 patch 한다면 다음과 같은 결과가 나타납니다:

* **"중복된 token에 attention을 기울이지 마라"** 신호는 변하지 않습니다 (이 신호는 여전히 John을 가리키기 때문입니다).
* **"중복된 token과 동일한 위치에 있는 token에 attention을 기울이지 마라"** 신호는 반전됩니다 (이 정보는 두 번째 문장의 `Mary` 위치를 가리키며, 결과적으로 첫 번째 문장의 `John` 위치를 가리키게 되기 때문입니다).

그것은 단지 하나의 예시(positional 정보는 뒤집고, token 정보는 그대로 유지하는 경우)였을 뿐이며, 우리는 다음과 같이 여섯 가지 서로 다른 유형의 flip을 수행할 수 있습니다:

| Token signal | Positional signal | Sentence                                                          | ABB -> ? |
| ------------ | ----------------- | ----------------------------------------------------------------- | -------- |
| 동일         | 동일              | `When Mary and John went to the store, John gave a drink to Mary` | ABB      |
| 랜덤         | 동일              | `When Emma and Paul went to the store, Paul gave ...`             | CDD      |
| 반전         | 동일              | `When John and Mary went to the store, Mary gave ...`             | BAA      |
| 동일         | 반전              | `When John and Mary went to the store, John gave ...`             | BAB      |
| 랜덤         | 반전              | `When Paul and Emma went to the store, Emma gave ...`             | DCD      |
| 반전         | 반전              | `When Mary and John went to the store, Mary gave ...`             | ABA      |

우리는 이러한 각 데이터셋을 생성하기 위해 `gen_flipped_prompts` 방법을 사용합니다:

In [ ]:
datasets: list[tuple[tuple, str, IOIDataset]] = [
    ((0, 0), "original", ioi_dataset),
    ((1, 0), "random token", ioi_dataset.gen_flipped_prompts("ABB->CDD, BAB->DCD")),
    ((2, 0), "inverted token", ioi_dataset.gen_flipped_prompts("ABB->BAA, BAB->ABA")),
    ((0, 1), "inverted position", ioi_dataset.gen_flipped_prompts("ABB->BAB, BAB->ABB")),
    (
        (1, 1),
        "inverted position, random token",
        ioi_dataset.gen_flipped_prompts("ABB->DCD, BAB->CDD"),
    ),
    (
        (2, 1),
        "inverted position, inverted token",
        ioi_dataset.gen_flipped_prompts("ABB->ABA, BAB->BAA"),
    ),
]

results = t.zeros(3, 2).to(device)

s2_inhibition_heads = CIRCUIT["s2 inhibition"]
layers = set(layer for layer, head in s2_inhibition_heads)

names_filter = lambda name: name in [utils.get_act_name("z", layer) for layer in layers]


def patching_hook_fn(z: Float[Tensor, "batch seq head d_head"], hook: HookPoint, cache: ActivationCache):
    heads_to_patch = [head for layer, head in s2_inhibition_heads if layer == hook.layer()]
    z[:, :, heads_to_patch] = cache[hook.name][:, :, heads_to_patch]
    return z


for (row, col), desc, dataset in datasets:
    # Get cache of values from the modified dataset
    _, cache_for_patching = model.run_with_cache(dataset.toks, names_filter=names_filter, return_type=None)

    # Run model on IOI dataset, but patch S-inhibition heads with signals from modified dataset
    patched_logits = model.run_with_hooks(
        ioi_dataset.toks,
        fwd_hooks=[(names_filter, partial(patching_hook_fn, cache=cache_for_patching))],
    )

    # Get logit diff for patched results
    # Note, we still use IOI dataset for our "correct answers" reference point
    results[row, col] = logits_to_ave_logit_diff_2(patched_logits, ioi_dataset)

In [ ]:
imshow(
    results,
    labels={"x": "Positional signal", "y": "Token signal"},
    x=["Original", "Inverted"],
    y=["Original", "Random", "Inverted"],
    title="Logit diff after changing all S2 inhibition heads' output signals via patching",
    text_auto=".2f",
)

이 플롯에 대해 어떻게 해석하시나요?

<details>
<summary>몇 가지 생각들</summary>

이 플롯에서 기대할 수 있는 몇 가지 sanity check 결과이며, 이는 우리의 코드가 아마도 정확하다는 것을 입증합니다:

* token과 positional 신호가 반전되었을 때, 성능은 원래 성능의 음수 값에 가깝습니다.
* positional 신호를 반전시키면 성능이 저하됩니다.
* token 신호를 무작위화하면 성능이 저하됩니다.
* token 신호를 반전시키면 성능이 저하됩니다. 이 효과는 무작위화했을 때보다 더 큽니다 (단순히 무작위 방향이 아니라, 정답의 반대 방향을 가리키고 있기 때문입니다).

이 플롯에서 발견할 수 있는 두 가지 주요 흥미로운 점은 다음과 같습니다 (미리 추측했을 수도 있지만, 확신할 수는 없었던 부분입니다):
* 행과 열의 차이가 거의 일정합니다 (열 차이는 약 4.5, 행 차이는 약 0.75입니다). 다시 말해, logit diff는 positional 및 token 신호 상관관계의 선형 결합으로 잘 근사될 수 있습니다 (상관관계는 신호가 정답 값을 가리키면 1, 반대 방향이면 -1, 무작위 방향이면 0입니다).
* positional 신호 상관관계의 계수가 token 신호 상관관계의 계수보다 **훨씬** 큽니다. 전자는 약 4.5이고, 후자는 약 1.5입니다. 이는 positional 정보가 token 정보보다 훨씬 더 중요하다는 것을 알려줍니다.
    * 이에 대한 한 가지 가능한 직관은 이름 정보(즉, token의 정체성을 표현하는 것 `" John"`)가 많은 차원을 차지하므로 모델에게 더 어려울 가능성이 높다는 것입니다. 반면, 상대적 positional 정보는 대부분 더 적은 차원을 가질 것입니다. 모델은 전체 이름 집합에서 이름을 구별하는 것이 아니라, 약 15-20개의 positional index 중에서 단 하나의 index만 구별해낼 수 있을 만큼의 정보만 이동시키면 되기 때문입니다. 하지만 이는 단지 추측일 뿐이며, 다른 해석이 있다면 듣고 싶습니다.

</details>

조금 더 자세히 살펴보겠습니다. S-inhibition head들을 집합적으로 보는 대신, 각각의 head를 개별적으로 살펴볼 수 있습니다.

### 연습 문제 - S-Inhibition head 분해하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> This involves a lot of duplicating code from above.
> ```

위와 동일한 그래프를 그리되, 이번에는 각 S-inhibition head에 개별적으로 intervention을 수행한 후의 결과를 그려보세요.

이를 위해 `(M, 3, 2)` 형태의 `results` 텐서를 생성할 수 있습니다. 여기서 `M`은 S-inhibition head의 개수이며, 각 슬라이스는 해당 head에 intervention을 수행한 결과를 포함합니다. 결과 그래프를 그리기 위한 코드는 아래에 제공되어 있으며, `results` 부분만 채우시면 됩니다.

(참고 - 그래프를 더 명확하게 만들기 위해, 결과값을 `(logit_diff - clean_logit_diff) / clean_logit_diff`로 계산하는 것을 권장합니다. 이렇게 하면 "이 patching이 아무런 영향을 주지 않음"은 0으로, "이 patching이 모델 성능을 완전히 파괴함"은 -1로 표시됩니다.)

In [ ]:
results = t.zeros(len(CIRCUIT["s2 inhibition"]), 3, 2).to(device)


# YOUR CODE HERE - fill in the `results` tensor!

    imshow(
        (results - results[0, 0, 0]) / results[0, 0, 0],
        labels={"x": "Positional signal", "y": "Token signal"},
        x=["Original", "Inverted"],
        y=["Original", "Random", "Inverted"],
        title="Logit diff after patching individual S2 inhibition heads (as proportion of clean logit diff)",
        facet_col=0,
        facet_labels=[f"{layer}.{head}" for (layer, head) in CIRCUIT["s2 inhibition"]],
        facet_col_spacing=0.08,
        width=1100,
        text_auto=".2f",
    )

<details><summary>솔루션</summary>

```python
results = t.zeros(len(CIRCUIT["s2 inhibition"]), 3, 2).to(device)


def patching_hook_fn(z: Float[Tensor, "batch seq head d_head"], hook: HookPoint, cache: ActivationCache, head: int):
    z[:, :, head] = cache[hook.name][:, :, head]
    return z


for i, (layer, head) in enumerate(CIRCUIT["s2 inhibition"]):
    model.reset_hooks(including_permanent=True)

    hook_name = utils.get_act_name("z", layer)

    for (row, col), desc, dataset in datasets:
        # Get cache of values from the modified dataset
        _, cache_for_patching = model.run_with_cache(
            dataset.toks, names_filter=lambda name: name == hook_name, return_type=None
        )

        # Run model on IOI dataset, but patch S-inhibition heads with modified dataset signals
        patched_logits = model.run_with_hooks(
            ioi_dataset.toks,
            fwd_hooks=[(hook_name, partial(patching_hook_fn, cache=cache_for_patching, head=head))],
        )

        # Get logit diff for patched results
        # Note, we still use IOI dataset for our "correct answers" reference point
        results[i, row, col] = logits_to_ave_logit_diff_2(patched_logits, ioi_dataset)

imshow(
    (results - results[0, 0, 0]) / results[0, 0, 0],
    labels={"x": "Positional signal", "y": "Token signal"},
    x=["Original", "Inverted"],
    y=["Original", "Random", "Inverted"],
    title="Logit diff after patching individual S2 inhibition heads (as proportion of clean logit diff)",
    facet_col=0,
    facet_labels=[f"{layer}.{head}" for (layer, head) in CIRCUIT["s2 inhibition"]],
    facet_col_spacing=0.08,
    width=1100,
    text_auto=".2f",
)
```
</details>

이 그래프의 주목할 만한 특징들은 다음과 같습니다:

* 모든 head는 token signal보다 positional signal에 더 많은 관심을 가집니다.
* Head `8.6` (가장 큰 S-inhibition head)는 positional signal에 훨씬 더 많은 관심을 가지며, 실제로 token signal에는 전혀 관심을 가지지 않습니다.
    * 이는 아마도 `8.6`이 이 태스크를 수행하도록 학습한 첫 번째 head였으며, 이후의 head들은 기본적으로 나머지(token signal)를 제공함으로써 도움을 주었음을 시사합니다. 이는 head들이 전문화된다는 것을 보여줍니다.
* token signal에 어느 정도 관심을 가지는 유일한 head들은 `7.9`와 `8.10`입니다 (하지만 이들 역시 positional signal에 거의 두 배 더 많은 관심을 가집니다).
* logit diff를 positional signal과 token signal 상관관계의 합으로 근사하는 방식은 각 head의 계수가 서로 다름에도 불구하고, 개별 head별로 여전히 유효한 것으로 보입니다.

## 추가 읽을거리

앞서 언급되지 않은 추가 읽을거리를 위한 링크 모음입니다:

* [Some Lessons Learned from Studying Indirect Object Identification in GPT-2 small](https://www.alignmentforum.org/posts/3ecs6duLmTfyra3Gp/some-lessons-learned-from-studying-indirect-object)
    * 이 논문의 저자들이 작성한 블로그 포스트로, 실험과 결과에 대해 더 자세히 다룹니다.
* [Causal Scrubbing: a method for rigorously testing interpretability hypotheses [Redwood Research]](https://www.alignmentforum.org/posts/JvZhhzycHu2Yd57RN/causal-scrubbing-a-method-for-rigorously-testing)
    * mechanistic interpretations의 품질을 평가하기 위해 제안된 체계적인 방법인 causal scrubbing의 개념을 소개합니다.

## 추가 탐구를 위한 추천 주제

다음은 Neel 또는 논문 저자들이 제안한 향후 연구 방향입니다. 이 중 상당수는 훌륭한 캡스톤 프로젝트 주제가 될 수 있습니다!

* 3글자 약어 (또는 그 이상!)
* 이름을 이메일로 변환하기.
    * 확장 과제 예시: 다음과 같은 스니펫으로부터 이메일을 구성하는 작업: Name: Neel Nanda; Email: last name dot first name k @ gmail
* 문법 규칙
    * 마침표 뒤의 단어는 대문자로 시작한다는 점을 학습하는 것
    * 동사 활용
    * 적절한 대명사 선택 (예: he vs she vs it vs they)
    * 특정 단어가 고유 명사인지 여부
* 감성 분석 (예: 어떤 대상이 'good'으로 묘사될지 'bad'로 묘사될지 예측하기)
* memorisation 해석하기. 예: GPT-2가 사람의 연락처 정보와 같은 놀라운 사실들을 알고 있는 경우가 있습니다. 이것이 어떻게 가능할까요?
* 텍스트에 묘사된 객체 개수 세기. 예: I picked up an apple, a pear, and an orange. I was holding three fruits.
* Alex Variengien의 확장 제안
    * adversarial examples에서 어떤 일이 일어나는지 이해하기: 특히 S-Inhibition Head의 attention pattern 분석 (어려움). (S-Inhibition heads는 IOI 논문에서 언급됩니다)
    * positional signal이 어떻게 인코딩되는지 이해하기 (상대적 거리인지, 아니면 다른 방식인지?). 만약 positional embeddings를 포함하여, Duplicate Token Heads / Induction Heads가 위치 간의 차이를 어떻게 계산하는지(상대적 위치 프레임워크가 맞다면) 설명하는 스토리를 구성한다면 가산점이 부여됩니다. (어렵지만, 문맥 의존도가 낮음)
    * IOI에서 MLP의 역할은 무엇인가? (범위가 매우 넓고 어려움)
    * IOI 외에서 Duplicate Token Heads의 역할은 무엇인가? 이들이 S-Inhibition Heads와 함께 다른 Q-compositions에 사용되는가? 이들의 QK circuit가 파라미터 수준에서 "collision detection"을 어떻게 구현하는지 설명할 수 있는가? (마지막 질문은 문맥 의존도가 낮고 상당히 다룰 만함)
    * IOI 외에서 Negative/ Backup/ regular Name Movers Heads의 역할은 무엇인가? Negative Name Movers가 다음 token 예측에 긍정적으로 기여하는 사례를 찾을 수 있는가?
    * GPT2-small에 존재하는 5개의 induction heads 사이에는 어떤 차이가 있는가? 이들이 의존하는 head는 무엇이며, 이후 어떤 head와 compose 하는가? (IOI의 문맥 의존도가 낮은 형태)
    * 4.11 (매우 날카로운 previous token heads)을 파라미터 수준에서 이해하기. 이 head의 attention pattern이 거의 완벽하게 off-diagonal이므로 상당히 다룰 만한 주제라고 생각합니다.
* compensation mechanisms가 발생하기 위한 조건은 무엇인가? dropout 때문인가? (Arthur Conmy가 이 작업을 진행 중입니다 - arthur@rdwrs.com으로 편하게 연락하시기 바랍니다)
* Arthur Conmy의 확장 제안
    * GPT-Neo에서의 IOI 이해하기: 모델 크기는 같지만 MLP의 composition을 통해 IOI를 수행합니다.
    * Stanford mistral 모델들에서의 IOI 이해하기: 이 모델들은 모두 동일한 방식으로 IOI를 수행하는 것으로 보이므로, 학습 과정에 따른 circuit의 발달 과정을 살펴보는 것은 어떨까요?
* [Help out Redwood Research’s interpretability team by finding heuristics implemented by GPT-2 small](https://www.lesswrong.com/posts/LkBmAGJgZX2tbwGKg/help-out-redwood-research-s-interpretability-team-by-finding)
    * 6개월 전의 이 LessWrong 포스트는 IOI task를 연구하기 좋은 선택으로 만든 몇 가지 특징들을 설명하며, 이러한 기준을 충족할 수 있는 다른 task들이나 그러한 task들을 찾는 방법에 대해 제안합니다.

## 추천 논문 재현 과제

<br>

### [A circuit for Python docstrings in a 4-layer attention-only transformer](https://www.lesswrong.com/posts/u6KXXmKFbXfWzoAXn/a-circuit-for-python-docstrings-in-a-4-layer-attention-only)

이 작업은 Neel Nanda의 지도 하에 SERI ML Alignment Theory Scholars Program (Winter 2022)의 일환으로 수행되었습니다. IOI 논문이 어떤 의미에서 3개 layer가 필요한 가장 단순한 형태의 circuit을 찾았던 것과 비슷하게, 이 작업은 4개 layer가 필요한 가장 단순한 형태의 circuit을 찾는 것이었습니다. 이들이 조사한 작업은 **docstring task**입니다. 즉, 다음과 같은 상황에서 parameter를 올바른 순서로 예측할 수 있는지를 확인하는 것입니다:

```python
def port(self, load, size, files, last):
    '''oil column piece

    :param load: crime population
    :param size: unit dark
    :param
```

다음에 올 token은 ` files`이어야 하며, IOI의 경우와 마찬가지로 transformer가 이 작업을 어떻게 해결하는지 심층적으로 분석할 수 있습니다. IOI와 달리, 여기서는 (GPT2-Small이 아닌) 코드 데이터로 학습된 4-layer transformer를 살펴봅니다. 이 덕분에 (circuit이 IOI보다 더 많은 수준의 composition을 가지고 있음에도 불구하고) 많은 분석 과정이 더 깔끔해집니다.

추가적인 도전을 원하신다면, 저자들의 결과를 단순히 재현하는 대신 논문 저자들이 어떤 도구를 사용했는지 보지 않고 직접 조사를 수행해 보시기 바랍니다! 대부분은 지금까지 연습 문제에서 사용했던 도구들과 비슷할 것입니다.

다음과 같은 경우에 이 재현 과제가 적합할 수 있습니다:

* 이 연습 문제의 대부분 또는 모든 섹션을 즐겁게 학습했으며, 배운 도구들을 다른 맥락에서 사용하는 연습을 하고 싶은 경우
* 이 연습 문제의 내용에서 *너무* 벗어난 시도를 하고 싶지 않은 경우 (이 포스트에는 막혔을 때 참고할 수 있는 Colab notebook이 함께 제공됩니다)

<br>

### [Mechanistically interpreting time in GPT-2 small](https://www.lesswrong.com/posts/6tHNM2s6SWzFHv3Wo/mechanistically-interpreting-time-in-gpt-2-small)

이 작업은 Arthur Conmy의 지도 하에 독립 연구자 그룹에 의해 수행되었습니다. 작업 내용은 GPT2-Small이 다음 날짜 예측 작업, 즉 *If today is Monday, tomorrow is*와 같은 문장에서 다음 token을 어떻게 예측하는지 해석하는 것이었습니다. 이 재현 과제는 핵심 circuit이 여러 개의 composition이 아닌 단 하나의 attention head만 포함하고 있기 때문에 이전 과제보다 더 쉽습니다. 이러한 이유로, 가이드 없이 이 재현을 시도해 보는 것을 더 강력히 권장합니다. 즉, 전체 포스트를 읽기 전에 직접 도전해 보시기 바랍니다.

<br>

### [How does GPT-2 compute greater-than? Interpreting mathematical abilities in a pre-trained language model](https://openreview.net/pdf?id=p4PckNQR8k)

이 논문은 Redwood Research에서 운영한 REMIX 프로그램의 결과물입니다. 이 논문은 앞선 사례와 매우 유사하게 GPT2-Small의 circuit을 분석합니다. 여기서 circuit은 'greater-than'을 계산하기 위한 것입니다. 다시 말해, *The war lasted from the year 1732 to the year 17...*와 같은 문장이 유효한 두 자리 종료 연도(32보다 큰 연도)로 완성될 것임을 감지하는 것입니다. 이 논문은 circuit을 식별하고, 각 circuit 구성 요소의 역할을 설명하며, 해당 circuit을 활성화하는 관련 작업들도 찾아냅니다.

추가적인 도전을 원하신다면, 이 논문을 단순히 재현하는 대신 저자들이 어떤 도구를 사용했는지 보지 않고 직접 조사를 수행해 보시기 바랍니다! 많은 부분이 지금까지 연습 문제에서 사용했던 도구들과 비슷하겠지만, 일부는 다를 것입니다. 특히, 이 IOI notebook과는 달리 이 재현 과제에서는 개별 neuron에 대한 일부 분석이 포함될 것입니다.

다음과 같은 경우에 이 재현 과제가 적합할 수 있습니다:

* 이 연습 문제의 대부분 또는 모든 섹션을 즐겁게 학습했으며, 배운 도구들을 다른 맥락에서 사용하는 연습을 하고 싶은 경우
* 이 연습 문제의 내용에서 *너무* 벗어난 시도를 하고 싶지 않은 경우 (다만, 주로 MLP의 존재로 인해 docstring circuit보다는 아마 더 어려울 것입니다)

<br>

### [Towards Automated Circuit Discovery for Mechanistic Interpretability](https://arxiv.org/abs/2304.14997) / [Attribution Patching Outperforms Automated Circuit Discovery](https://arxiv.org/abs/2310.10348)

이 두 가지는 자동화를 통해 circuit discovery 도구를 확장할 가능성을 모두 연구하기 때문에 함께 묶었습니다. **Automated Circuit Discovery** (ACDC)는 Arthur Conmy와 공동 연구자들이 Redwood Research에서 부분적으로 수행하여 발견한 도구입니다. 이들은 우리가 이 연습 문제에서 사용했던 여러 기술(특히 activation & path patching)을 자동화하여, 작업을 여전히 효과적으로 해결할 수 있는 최소한의 circuit이 남을 때까지 구성 요소 간의 연결을 반복적으로 끊어냅니다. **Attribution patching**은 약간 다릅니다. 이는 단순하고 계산 비용이 저렴한 1차 근사(first-order approximation)를 통해 모델의 output에 각 구성 요소가 미치는 영향(나아가 특정 작업에서의 중요도)을 추정합니다.

이 두 가지 기술 중 어느 것이든 일주일 정도의 기간 동안 적절한 재현 도전 과제가 될 것입니다. 또한 이러한 기술들을 개선하거나 정교화하는 데 있어 여전히 할 수 있는 작업이 많이 남아 있습니다.

다음과 같은 경우에 이 재현 과제가 적합할 수 있습니다:

* 이 섹션의 연습 문제, 특히 activation 및 path patching에 관한 문제를 즐겁게 학습한 경우
* 이론보다는 코딩과 구현에 더 집중된 프로젝트를 찾고 있는 경우
* circuit discovery를 효율적으로 확장하거나 자동화하는 방법을 탐구하는 것에 흥미가 있는 경우